# 통합공시 기준자료 배분 실행노트북

이 노트북 하나로 배분안 기준 결과표 생성, 새 기준자료 자동 배정, ALIO 숫자 대조, 증빙본문 숏리스트 작성, 개인별 전달 ZIP 생성을 처리합니다.

새로 받은 파일은 `20_기준자료/01_미배정_신규자료`에 넣고, 실행 결과는 `20_기준자료/03_개인별_기준자료`와 `40_전달패키지`에서 확인합니다.


In [ ]:
from __future__ import annotations

import argparse
import csv
import contextlib
import io
import json
import os
import shutil
import stat
import sys
import types
import zipfile
from datetime import datetime
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / '00_배분안').exists() and (ROOT.parent / '00_배분안').exists():
    ROOT = ROOT.parent
TOOLS_DIR = ROOT / '90_실행노트북'
UNASSIGNED_DIR = ROOT / '20_기준자료' / '01_미배정_신규자료'
ASSIGNED_DONE_DIR = ROOT / '20_기준자료' / '02_배정완료_원본'
PERSONAL_DIR = ROOT / '20_기준자료' / '03_개인별_기준자료'
SOURCE_DIR = UNASSIGNED_DIR
NORM_DIR = PERSONAL_DIR


def _make_writable(path: Path | str) -> None:
    try:
        os.chmod(path, stat.S_IREAD | stat.S_IWRITE | stat.S_IEXEC)
    except OSError:
        pass


def _rmtree_onexc(function, path, _exc_info) -> None:
    _make_writable(path)
    function(path)


def _rmtree_onerror(function, path, _exc_info) -> None:
    _make_writable(path)
    function(path)


def robust_rmtree(path: Path) -> None:
    path = Path(path)
    if not path.exists():
        return
    for current, dirs, files in os.walk(path, topdown=False):
        for name in files:
            _make_writable(Path(current) / name)
        for name in dirs:
            _make_writable(Path(current) / name)
        _make_writable(current)
    _make_writable(path)
    try:
        shutil.rmtree(path, onexc=_rmtree_onexc)
    except TypeError:
        shutil.rmtree(path, onerror=_rmtree_onerror)


def reset_personal_dir(personal_dir: Path) -> None:
    personal_dir = Path(personal_dir)
    expected_parent = (ROOT / '20_기준자료').resolve()
    if personal_dir.resolve().parent != expected_parent:
        raise RuntimeError(f'Refusing to clean unexpected personal folder path: {personal_dir}')
    if personal_dir.exists():
        try:
            robust_rmtree(personal_dir)
        except Exception as exc:
            archive_root = ROOT / '30_검토산출물' / '98_개인별기준자료_백업'
            archive_root.mkdir(parents=True, exist_ok=True)
            target = archive_root / f"personal_folder_failed_clean_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
            suffix = 2
            while target.exists():
                target = archive_root / f"personal_folder_failed_clean_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{suffix}"
                suffix += 1
            try:
                shutil.move(str(personal_dir), str(target))
            except Exception as move_exc:
                raise RuntimeError(
                    f'Could not delete or archive old personal folder. Close files under {personal_dir} and retry.'
                ) from move_exc
            print(f'Old personal folder could not be deleted ({exc!r}); moved to {target}')
    personal_dir.mkdir(parents=True, exist_ok=True)


def _manifest_top_name(source_value: str, source_dir: Path) -> str | None:
    if not source_value:
        return None
    path = Path(source_value)
    try:
        rel = path.resolve().relative_to(source_dir.resolve())
        return rel.parts[0] if rel.parts else None
    except Exception:
        parts = path.parts
        for marker in ('01_미배정_신규자료', '미배정'):
            if marker in parts:
                index = parts.index(marker)
                if index + 1 < len(parts):
                    return parts[index + 1]
    return None


def _next_done_destination(done_dir: Path, source_item: Path) -> Path:
    destination = done_dir / source_item.name
    if not destination.exists():
        return destination
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    candidate = done_dir / f'{source_item.name}__moved_{stamp}'
    suffix = 2
    while candidate.exists():
        candidate = done_dir / f'{source_item.name}__moved_{stamp}_{suffix}'
        suffix += 1
    return candidate


def move_assigned_sources_to_done(source_dir: Path, done_dir: Path, manifest_path: Path) -> None:
    source_dir = Path(source_dir)
    done_dir = Path(done_dir)
    manifest_path = Path(manifest_path)
    materials_root = (ROOT / '20_기준자료').resolve()
    if source_dir.resolve().parent != materials_root or done_dir.resolve().parent != materials_root:
        raise RuntimeError('Refusing to move source items outside 20_기준자료.')
    if not manifest_path.exists():
        print(f'배정처리 목록을 찾지 못해 원본 이동을 건너뜁니다: {manifest_path}')
        return

    success_tops: set[str] = set()
    unmatched_tops: set[str] = set()
    with manifest_path.open('r', encoding='utf-8-sig', newline='') as handle:
        for row in csv.DictReader(handle):
            top_name = _manifest_top_name(row.get('source', ''), source_dir)
            if not top_name:
                continue
            action = row.get('action', '')
            status = row.get('status', '')
            if status == 'unmatched':
                unmatched_tops.add(top_name)
            elif action == 'classify' and status == 'excluded':
                success_tops.add(top_name)
            elif status in {'copied', 'skipped', 'extracted'}:
                success_tops.add(top_name)
            elif action in {'copy', 'extract', 'extract-member'} and status not in {'failed', 'unmatched'}:
                success_tops.add(top_name)

    done_dir.mkdir(parents=True, exist_ok=True)
    move_tops = sorted(success_tops - unmatched_tops)
    moved = 0
    for top_name in move_tops:
        source_item = source_dir / top_name
        if not source_item.exists():
            continue
        destination = _next_done_destination(done_dir, source_item)
        shutil.move(str(source_item), str(destination))
        moved += 1
    remaining = sorted(p.name for p in source_dir.iterdir()) if source_dir.exists() else []
    print(f'배정완료 원본으로 이동한 항목: {moved}')
    print(f'미배정 신규자료에 남은 항목: {len(remaining)}')
    for name in remaining[:20]:
        print(f'  - {name}')

# Run-all defaults. Change these only when you intentionally want a different run shape.
RUN_FULL_PIPELINE = True
APPLY = True
EVIDENCE_RESUME = True
MATERIAL_MODE = 'copy'
ZIP_PACKAGES = True
CLEAN_PERSONAL_DIR = False
CLEAN_PACKAGE = True
CLEAN_ZIPS = True
ALIO_REPORTS_DIR = ROOT / '30_검토산출물' / '01_ALIO_원문PDF'
DOWNLOAD_ALIO = not any(ALIO_REPORTS_DIR.glob('*.pdf')) if ALIO_REPORTS_DIR.exists() else True
REBUILD_EVIDENCE = False

print(f'ROOT={ROOT}')
print(f'UNASSIGNED_DIR={UNASSIGNED_DIR}')
print(f'ASSIGNED_DONE_DIR={ASSIGNED_DONE_DIR}')
print(f'PERSONAL_DIR={PERSONAL_DIR}')
print('Run-all defaults:', {
    'RUN_FULL_PIPELINE': RUN_FULL_PIPELINE,
    'APPLY': APPLY,
    'EVIDENCE_RESUME': EVIDENCE_RESUME,
    'MATERIAL_MODE': MATERIAL_MODE,
    'ZIP_PACKAGES': ZIP_PACKAGES,
    'CLEAN_PERSONAL_DIR': CLEAN_PERSONAL_DIR,
    'DOWNLOAD_ALIO': DOWNLOAD_ALIO,
})


In [ ]:
import base64
import zlib

MODULE_ARCHIVE = {
  "alio_disclosure_compare": "eNrFPWtz5MZx3/krYJQVAncguLxX7JVWDH3kRefcK7w7q5zlCgXuYkmI2MUawB5J8TZ1caSUE/mDU4niiiOp5LKqYidK1ZWtqJQq59fkm8j7D+nueeOxS8qn5D5w8ejp6enp6df04IZZOrKCYDgtplkUBFY8mqRZYYXjcVqERZyO86Ul8Szbm4RZHon7fv5EXKa5uHo7T8fiOpOg+bEEKOKRfDzNkiTe9aMsS7PSM6Mn/iyLfjSN8mJpiDT30ySJ+kShIPpmOh0XUSbej/vTLIvGhc8GJ8Ee7WdROHiQpsnWUdSfFmnmWWEe9NPRJImKaMDaD8Ii7CdhnquG8hGHiPrxKEzE201261m3x0/CJB7cn0QZcZBBT8JiHwYhoB/ALXtRHE/i8Z54fhsGEO4mkceugLglBpZOovHk+Eh2l6ThIDhMs4PdND1YWlp6eHP79oNHwebtbatD2B2Y1TiBOXWBcXmaPIkcF9kKHFn63sbDLQ6q2vGXVjzUH47DUWR1Opb93VZw9v6nL/7pb0/f+/LF3315+sVf2VaU5JEGvLS5dWvj8Z1Hwb3723c37tz+i61N3ot9pRV89eXzs0+fnX38s9Nf/XS1dTX46vmHZx99efq7d403tkSyvfXg/vajhwLDVcDw22cv/uaTsx8/P/viw9PPfr/aWgugl/vB2b/87PSzLx9s3lKNb96/+2Bje2te4yu88U/+HXv+6bOzXz63l7a3fnB7682tbWz4EFo69lfPP7A9yz77hH5e/PNv6O79T2136ebGo60/vb/9Q+htc4tBX19Zw/c3Vq7gz1qL3a5dlb/XoR1QGsi2dze+fx9pPFmy4B8haMOP7bF7xNTmCNkTQtrGX/nkKn9yVXtyXT6ZLb2xtbG5RePhvTzOo2xlYw+mG6Hupu/ESRKuXvdblvNmPB6kh7l175G11vJbr1rw4Ma1V62jG9dca2MCa+TNaPfP4mL1+tU/9q/eED1uR8MoizJEt18Uk7y9ugqrIPX3Uv8gWyUqxNzcuf3wUfB4+w5OTA1smu2FY/Z3M877SZrD6r0T54U/SO2l2/d+ANO2Gdy6fWfr3sbdreDmGxs0suXXXm/bqzs7T9cvLS/de3z3ezCJ21vwIot8XNuwFpzMdtZf+1Z359DvuSvrznp7Z3Cy5l2dwZUHl1dn7uWnO4PLLr7x8WL9lXWcrzfu39uqQQZNrkDrFWzqXaPfazOA34S5hVHWtrnSwlaznc2TFjSh/st3gOGHWxvbAQz0MYnVSV5kznEUZq41TDMLr6x4bGXheC9yrrRaLc+60rq65s5AD/yJ0lL017o3He1G2XbUT7NBm+YqT6dZPwpA8URtC1DTw35YRHtpdqyegBabagBJ2id1pjVJQd0eFexBTc8oX/3jW2kyiDLWcxY9iaNDuFM49qP+QTBO1RNUOOoO1WabacuaHrYj1IW3gLMMfzjZDYN4cC5kg2iIIxjGe2j54Ap0JKhIa+V161465ghBF+6HeVgA/8GG+XkxSKcFLOQskk1tl4ESZyWMr0E4wIZ0AEq+Y0+L4cp3QBOQycs7gGeShH3A0dAbwC3sDWAu3Bsb/iDO4cFxgHxxFHOsp8gx4gT8sv5wokESURARUNKb5spCoDKRxME7bAMEhlmRH8bQg70D/9Z3dh7fu7mzow+EiQYY6bFFMLZ1mRp3v9PunQdhAy5CcY2j0J7x0Q/zCwy8wqoKB75V5oDeIwnnLkjYtABeYRvE4cMjQogwEqGAWzRSjl+AL2xe09acEeC6aNy9YrJN9q/BcDaODgZxpjGxtIJgqKPwIAKY3NE57oJYHoFSD9KDzqNsGgmZzMNxXMTvROS8IGMdpYdKM4N+EpNJ6qIAfZg7J32gww5s0pT9fdST9SZj5iotB1jo1yccCahCh7CbIKDH8+ku6v38MlpzC/7QOxd4ncUT8K8ydmH7Fl/SnH8MBVAElPGBjtMM3EUc6UF03DhIk7x+mEdD0KfYE1vLjv0/P/kQibGbaHXAmJ396vdPzz7+7OnZjz9zd1wCF6TX9CJRQ7Oz9z968YufQruaPsrQH392+v6vT3/7AXh154D+8Vxok/6vfvef6CP+/L2n6Cd+8sHpF8/g/qvPnz2Fnxd//QzfEwg8crtvOW7v0o777eo4+WxI5N23WivfDVfe+er5s5UXH/+qZ7Zg80SRSEC2blwEqAfSPEZDWCehoEFQ9M05jHGyC25PK0vQpiejsABhZeMG897fB+oE/p1LncePbq18Z3nZ6b71au+yK0j0EPy2VByEo9KBHlD50/GPpilIN4H6e1k6nThrrttEwrIgoWOvQ982db6+/I10r/jB+J6RbQ+w9wAjIMVucUEc98gdasM6B7v8oymovIjdmXPAjJQlmurPlKj5k8FQSA2+9KxhEu7lHTXMUpOhHayfYP+z0/fehUve/+z0i3dBGHVUxhjxAVcZdmBJcxznAUWOgTZ0Rw6UxgPBXmIIFb0u8/xWCHGZgtL1DNGCSvEEKQNv9MoNIH2NEWzP5qDiD1BXc3q5t+WMycOsYTpvMrRvnnCY1rXBTEzwMAIhCAhJAobAwT8BiAtbSB2rHC4QbgTqDuJ+0aW5hz+9nuiMsgPQ0EwX+NvsV+L3rH2I/yNwiHhYxOYGDWW5Kdxi3O3we4+yF+Ddda62XCvMoU/QBuNcmwGuJ0hG2Dsfsw2OqzsTHMgfgNc2AF043yusUw3INQyJ2jv5pe8/vH9vhy2wHWfZWX/wGiZhXvcvr7vLpOyxV1qrD11ddMrrNYwxmD/Oi2i0dRQXjn0znSYDAh1CTGiJPi3sEEWoIUTjoh4eBkgHEK6vdhufcRD05uE1PvExnZE7opGxWronNNzbA4gtYyCuK257IMB4eS803sBtb0bmH58hodQRjBrv/b0IhsYxuD3hCaeHY8qosKXn4HuuXg7G8C5YrG88CwQD7EPG3CC4fRJlhxn02aZl61mDCDzINuiUNGQLxZRjua4ZnJwYlDk/T6Jo4tAbxhwQTdRndQG0GMwqqDMf2bnOhts5wd/ZH8GA/pwR3pEaCx/+EAbWYerM5hOQT5NCZg0ozueLHjiOV556Ae5mMc0x/re1pyHFgMS+8iuhi8vP0TssP9s9LqIKbpBMvfGMLbHsWFdiC3RCszp4eSpBUwsUc+u6gXfNhJIDrSCQ5hKJf4Jf0P4cPsn8LrQmtmt2o80YUbrICrMFIdeCi16uWjMGar7g6zRjdNSPJoXl6ClhH5T+Fl541iPGZH53/yFdEL+hoT7fKLD+dDJAF55JZMcmZGiHQV46GMFCG23Q0inEtkua4HeViPbKvoOA0OUbgbR7A44JcI+H0IDIQWYAFWLR27BYbVK2oLB0cdGcfmQtQmBTPbzbtV/B5Ke7iBHQNsBuOCv0XprYodmLWvdEG+/C7neTcHxgz2X8MEyS3bB/EGjCPrRPlDwhC0iLBcrvKrld6MUx/ZWHw0gIcjWulF0Aykq/3EIx5cEVu7WKtPDeJW5DV3aZ9uqVMwaIyFX6HfQIBb85zGk4ZjZWmYtF4kwt5/ORBeXUEcvqaz4OqS8RihNhoGsOd21aTvtATqKRwO59IoyJ7FIzZcLwRAO7FG8xykxTC3GS7jp+04aV01Jn6ZUT6zKP3dBosn+tZ96nK/OIoxEwKhDInTDby9tyx8y/B3KST8ClKydHuGB1LEduyaxis9yHV9rGjTapvI1KFyH0bjYtQElkfW3qQD+Azs8Bue5BVccrXSjbZm4Te6UyzNQDqRuP9RaByF621lyWIsJNoLpey/49NRVOuKvn9fibUVyYVlPh4lfdtgarUoOTDMTEGdp3SPasE1SvvIU7s9jQc4u20U4MKjDVzoQjPYT5qgsukH09tXiqm4cQzB7RRhy6EISd33DzxKDUwMRuJOA1BisgMbKE0TnGS1JGpuR6FQA2ZM09boBQLjBMO/kZVUgaCC2D+ldibVTeirVS30wtoPr3tKrMdyUnCOSTjQLlk0+yhFAiQe7/eBAdeZzhCB6BZOOmKoi0tt/r8BmBtbBWTiSnhyjI9N5nWsxxyxC5H05ApQ4cuDZfYsiBNFivsNFNsnQPsORBBGw4xpx5q904zSDIYLWmI2fNiGWwQxHLdIXT3WMRvaZ/YVq5nZi5lS7Eejkh6martGAEF2ZsTyYaUJYnHnAXEJbHicIvFw6ovXhYtpI27ayevv/p2S8+P/3lh6f/8GFw+m+/Of3lR34/f8LMJQlBgGstdwQOjwbnWY4MMjwZVnhmKOFp4YPHQwZPhAkeDw24j9fHsgBcb7w+AOdJ4xwyF+eZ85a1ERy6y0nDwev2XJDszrilQywMpcc6RHw58A3kixHg44yBwddETM4DazmDXgh2plJCQC/5JBAbp2khLGA+HQ7jowjmJI9IWfXIsIiSgS5CcbMHqPsHqGywPVdk+4CQvdASF6xYghxUeOFP0okm6kZUpTwJMAt9sHtglYQ/wdG4TPmNiywGKisC2AcKBgCHvTEecVAIt6PjThKOdgchNQezTj+0x6K5woo0HjnwmKAadsXjaWSoBcJH043LMMcJ4uSUVj89xnQNZ82qRkp5nbM34CITM9IkSQ+D/HiUxOODvENJNLfKBsZprj2oN3OtRomOGcWgATU5kYTAZ6KhBw0oiEJeKiQcx1HCmxr5bu7jU59qb2d+slXtxvI0uJF6pQ0vyTpa4Vpmiy6dzH7LuUm76G4gc7Aq+4qNMMaSbWUKGTzRCaxsl7wJhZrqU3iGPc3ivXgcYtYE8XZR5yGoi66M1e6JpqyRIvSc4bBAXw6Hv3YERd4qclH3XhWLhePaoTQM0dvRUHnE7w7bqWQTq/kOoE3Yr+5Wz5ta5RQpCqRfzc1DLQwmj2ZSQyI5KI+6WlOEeGDCKHibGYygcqdOg1zWsltjOYNfoBU4DT4oU2BUiFbezJozAJJe1+PgpQiHMBhLiM/GkOof2FzrK8mohSAm6/USxgqqXSfrD14bp6/zAhZ3x9/JL+Gz3XRw/Lp/mXa/5JKbm/ctSRoieLkbcEb+F9GDZRY7pjoT9fE7gj0dceHJWpEOeS6FsYu0PE6XwbS2roJrwlcD9tS0DIy5ydVsD9SCUNsNOl1c6Kk9mrYaCBUxkGvByUexN4rL9BlgIDwUNIkBqyPeV2VbtROmx10g6k12W8c1x3gb9nOO6UZfh+XkK4a7aoNIK3RMNjTaWm38X8vm1nLFzH1KZdOwgtXadetIY63qexZyI4w+gzVWgQDh4ipKPANRoBXk+xHEPOWKC+EItqtZ8MNdGJFRL1quxMDETpCOk2MqxcDxUX6G3xsZ2i36wTqwsgYBChzhDE/HFL1oTyBsi4uEwrDDXZ/GgAzV5IsKNtIBgZiVle2ytyXACGWV09S/Hw7Q9R5w8ksc8XHTKnIWDW2CtWb6ViZiFvuXED0JERGMzYU5N9WIMo2aZ757HKiiuxLEQrOpuiGreZTkR6ht8WJk2M94qMyAURL0l9+2FykLTl5M2Yq5guias8ghKJUtcZidaaPX7a54xhS3sTC0FiLtFo7TcdwPk4ClrFghhtiDNkwosIGVlsQ5PcWkb5zHY+DIuM8rODxKJ85xwjDsrrRBF6m6FgpBTHNDlq5UTQdRH3e9WRm5ozC4vrQITiXpO8RXhQNtMSS2VQVSS7v2bXKM4RdnpNyEObzlp+WqP7PQSX9LVXmyrsdjGSX14BVtK4vX8Kn6g5WakgPJbmPJmtyhgrkKX/g6Lhfgz/Oqx6DZKdNSO37FNAGp45JtxWUT801PkcEKEY6ShFmWLD3k3nc/TZorWHg0w2L4vqgGYcs3Qc09or2HJ6MUtV9irVhrnnXlhhIcjqe/nzk3rkPsAy0wBNJ2rHjRxgntbpwAXbJkA3NmYb8IeOUvEwpwgsSphW66+3bUL3qexbK5OAroa+1GyxzLBNWQ6Sfx1clL5XJDgRkrd4HOqoqslMSdMavaM6VYk0sTNREpDLWq0ORjx6kADWP7b6fx2CFYY545WLfNc9WMgfAkQwaipAqtlRthgVGHrXxQvYpbBl54I1xQA0BLWr8U248ThGlHst2IiKl+Y5Kk0ud7mIe5X2Oeq7EX5jM7NAbnkJJkWYDPuGxplJg2JoPY98gTmTuV2WUpxHIil6wSayEFzWhTbUJrhe2EdJosTa1nyndP6qR1oQOaREOqHUoPu6PwyGl5jGpYxddcqy1v1npV13U3fRKxPN9h3o17XQVLsyPurNcszHxwKK78ySJSllnt+vD+M9m/uhY7P2bNNUmjWC+1w9ZltB6idDwBNzthodheI7CQuo50HBpBaco6bHqaocQxBwwypeM0a7dPhDzPvnWiKW0mU8RddzaPTqYOOmUN2r1EE+dZl3Dqe249hqqgvUynlk+dSF4MhqAf9mDpEYGlMIN0TTGdgK4nW6UV4BnE8FNqw7h4xzDN9JgytdXijUoB2oPju9MHm7ccxOLiisKimxhtLsoqvKE4LR7voYND23uAkO1LDNI+lWwU7/jGfjcpPZ7jx0HiinHYJs1ltJT4EPfH2OBt/Gu7rr6dhBCm+oC++IKAK2MOOIOpJ5F+iMdRMArfTrOAiv/w3qj7Xpj5wUSPOLp0ZYZnlVbE9TrlgS7vPLQ96ugieR9+b+ZVRY0/0ctVH5KtKAYpMPbXMTGDEH4+SXAOV2xUrd1Wzy2ZQIYzZ2c4iQ3ClJXFy2P9lyxivRBShAMMk8PipKdUh6iNgm6UdeOT65Doc+aZs0wUmoFV3Vy6uK3H6F0q5fqJMOrIjITA4h6IiWJgFeMhSiF7TLojCq9RjzOqzGEoHc6wkWArYFd3GQcRGd/qOMA6IK5ed61n+EnUBLcbKjLhWq9LbptjZ+TOGzkfH+uYqG5DI+E1gbc2CvLp7vmEhZoHUko97F7d6hX6F5Gjry0kaukOae2ewIMo74cTPj1Emjvr7vhubycXC/cbEB6ibY780F4T51R1H/38QrV43KKb8qh1mas6Tk1SdBFJEsoHjRyXpoBAz+OD10sKygJZC7QlNcaTMXeeyJqBEOEbp56MlFUH5rQgSkqC0ak41LZsJCXWERhb45W4R9NjVX4TNuHPOZIsEk+5+w/LHc+bVw40dwUP2cA4rwF0jvrnel5Ko4wo8HAhO+Oszj5KhGXlwH/xtDWdcODhN+2lljFevyDG63S22wzo+VtxfG0fi+aD/CCekJDhwEo23jjDIY4Wi9L+0uLXD13wFvrh4nO2Ms5yVBfC/0EYSvOrzX79+lPdVg3zfJVrhm/xnHlwz1EhwLwu6EUeI/cx2YuxaR2OIj2IyuccWuYiE8faKnEkta0syEqqUgat2sHwmrKKurByUWC2OCgzAjKqH176A0IxFoaxHOjSRSKwyQnKwKyNM8Arl5oo4cEWQnbbazdaNaVwbk2RWW08pFcqGfu3dSWDPP8FXsgQCyvYDodKj8lKHc3Dl8dnF1Tq8nSNfYj7m9Ehjq2DOdTSce+VPN6rL+WlgVAyI3/ibwLJb9IDh8HpFHcoG6PutdQLQ8LqgdnJAqeUl1GlVKWKIa0lvGXVcjqL2YoIOPebeG0omBIvy1PlUTbED+hREJTp62GpmSbcqNyF7OI1ySheCIkkACZbWFym5USzSNEN2maOTvQslvubA0KjYk5Co3g1vBDe63Q0CrPjgCfXuBIexMNh6VF5t6Zp3w0HxRJyPMXkM42kcVQbN7l/AKObW9MfECmBZBFWg1nnw4uE9Fmpn1b0p0bgliiowuqUKWhinySXb37jFqfWoYufEIBHBmbX2Brnh4UdsZXBMoni0LFb3nzDPb7hkBikmCPNgU5TPNSHzqB7+FkCgxj+XLFLl5VaI3FS0Z1qlbStZmVvK9+CdQ7QGDBoM1HTSM+MG82MSalpiKyIUIEyGKOtyctS69lS1SYYTDZblzYJYDDRUYilxEKIv8aioIBayX+Hn5bvddtXe6UicWTCnA7PvV7O3aXUGo0exEmtET6fkBAkU7Rta45TYEvRBriqmDc00iQfmtWtgzm9CS7jyQjrqdzqAa8E2SZsAhbtEo+ZXZjZ5ekXaNy5NL6EzkxENd3Nml0eXQl4aspLx1u4qTvv6RattK/uhAt/XT7lUip/qmmoIMpt/6DzNDrNYtOsplzS085GGGchXKMyTGCYW2mmHfCGB6ZpDpMk4NNiPpT2gCcDUQR44VKsyodqKihFmSMe/zDrG1lzVip4rhJKneSL6gV1UHgU55i9D5pjClvU/OHJJUakeNLQwDxwrA2sAV6UcilgWXDY0IE8AG3bzdpj3utyJU1cOdpsgJeOOS9a0ZVgUPnuokiJdkaai5d8s6Knhlzc/Pa4iuKOkESiCdBCKalxLTTpYK1rhn0hqbiIRFxIGjRJ4PXB/EG9K2S3zSOhvA0L65bOKSQ1T2vaNkiM7ugY3vK5UjjK/Tg3+HmjixIdPpg2FJW6HJXGNS1bZEq+YJJWtMaUnpld0tScAKtJOusjLhNmVnEIJAZdS6Vj9ZMsGsZH7FRzSXTpUHH5kLKumflJYxXpliJm85iy3t8seLB5i3/oUh3E8nSmu18f99nP//7svWc16A3mueWIg/kZOX0PpTGEbkSiOT4cX+XonjieLOvGy7pCP9zR0RWDLPDtlLWAOD/bMZd86RMKynTXng3UaK8JKf6/KSeSDLoryZWl0mlPeeYvYEcATv/107P/+g9dFmptgnrqaOf8pHKvnvmT6tmz9DOC7JsGtSpTngVkfbkXGAr7JOzZL/7x7IP/Lg1Cm2B9DBenXE83VcJlrzYa9ppC3a8xRLZmz57/+uyjz4OzT949++3nNSMlgXh545RpNRXSeWao5pUCMK8SJRljFccot7nvzo6ottk5cBFliIOaAnhD+On8tCsHF+5zBf4uk1sAU2dy+Ro2/HcWcneXmTwv9zCyXuYQyxWs96fFZFo5XSpCEnUOVERg7FwCfjnhnDEY25Ck8xX0UWh2DnkwfKmRk7bNcPHU4jmMvhhExeIjBWgqy9/oqNpRgcLH6iJ3JlCXDyTXWD1E4VmG/RGz9wBZPxByxgBmvAAwr5tUxKXN6u40TvicZuzLs3I+N7K96QhWOHWhqmZzylM2QOHHPPpZTNVeHXuTH9a2cKWL83HJMe7V5uy8KLO6VrEfxeKjDznbeFBRKsxcPuE7o1hKltBncHJR1JxPdxlVuTg0k+G5i0A9py+MdGwuwLYna7jEVz/1M+904l20JEScOfJou42fa0om6lsjbHg0qN1ji76DsYIfwsBvqt9swT/fv/ld+Ce+OcabEfKQs8+xV1Zgemz8qgcdQ+jUfIN8QXvMB+C31nCTjLb2BSr8xN6CtnxyapuvLWhLJQ61LVsLWkY0HdV2yK0FTdXnRgBD2GcClxcp+HBFRsqdTRKeMjNmAcWpiGDW0qE1xdgfpQ9uhnE/DhNWWye+M7doxjD6WJlmSc20yU8GLsIwiht4VxEzrEMmYodxlhfWPaKVaGCLpIiLKf1vBK/CwKLhNGFnniL1qZ5GMvj3QuqlZ0Fb+mCGaEknSrRB+K3rC5qLL1Ks0Bcp6gWwtUiQ5Ec96oWh1DqPQFEz3LkznI77nfLXc7hSkLauWSvQ7wqASImjJ1KZsf85ATAIJSHMr0BdGgs6kvMhmhSF9v8NlBE0D1jC8BELjdw4Xg4gRysaaN8MUeqwrMk19W27en+VIXKf6Tz6sAmF6rcGi/nfQ8xHdE52NzU/h1puanoOrSyaNk8xhyiVA+NsytLaeFz+6nzNZ+lNB8B0HJbE53OUDebnV+GZ9tpHysgDRAcEHNWAfcggoHKoIEBSgsAWlcdI19L/AmsVGBw=",
  "build_content_review_list": "eNq1PP1zE0eWv+uv6Jv8gGYzEjYOe1ntKncseBPXbjBlzOZStm52kEa2Fn3VzAhDWa4yIFJeYCtOLT5MYlOmzgkhR2oNOKzZI/+QZ/Q/3Hv9NT0fkmySoxI00/3e636v31e/7qHqtBrENKsdr+PYpklqjXbL8YjVbLY8y6u1mm4mI9qchbbluLZ4L7tXxWPLFU+O7Havu5kqUi+36nW7TGkJ8hW7anXqXqVW9hhMxfKsct1yXTuEEU0GqdbseoUBti1vsV67LIAuwCvr8K63a80F0T7l2Y51uW5nMpmLZ2emLsya56ZmSJHCZ4HdWh2Y1fOO7bbqV+2sngfO7KaX+e2Zi5McNMTjnaRWVRubVsMmxSLRfjVmBnd3+//1F//2Qf/Ogf/qhkbsumsrwJnMucnfnbn0h1lz8o9T5ybPn51U56NNjJmHL1b7n+0EN/eCV1v+szcnx94zg2+2/dcP/ZcH/rMDs3/7r8GdXSAfvLoPIJouKU5fmh1J7LTJyLAOM1j73P/mGaMHlM6emZ38cHrmU/Ps9LnJi0Aoq53OjWsG0X6ZO4U/42PsdXxC/p4GvAszU9MzU7OfmtMz5yZxAsvaR1MffqQVyBgAfTx5burSx/AyDi9/mP4Enk6tZDK/n/z0E4A3P5yZvnQBB1vOEPij+a/Wgt5zEOThyx+Cu1sA7dj5cqvRhqXKOmH3vPsLBtGNYXT9vx/wp/6DtWCrx1/YT7C563+92Q2erAYP9vgQusGGDtb+J3i0Hty6EWzvp427/SZ4Dai0vxvsfYs//t1d/5teF8hCA7x9G6xt4tPhwVqw8RgmKcC3D4Kv1pV3OpZ8l3Pw13r+9o+J0WkrcubvbQabN/Cp//mz/gYM+XoneLJOmT7YCx49w6fg0W1/5+suw/Jf9RQON4OdDfZffAzWF4qVvYsXitIF+kLIL1cPX/wYQrMf5IeihSMy5X21Gjx+nhiR9iHOg8+68oXBKu/Az84WaDCMvu7/9z0BL17CfvkkR2d63t/oBV/ej4/ef7gBiwIEWHeXwYbvwCsD6YKpHe6tYg9//+ye/93tbv/Oa1i54CYsw/fiZWc95JxqWoJnrn+bu7BcVG1u7oHwevCDjPFekOHdbfmOJFcyF2fPzE6dBds8Pzv5H7PmzCTYjEKaDurEzQdoZIP97cP9Xv8LEGAPvNKqf3M/+PKp3tU4CkzkcB/MYT3o0QV89AVQ6G9sgo1ImP7Gvn/3NfZu7fQ3QQo9XHqmcJIOHREa+xtbYCswUr+3J3v9F/dAOsH2KkOEEWMTQOJUsGx++EonLsFYb3+DrtGtVZgPDIRgN5/BrEKwnXVYysMXYN1vEPKzH/yX+2jmGR2iwL/LcJKhf5MZ+2rNXoI40SgwAvTddgrE9RzaUl60y1fMZitssRbsZvm6WW3VKxFAy7MXWs71sMUttxy7QGoQNIpkjNO3XIiAAGN7cwBXgh4a17I8GJpVq+wBlSIA6IxKq+OUbbPc6jS9KDH7mtVo122gVq+5I8ghBKOHUe/IE6AY0EPaTq3l1LzrWdeuV3WS+wC5ZDLDPxAUsSNPWSYfFMn7Y2EnYxxSiyZhoWEw2sTpdDQeRjLxdowosK44w3KrWa0tYP4CTxDQIZ7jNM+3mjajCYMtWq7leU4WspK861VaHQ+ikmNLVE0Phw9h8gpEFta+VYEco6h1vGrufQiEtuO0HLcIdNp1qww0BowGcCNHA5hjj8bYr9RcaLhuYmqUxb8KNBMgXVyo6IJ59jVUIningHK+LTdMZ5qeFlldxIEJWo7nLtUwwZiHP/82P3/p/Nn5eZURdXUQRiPvUuS59wuloxAcQIuSeI+TUNo491X3GIwnRJWQwL/EJaCOSH3AZdCwjgeyQhykkYcmShBhJEEBN4pTTl+Aj0RPwY2uCEhdIM+diopNjq/AcDE2rlRqjiLEmAUBqw3rig0wblaVuA5qeQ0cjNm6Upx1OrbQSZ5YD1kafGW0r1r1ji3S13BR+Jxpb5ioo2xYU801BQ/QTBPurEzfTzIgJcOXE7MqJuxc4qxSP4r7EXSNBs6yVJLOA/ZCcrEpuzEhJJdkjskdl4202nYzLrSmvVSvNe2ihnYdtfScW1vQdGK54EWalbqdoI1TzQIL+XMw3Rngx3ayDFQXXC6Bw7ZjbBrEaS2JiBFy2rr8Z9iclUp8l4VGAEBi/0QDRUwXQlXhGyN9OK/akvaWDFM2HFANwe0ntIFzazBJhNPW9Rhmnv4sMhGld6JMsviXEF25bltNE005S1WowCUU9SdCC0nNVSSjmpoWd7hSIan3BmN2cCtFNLWpSZtORhuvWWMcUrULSAHdzmXILOfdd1m3QUfTwWM4tbZUeHcR9sPUNdH0BIXWqMl0YuLUWKqjVKQQejXFFaIA6rDUtJP8psioMiukDrvAW3JkogTeRsvn8zJc83TJxH17OAafXtWq1y9b5Sv0FWaiadEJYt4LeQpLgiMSEAJguU7LAdAKLFCTRPe3keAmQDjV1ACEMGLRteDuzuE/toLnsFV4qqViCgXALXSI9uQGbAOTaASmqQWPeoP6k2RxSx6SBUxAe7Dn7/8lSRZ2mNBPQYbTpPv7kOjmGpYObt3wX/7gf/Os/1UvuLWdoK52Yub+4PsRY0yoY4zDfgoS+UEjpKKf1lQtFHoilIon60ypWKKeTebrUV1qWF55kWkSfQRd+s/sfGV53JhY0efzoFARCpTvqBVWtWUwpCzFzi84rU47Ow7hYGyisqIho2wEahaa0P8r9vWlllMxKbgb6n4Yh9DpSj+T2P/lXdtyyjzdGBR5xBtNatAY8MHA8hn4vCaKOlqIyYMvbLgsvnKgyDglPnl8od6dCbphNWtVG7wwuNACicZPcPRXaxWQn21CwFBiLc8EQr/pWEu4Cq2l/ILtZTU5BrVqXY3CAJngGOnQNkQReQQAqrndMUI3NjMXwjUMSKqMoHvGGaKfpmRRsiOGEqRShpN6HOeIyduqVExcGrbPp4vE5ex12iw+G0QRebitLbF6RNrCJHetRmzbaih7VgXgF4a6/QxdtCH3lyZLOMKOWPIgttnqeos2ZbmjdqfARjoUBGH/GLtGugKGAnaItU4xuiERDSkbXUodIKnwwSg8vl/OyqUESoZ8CVcgG4smbJyiHDDSLQYvyllEuiPzL0beYnT41IviIezmFaqQKb77frfIVl5pVyoP2D0edvE6Rh4UM8uepZ1xtSCQndHkgMKLUgVkCUTZ4kf68lYb8sZKluUqvFWXdEO9iqLTegadiIQQWQ/4pY5VN8swlVoFBGHSPC/pj4zjmdSIrUJqgk37QX24Y2aidbFBcpPFIqz/+UNWZzRZIbJ/716w/SYPqS+o+cRpg0SACC9u6oZKZeN+f/Og//C+6b/cxyeVxL9iUZ61n5SAhIFEycTOGujpxr1vOZlfIhkJQRgIYSBRMtNnZyCyw4jmhXO/49inkA/oIKyHQI9A4uKx7SYmzGDrYSKHy8viF9VTgzsmjGJcnKFmIDysA/bJvV7MfQtysXoDV3oeRaTHUZoVf6OYPjoRZYYheBSyRiM/nTPwGB2aJbZNr9bs2NE5ASjVcEDVU+0cHV4yoVZmYYSsCEDGR4weDM9yf+3XkK7/uVVrZhNzhP2el2hEiWMHspZE4XbBZelZXsdlwxvDQRu2t9iqHAkUg/FR4CCBsAcB6okWmgjF2I1CydhMfQiVcxg6YrpqCO9Y5II2QrdWHKgz6FCEc0xMcDmV2zCaFgZF2HQxaSLwAGJqCB0WgwfRjIIWhsbxQdMSSlsgyZAW4xwlTU9eqMjTobgOFo6jlUIbC8fRT6qXhSOrKVPPwtHUlXGieKaC6niS8CspaiwKCaBjsZ3JYs1jAZNle8mo+bbJKM8fG9Y1HAOSY9uhZ/E85/wZwqvYkNBdXbr718TJ+rODoLcVP1n3v3vqP96m4SoMQDygCOKh85ZbFRhvyNYouhXS49VwRiFRSkqNCseOUj9noKCLBgKhiyZPo6ggYCMRmXiyHCg51RMlwLTTjbSCoBpysJxIk3V8wNWxm52G7cDMZX2Q1s6L43oh1X7Y/hu4iG3IkZ6eisE3oQwunejASJ4uQplbpwwVhfwgxWoGzuAy6P6V9GDIMqcipoJ4I4T0b+4GX93z/7ZVwFMBzeChn/GYLga2ayiS98ZoLSd+cwPXgsuWFj5OjaVSebvoiYtzpNB5pBA6PJT+5JD6/xlaf8YQe/xQKzEipgNocf0Zgc5tWCtIax4xQZY7YbSjW0VqqiNwjhElI3hhFaoQPTdU/NhACiupPaGG2tfKdtsj0xcn0e0NcfnJIE05xyDIQvRP3r+KA6AwxobRFP0srX6ASbPyBz3FcLNqHWuQgSWNShN3CUCkdAcv7xakiFKjHkBAMneQGWaavDyRVlyJGyEFTS+0pNoUhR9SdIlbD6M/0IQ0tb4iOVTaUhmldRe0Mbk/c0ET7EpWrcukuQtN1FkQmXS7Aj1an0nBowWWGJI6Ju3X5wrjY6UYeqj/ulSrPKJGimbFutW4XLFYjTKqPNG7hdSh4SEaQM6FWgSjkl/FRs5hPZ6CMf2Jz0xSkZozEEIqzGAIseQqhJ4ZlmNzLXI7jYYFmZg8fxx4MDvaZqmvtSupTgDsP4mGty5X4sktnYSamcZkTS+H0HucwxNUXhFJlXOqbPVIVQ9wOT+ppdZEuXWAp/FabVPxNjFeUnQdsyvhccaGO5oYW8NdTYzbIzgbihFtTEOr1q2FBRt2OCKpF95kbLBvqlFznisN9S+09hYz57h2cxWJWulcmoWWyG/SwHCp56KrBLBR1UqDYXsedYRMFDxcR4SFtyw6hEQf6GLCUegxUgMkjHRDmoOAdMimx2MElVUoiZBZ1ZZxEicE/onSSpa1CBah5SRroROFV12LT1WsXinfaWO9O4ulK3ETIFKhU+ySo+h5SG5qkDD+WtNF1UugCi/mghGmpwXCWFMSA4p11NQgZrBpaz/CahNrPDpdOLoZ8wU8jh0zlJ9gyMMVbJSVh2lCUvn04ycYqsKMCvZs4d8+3IeOJO4gBsX6iGX/rPE+LZpT/iJXrXg4P851K7F1GlzVMsQx1mCQ2OnqYm0Bq0Iwm+x4PLYPCun0Xi7jr2FXap3GcQnwG7qMBO7Ioodb2jtE/fiEqB+fkODL+8HGj1oobk15rmo5grMrkGVkbOXw+X6slw0N/WzmKRDxKgfpP1qDgQEFTyjVNdBTsKfPzpwUR2a0OsjOuYj/t+ckehpHj7M4VWXV4kRV9rR33gH+94LeDuFU8MbOVo+cHhuC1A3WtvwXa/yzheDRepffqu9vfO9/97TLKAIQUOrizfu1za64Ka+SyeVyBfwr/j9r1NQzOerumxX7miHUISy1IZNzhdNjpbRqG1WG1ABQ1brLlORKl4U2YZ0Q3XiLsEZoIawl4koRUIvRTETSbiKSdqORtLssrP8Ed28nSuHduxPdEwY5cfKErg6l6DlsnrwIa3ORCanrJhd8ayfY3mQfj8S74+85EmzvE1aQDrZXCfvIhbDqNT+5lQouq9gPV+Wh7K0b/Vtb/tdvCKgIfkC09cZ/DBawsUW4LTK1xc+G7q75d3fzKTO4sxv88z6e0xokcXjMKugG+eiTC9C52n+4yQ+C+eRw0uyrDmQEv9z5HOcNP7f9dfzOhAQbd4KNXpot4Uz9Vz1wFf7608HTo59TEf/eavB4j7NJgidfBLtfcBZD8f0p7cu4CZORYBQiX8b9iUi4w5dvQGzfHv7vJvH314MNfEuZVEmJFQNuydIGFjNo/R3vfLJYSzUqpTQurjRc7tTqlazlLEAoEJ9i5s/jvdc26GosEESOPIrR69hIIp88j2h1vIHQ0KdyxUH5RyLHqkCFW0/FS9JEfdSFDX5VQ97dkY6b5qX0zjxOtdkyRacL1FgJOnGwlULYSBbYi5RiollXK2Vp5bjIHSYON2jfz4UY3tiW2iQW5KSo1fOvN2UsC55sgK2wmxVqniUlo97GUOvVMtExSLJKrJ79RMu7RliwNcJarEHih09h5TRy4+gYLKr+gLmDGJeK8vxkJvmxsUGUg+DwdJezKA5jVVbfnrvQycT4ijMkUy1D1EMN8nZcRkqORpjuG0p50BAlv7fljKUi/ssey+tizCkGofIYyfENdU93HF4HbJOMyK5I4TuFRZHED1w3xlQD9YIyETU2VSm5VQNbsEOpar9ncPRILzX15NmzgP+YUiLSG7ppmWUM56JwRHzjQRFSIc9Q2RHOrgqurFACa7rjtTseAEYOQ0QoWJFxigYmEz0n+wwvGasKPBJCI4YbCXDGWYC0suldoD3Ziu2WnVob/+GAovZbjH7sklLTyzGtCH0vof8EAJiqY5VhwyrDH/9WARPNvGCHEsc7VThHOl5Wy+UERg6YwQ8JrrftItvG8RJkMe0L/qEkQTTDKfEv94cSAWPIodbkIPjk0DgFSbwwISmeGhtKpNnKcXXLYUzEe+1lJlbXAzszPadjR++1c0LqWsqblZCrxLKNlA8waTtLWVQiSAUitWniZTnTpPtH00SapqmJS/k4QOb/AGGFS4Y=",
  "build_focus_review_list": "eNrtPWtzG8eR3/Er9japItZeQiApyjJi2MdIVKSKJKokOUqOQhAQWFCIQCwLC5jikbyiLNrFWK6yfGdGtE2q6DrZinNKhdbrpDr5DxGL/3DdPTO7M7uzeFBOKklFlRjcme6enle/pne22nQXjGKx2m61m06xaNQWFt1myyg1Gm6r1Kq5DS+VEmXN+cVS03PEc9l7T/zpeuKvZlDtLXupKlIvu/W6UyZagnzFqZba9ValVm4xmEqpVSrXS57nhDCiyDaqNadeYYCLpda1em1OAF2AR1bRWl6sNeZF+ZmW0yzN1R2b/dVymykG5i46jcXlG3UBWHdLleKS27w+57rXU6nUpRMXz1y4XDx55qKRJ+ppGJxaHYbGyjQdz62/56StDIyD02ilfjp1aZqDhni80qhV5cJGacEx8nnDfDNb9G/f7/7+d50Pnnc/et55dtM0nLrnSMCpk9Onpt49e7l4fubiuamzZ/5t+iRvxRzPFg+e7/v31/17dzr//fGR7ETxYH/H333eebyh1JgBkRMz5y5MXRR8mhNA4dF698M9//19/9lO5+HLI9nxIrQyU/Q3/weRP173v9oP8WfevdwLd0JBg4dPOt889D+6D30LaVycvkQ/MzOXkc5YVub60f7B45fdT3fM1MXpX5yZvjJ9Edu7BIBp82B/y7QN09+jn+7n39LT7fumlToxdXn6ZzMXfwUdPDnNoCdHx7D+2Og4/oxl2ePYRPA7CXhnzv8CenuyeOrM2enzU+emiydOT1FrI2+9nTOPXL26+s5rI6lzU78UXF86MXUeWL8CIGPjWVYDHZHLxyezqRT8WQyYOjd1AcpXUgb8Gz+WMxhv7PEN9fF4jrPMHt9UHieyOdET9jwWeR7PiR6y54nI89Ec7zl/nlSfj04qzBxVWT36hsLMUZXVo2+qvExGeJ0cU3mZjPA6OaHyMinxupa69O65c1M4vWJIf4pTfTYYVZXxYwpnb6iMHFfbfVNuJvXz6V9dmbl4svizizPvXrgUkDc7zzb9je9grR08furf3jFzINsyZXdhEaRBuhlWX/VeYxCrEYzVzp+f87+6dzf9nQ3+wH787fudr7dX/Qfr/t193oTF+DPZjvJv3fR3n+ja3X3pvwBUql/19/+AP53b9zvfbKwCWSiApz/4m9v418HzTX/rK2BSgO8+97+8Iz1TW8FzwENnc6Oz+32sdSrFnnX2t/3tm/hX95OH3S1o8sWe/+AOdRrk0L2H+Jd/74PO3terDKvzbEPq4Tbsafa/aBusLhxW9iweCGUV6ItBfrx+8Oj7EJr9YH8ILWzxm93Oi887z0BKfRdrkeoQ5+6Hq8EDg5WeoT97OyD3VpmIFfDiIawP/gpaZ2Kzu7Xhf/FZtPXu51swKUCAVa8y2PAZ+spAVv1nnx3sr2MNf/7w484fP1jtfvQCZs5/H6bhT+Jh707Yc1ppsT7z9bd9H6aLls37+zB4G/CDHeO1MIa3d4NnJAkb8/LU5TMnYC+evzz9S5TtsGck0tRoM7p9gEbaf7J78GQDJP2qvwGKb73z/hP/i2+tVZOjACMHT2A73PE3aALvfQoUulvbsEcCmO7Wk87tF1i7s9fdhlHYwKlnCy6gQy1CYXcLlMxLaKm7sR/Udh59DKPj764zRGgxwgASp4Fl/OEjMR6AsdruFs3RrXXgBxpCsPcfAlch2N4dmEqm3hDyw6edx09wm6esVOrkmVOnipdPg4o5PXP2pCR2UKTljPRk1jbGs2IKUbJB4ZhtjIkiEm9QdgwAJwJAknIIOQHFx+XiSSrOQvGxLE1jKvWvgYmVov8aU/NOo7ycI5zyNad8vdhwc4bXalIJmjDhU9N5r+YsOc2wBE2zHNlMxqpx3m2AvUM/upZOueW2B6bZQlJjJWKl2K/NcqnlzLvN5bDEK7tNQKmBBZY3shyt5IHxCTBOaxbgClBDJmWa26HFaqkMJuJyHgAsQrnuLINRWCleq7UAr17z+iAiBMNstBfmnGaxUqtWi2W33WipvCxWqkUOoql1bpSdeo96ourcKC0s1p0h+UIjduAhYI05VWOxWXObtdZy2nPqVcsYfRvHmc0Z/gMbFysyNOjG23njzazhNsF7WE6zQc94rVKz5S3VwJA2A0vPCISDaRlVQGDA0FVGjk+YFTbEphF8lIZhnj7zs9NmMgtg0WjRzk2fPPPuuRBRlJ+duWLCEsXelt1GtTaPjhD8BbY+mPrYZVzEjCY0dq3klVqtZhrcG+hcxW23wLRsOgGqKXEdwmQkiDSsbLcCzkrebLeqo8fBNnWaTbfp5YHOYr1UBhoJrQFc39YAZujWWPcrNQ8Klou4kdPKboZJVye/5dzAVQnPBBjw63qhp9NomcpKQRxlPVyFf+9cvfru+RNXr5oJk40wpvE6Ic8ezxUGIZhAi0gc5SSkMt77qjdEx2NDFRuBf4mOgNwiSbg5WGHtFkpJwEEaGSgigggTEBRw/XrK6QvwvugaXHVGYNQF8uy4OmxB+xIMH8aF65VaUxrEyA6Cri6UrjsA46XlEbdgWd4AYVV0r+cvN9uOWJNeudQAYAL0omSFbz/baoNAnMVy2wBHvs7+WyiwNrHP2DCnpbZrGSXPAH+9WQPxGAwICiUsXCaZ5DZbTiXNgWxUDvl6aWGuUjJqqMLovzTrmXLJc6ouiFUrsgbL12r1Ckw1tmkcYbQJRYFaRpHMYG0OU/OKxLNbr7tLRW95oV5rXPfyp0p1z7EkIJTvCVB8KJdK9esE5qWbrtvSjSOW8EGD5VK+DgzPIiyb/CXgy2EVYefK7SaFO/KsIrPoLqatoBaY04wDIAA86qa0Or+cmBUSQI242DJmLk2j0KK5ulFWSYKOarTSVXP2ytTF8wUwCTB2RQsnZ6woG1WQXzPSK0BmzTItlTm30ao12o6yDPhssHmgXxxDXBdgjjhND1aG6FRkzmH3MSy1OBjdTGlx0WlwdJUTp86Qsak4trRMhOaqO6VGEeVG+r1SvQ0GkDv3W6fcUoUX0KRaoCxtSXlfm1HpTvAY+iJVAZKjiaEUw5SLGlR0xOCjybFJOodAN0pZjikDgd/gtefAHbnqvc6qbaqyQFw1a2IlxYW1dw32JMlJsvxsWEwLtcBYGgcTVye1pVEKRaxEHAeo7jRYpfFWnlFl4TnSHjleMmpMFED0mZlMRtgOFacF410UBqnMW7VUr8/hpsFHjH+ZKnfoOYH1pRkOMRqpYDG6FVp6avBLUbMChFPVqkKEESvC9G/vHfzvjv8dOJvfmlpMsTrQNwnRHtz0v7wTR0MD0PTvbSTVx8midxOSBUxAu7vfefK7ONlbN7GeQHrTJPcoJLq9iSHLWzc7j592vnnY/XLDv7Uboy5Xou9390992piQ2xjDeObtF0ktaNEnTXkJinXCVxQF24tNxyPzHAQBqgtJBdqqP0Rrinlw3P9iTS6UWuVrbG3Rn7C6fp1+58JbDfftq5WVMXtizbqaQfccy6CFtzOvv2MVx7Pjx2D1kVmCpYE5grKV6MQ6RL6e9Mx4SQvfLl81V1BQE3Jmvum2F9MjDXcENGV2orIGjWE7ebnaxBIzkAVhh/PiD0sZK+40ogJ2mn/ZkZpzK8swUtaPDztISEDZ8en0O7mDx08xlnH3g1UM5O9tdZ6t8xAFRjNurWM9gUCRNfvrtFV47SqxgJJCGTkkb1paMXq4eUGCmglgvc8zE5jNBR2p0FSAtZRuuM2FUr32706FNGFodODhzyzJR8YQtzsEYi4OgEGStUAUCg5wjykHB/KYMxBsGZBVVkBhiXpZfOLUyXjCAIvacwOYCkUyHhUDR6Yctxaw7SSLIdZgGCUhszK+/jk34VaJtFaSoj0xojADswwgIxYKjj8rkleTgOaT3wTzE1lFHbsEtjI+Y1wHVpiq9Miw8NDALITazanXccyWvFmGVwi5i6twhM4w+yTqFaqdYk0JawuWO2GWXbcJfjHo67X8CiKtmco2MUE2mJnfurUGM4I8sb5FbIi2ihcqeupfEJIJLK5YuDTjOaVmmXt5MQExqzhas+RL4tjgH7TZwFanSIl6bpFBH8RLW9giB1LaKXDmS5UKrI9y20vzUYc/+VZjThRtOOyAHQbpCna/rWnHw3h2PEBnRyJ0thSekwBe46hsGYVWkx0EsopRJ12NONqpiNcZ7BPRhcy80wokoB0TiWENyT6UroHkA+stiBHiqWMIK/rK1QAMHEbZcBBhKlo80JYOJhzw7eAhGGsdE1IwNM83JVsOAVO8NJTKgpd8wBRrK+SNR8xez7MJCct59C0DS4VH8QK9xqfEAJFGhjLBy7FSsJoNKf4WqxebkNnvnJwV0A/nViVBXisxpPhzAbglhwuEySRO9VHqUgEGHXJBagBzdu2eq3rQ/RFu/ZAGwnApIEWRI/Wq/Av5ZO5lyHVMpZHpIOrVQEKgxILqIVQYxSfCmIdEZL7uzqXN1zI36t4N04orr8AKUiJN//HjaDRuUGWmM3x7aDPscpJGS24ORYG0NyMaT+w8tbFYTINiI3MY0ZCTSaLhLTwCKbqN+jLFt7AnABw8R9x/FvWYpp+a29DGPcTKEnsqVon/VrSldCjkue1m2Sm2lhcdM4dRZWXXmHYyphgfQIuOWDKSJMRCPBJjyThitkOEQML14I5LPOxUr1541xyn1Q8ITYV+MIpBgMD+7vdoxrMz017EmfwzScWlYYKtHsC4EcycJvKsx1mLlVrJ+0KpQTGw5JEdNpfB5UAD5cUX30LpRhHsNFj4SxjVZQ+AnI1BknAji44EW6kx76THJsFlAeOKI9Jxp/G6MaYRF5I1gHJPa2JaWiwWM6kr6hlZSEguEVZULnEa4mYo9B0XSZqxYfPmrKhlGv3HRVbcWh1IeCmCTDXpkv4RiN0TRIjEAaAG2fKy/dcb6ni2d73u6JCd8hvsbN5gmSam3XcoYbzz/Lc3cGBckHObDJs8xX2Fc38h/erC+pWE9qsI71cS4jphPthSUkQ77M1WrVUfhEsu6JN2c+gtDkArphCiuWSD8BPqBsVWHgB1SE2RrDF6L/FQtqJETY9hdsr4X0l0DiA2hbTPx0LzYih7S7nBpOqAknVw6Tq0hB1cylLW5OQAC0iStpQsdaTzYuPg8d5gQnZoQTuMsGWrcSD7YvxYxL6YPPrD2xfSKoum/1KIQazspCWop8ptgyB2kjqUbVAqt9qlehF3mBRn672L2dCBZTY28Tezk+V+CHXaX0wnjqxMLrlhFhzJG29OIg5fW2/ljYlJdhw4OZmIypOZ8j2tFikBUmwrfUM8KdGQUxIFxj+ySUgT0BuEDfQ/Tb5/mnyvZPKZNg/zy5LB+iHMvIQdzbby358RmKR22ZEN+s5N1LReGtQu/pGfRBXM1G9eUcWat2usvnpZOpBJZFJ7uKM/M1pLFp/yWRTw20t99TmP6i81hK7nhx2HU/XsKArGKHI21dPS5S0z0EM2fFgrG0eYGsYh7sfBP4A2myAzgvUYM0NjkoIkwxt9wiBVyaLovn/f//Ljzn/t5FaI7NrfTuxjICXYWwG+mvI7tOI7rNI7tMIbXtkNqeiEksMAM4iy2WwhAz99lEBcswkVybZqP/TDqa3DqKy1ASTt0lymXHc9R82CwUUa5Cy48mlhjwNBXdYxq7LpMFoc+KF448dZtTBFgikHJbysHFxl1JNPrfAdMqFWIpyYVDt4YktRSX8VLSelsRCcQH6lk0HUs0TEa1ertRtSljXysWKyM0nboD8WzLWkjNkSnyo5c5adGA+8Av7aR8KCZXEum8CpNJKxFfLDHU0OdCzZU/rrpX5U2nvtuYVaq+VU1IlJkPpDS/thpfzQ0n2QI8i+x4/9jh4Pdew48JHjMMJYFcIDpNT3PV0MTxbRbYicLnL3QX4F30oN5JYIV4T9JkifwZyNV3c0BnEyBncw4iWDOBaJtv3QDsUAzkRiY4dxIgZ2IAZwHvo4Dv2dhoFtzP4m3vhAjsLRHlH8qniPmt5AN9hbyMN6DAN7C4N4CtZfMm1lOF3x95W+0n+5DOENDO0JvIIXcAgP4IfMb+lv8aOBU/bei75QmGyvcflGSW70oqJixqu5u+x9Q1jcUTur4SzVaw2WTxp5LXfUq82b9C7iNbDf6/FXs8i4B5YzJ4G9i8C/00wzUEtJvJfeP0dxvYC3EYWZ93+VdEevvbBQarK3G/g4S4zgi2LsviF2ZZD/xWf+1vcZgOL51/SSOyrsvgT4PSl0AUrR39vwHz2J0hEvyxfnlouwnpP6putMgfRRcFFVWnqXPjQwAmZzcl4v5gVDIR0BhuIGX9Ow6DyRV4idThVKEjsm1+r4n4X/FyJJtkl9JVghZXGrp2IXFLTcxYHyUllf+bxK737ynrFDU11vUxodn9D/qE8cEMfk98TT0JhR0TvV29J73jltZuis/OYDTxGPGy+xdO0YCOVtB72WFQlbEan4wZZ4y0YgBZpEgxH2Kbx0AhjGd3oCfKxx8N3XItmPDAjkDdrTIT5eTqFFj95aEcNkF1doceN3WsSwr9XmQTItOJVaG1PmI3ejsAkUqjC4GkWZSqnnb+eJnjql4jj3WDalPaqt8ktjLpw8NcoMJ//up/4H64YsYozuzZfplbCpNSu07OmFXZUL1h89HxOvxseXG53HT5JZ8ZyETZKKW8XM+A2NXCP+PoPND2MN+W0AGnjpFYdZ3YsQBTWNP3Y3Cjp4pRtpfaUtjafaZia6HmU60To7XNYRKvGVKdOJ19ryMpeOw4TcpXsh4nKY1q9udGyQsZZ65QDDRGEriMzmJgrqbHIPNS6KqiZt7vwKx6WGR6hsxDZGRqy1nximBisYnwhmUN4TWxqTCL5U06/9/AqzE2Pti2FgBDDDLNuTES0hxoiWlNkjdsGzU1g8s8HWjDLB6nswShXq6Im4RxoHFAqaubkBQkRLa52luJMkuzbJ+kjnyyjqLsGvifkzKlKybzOYTxPXMsxZCbevBkejXYLQVrBVNXh6hchQJbGjHQQUgnSdmj79xORTix6TsRqEkeJTP5sbL0RaCL0axXWJrAZxoUcTaEbcGDIv45FldhtEocDvUsX5lV+FQpsv8spceI8Lv9DU6u3bmEvmIR0c6ga+0CS8mytUwL0bmWO6ziodPks2ACOSoZ9rzDvSV1IkEP8jvCbSXixA+Arujxhg1i3uvUgmNKlKECLsdcD4QRBiDLrJxaVcJr95JrikS7dcSXULyISkKllsENzAsoOgpaLeQoO/XjiImGBsJMsK/pIi7rGfiB3G31mTX2K0dEMSszREe3ETZCAhpTc7BpRVSbZGKjEcw2Mpnk6+KO9i9um6EEN95ZSOEL2RGcGUx5/qrdncZD8JJxZvEdgmKbDCLnjLYaBfXNqWM8ZsdlNbzhhfC7ZYBltMS9cxQWHOSKskZ0XAK9w3BQv28yj3Umb5Fikw13w23AwFUSCWZcFSpDKyIN9HgyKhGMiXRBk8gPjg4ST0uyX/m6QKSymTO8POQGnQwhFbC174ZpuT++4/CHsU/nMqcgQpDhpe1nDYGY70XBt2COY2nLSYe8LZ1b6DLZOz+8ld0LxFSfZGpkGzSfCgScjfrP3DWGvUqlxU6C1zCT541gFX66X5eacSHLoEoimbLKJrtPlnkzqtk7DZnlIcJieqvtbsVNwur1WNQbY3WN8aMFwQs+o0Amwkf0EDw4JWcgspFTyc6IJwIUG4xOpAwsSEjhUhlTAbBR5W6Q1ELzhoeIvPh55PDZzMs44MRXKinZBWSUG6dwOJjAieRwpraVYihhVKjrCSWENQJV/WxloRq6eQaS9WgGo6WApBjZXxFuu1VhqNBIu7VyyXTG+eCVmhMdAIa1ATLSIqdIuqj7yILZ7+Zpuiugax3IYSJGLMDyNJeq/afmImtPDiq8saQgL1XOfD25jyMuvrxtHiIXNFPidQ7ZborYZaKReVXmrDoVGjiJ0eMNotrwYnI0ZRpG+qSUQdVfxTMDuuV9ylxjBOqub65cgBhS18Yi0lAok4tBgUZvdZDmRU4Q22ZKYwIREEp4fB57YNo1CnfJZh0NEa4rjgVJNDGYy9+SPDf/Cpf/9Tdls8Dwkb7DBNOvyWs4eq5qjh39r1N74z2KdCcjw7wAjTiv3b9/29Hf6C0cH+f9qGeP8oMZ3ANvAwzuBvNoWRa0MJXRssdh1hB0c4Z6xgDA3nx1o7+C4KwgaRA7Fp0IHBYHEYGGgdQMCv4X95p/PwudG9t9n96DlHUhwnDbbSCT7aHFVaiFFMefTNH8GUfbHvb8Bk0fX9Bt5muLNhTGZ7IK36mzudR5ur/Bq47tafOn/8dhUvsN/cXuUXzne3YOQfGtCnzoOvVxXs0dHRHP5H/D984GBsfaFQ4TEL2iZ48zT+UrIJG/NQVTYqzg1brF8Heo/psE5a0ACfL1vALVhqtvLy25ZhchUXZux8KObW0ulQUK31WAlEPjOkDaJVzlVzdYVYXlvl9geXZGBSGKxEUoJQuBozU3gJSVN8NCMNrAjhPMIVwkghvB51ZBXjzUdGwBceO55FYryjOojx8axC3wp3fwZQlM7NKlxEswRxtXW3nvgbuwYsEX93m30eIgoVfY4s9M5Hn+GG9j9f55uZfZzB8J99ht//ubMjwPHjFjf3oRkDP+yzuS3wv37JRY2/87LzFQiNrR2+eVDC3N7s3L6fMVCaHDx96H+1bwAF/IDIi4/9B+u0Tfa2DHz5+OkdxDx4dMvfBVa2NhD1z+tIESWIf/dh55ONYF/T7qJtSh/Q6G59y1uK9/Y3+uskgKMj8iuav4F+x0QjtuDf28Qx4hlXyBvb4VwqY4YDbnLqu7/9sLP33P/oaSI3p69c4J1AUt3tDf/eQ+PCCWz8dKlRdheOnK3NNZ2ZarVWdo5cdjwPtl65dbCP0/Oh/3vk507nk89xzozO4/Xu59tHZk5c5MPJvtjCvqdhdP74f93fb2qZKUhq/VBR4PiN82FcWBsBZs8sPkv3CTNbi9Z9kN0y167VK+lScx7UvfgUWeY8hoEXYRdFlH3kqse8kQ6+1XUEsb1MCCF92Cu8jpenmmgQeXUUy223kjCgKgrNxpXjcL9EHKmIbO2EKzStYS7uC6Mw/A0aTJKDQolFzTe6ZAS8fgwtj3RapoAWS/gYXDDGFIYgbklXbEoD0Pv2zkIq6QMciSkqygVzXEz2vQhOPvKmwQxvoaNJIwJqcqGS4pPQoj49X21CMhjEXCdlbcW5lM8XokcXrJJikYK6JjBpyR9Y4WC6KGFIhy/Q8LwpGAix6o+Ij1aJz8BRMhaT9JSSZSsnHXaSFgss4IhsZJ5MpDBMNUolOrqRqjCPJqU/eYjBMw8vUhx3miIAsVOCSL3mKCACETOLerAQ2EURGBaklyS7fAfkMLMpfdQvMpvhIvnnnP5dzSkzWTqPN6T8SzuSRdd7XpVIRITLMPrwl5vfhHCSfhXU4iOpj370XSyHHfDA//QfbIFgjAy4rFGkEVey3W1Dya6MJBLKKYJyZiXLELd5/rcdTeu2w2RtmydhH1pMkM2vFfo8bCZpPjsW/wrCUvrcjAK7/x2/uMFeS5OH6RDjEt/O+i2clCtiB6kgdpjyoRm6IP6VOGxsAy5UTBYXU8NfSqCLK2JiwIukJ9ca8ezkyLkZoSUcmLye5ycH4nMqU8IcRQMFT/so2BG8cChuyxbgZHPyuB5BsnhIBOqSMEdkSMnQiMLzW/pFRrmEJEmoGBaPKvH+juAj+O12GEoSNayA6lj8SFTAE5RGyP6cTQt/d1MTNorAn6eJo5xAg97KxUMKTxc1iiDOtFuL7Vb06zXCW1gzI189AHOVfags7hbl+Jf5oBBdkwBgqjnfXoAVfYFq0hXHKzdr9E5n3vwpOlrMtnQq3FCX+DfoO8r8ECUwdSkVjuKAzHqtYT6rS/3wMqJ71Bhe44w8U/tpc3Q0dG9MW6zovP4TyD3pcLNZQ0T6BHJPCsCvBpt/ALknJhr/o4G3MIpuAFAqldmIei3QhMUWCC0ovObUF/Nmqe659O4xvdFrBKgifz0cWIq5eV7bEWLBU++p5xzJa0F8iqsETnTEMdZ84o59joK8a5kIUgFnqEiCtFikiHSxiDSLRVN8MAMbSP0/OOEC+w==",
  "derive_focus_outputs": "eNqdWH1v20QY/z+f4nT7A1vKSwFNQoEgoRVYJUaramhCwbK8+NIYEjuynXVVEykb2RRtlehgYdmWTkUUBqOIrO2mIuALxZfvwHM+v9vpGNEW9+6e9+f3PPc4ddNoIVmud+yOSWQZaa22YdpI0XXDVmzN0K1czturWddydUbeVuxGU7vq067BMpfLrX52WV5eWUcVd0PAby/Js6P+/PY+vTmlLyfO4d+lpbdlOvyNPtl1dvr0hyksvnF+PqR3DuZ3TrGYW1tfWV1fufy5vLq+/CETtI0vrnx8EZfRUh7hSx8ur3x2CRZvwuKT1Svw11s90KuSOjKJospgn8BMK7sGiKjwPmpqll1VtZpdtWwzj+BLkso5BJ9NzW4go010lyWPdLLZ1HRSwTiPiF4zVE3fqOCOXS+8U7C0DSwixUINRVebhAtgH5NA1HRXiwDai8ugaR1MIabASUXRM3DT1GySsDCPTGPTKieNNK5+SWq2JOVRXSNN1Sdgtrs+fWroJNsHvIn/pyOueSZE3PfiirvheeFbIiboi+6jwR3OPmQeCuzLD0RL0XQh4Qc7B91BEn0klRCOw2V/QI9O5o8HzvFJEQgx12k1AIUsRiCjCrJQ3TCZTKTpXLRWZ88qbpuaAUZtYYkdeeAKkdWTcoH5Z9sRga1rRz60Ic/RwPRWl6TiV2TLEkQGAyZ6wzQ6baKWUZjtdN4Z7nsuecSPQH6YM7CyBbSezKJFbIiv0mnaQkDiBbeKaw1S+0rWDSzlY4fbsRX7YNtoy0GgysnA5dMMLeW6bNUMk7h1mj4PdJfPtsUlVjYAs1uyrrSITx/dymIxyTWNbBLTpw/WWcQ1xSYb4AuxgLy6yB2907pKTFnV6nW5ZnR0e4FrgFgLOiScQvQFMU7RC5dhbQAU412umgiwhN5LUgA6BJbsajw3kiiVYwqzaFhZxRXk4uRh9hgtrARNt4XUmZhHbN+V5W+JGaLSkcsWm0EXVZElBophKaUzkk+pqLShF6pCHW8zIW94Z1tvSD2B7/hRgJ0S30kpgiMRJ7X4eZaKnbYKUqHfmnYR8qK1oZexMmUbXr+JklvtpmYL+F3oupD5KJfXETxwex2wKgV171Y3CPTL+5rS7BBoJWHKI6y+569Z21mAeUWBp4BxdsF7SfrvFc8ZXqPk/QSdUfP1prKxQeBq8fAQlHQTbs40jMRXtQ38LsLFLw3tvzIv6ihnVsOZ3Sa0wIJrgahCAqZishUlOlEUOkxCCBy4rypNpXVVVRiUyygOqYzG5AI+2Zfi2gtBWccayhk0md0hRh+oDrElJvutmLjOg+NF9/rsdDo76TvHA/roPh39w6/2rHoLN6u5xQUWtzjifOIgcCGxH62CxFGA98T+AqgnFYaAzbAxHfyUcg6zcNtLuNfTAEObhqlyZuhqrMxeOdvNbx7QxzvOdxP6dAQTHh/veAY9g15LHJ3+QvdioyKX1dA2GhljYjhCZsyKlQrioyLvzi2iap3W/xHizZlcDIOK1/a9Io7MkMkyfPXFCC8T5BoxLVK5bHaIWC2fX5J4OtjbgHu5BOnC5xB9eo8e3OPvZ4iHCXHUR9KKI3/XcQGxIJTRNksAi6PYmz0/SZBwFyPjKifnMcti4DXHHvSgn+KL1BwwTyceeUKG83JIB8/p3YPZ8Qt6d1Ki4wPnp3GJ3t5xnt1CAbIQUNB9eDy55ez/hOZPhjC/g6IYXjNM5MBCHFJBsNx3EeCOojPBHI0fPgdBfzSlAwj3wxHdO0X06xt0MkDnl85g6tLhxDkadrnb3fnod+fZr11uCbDR4bhLbx7SyX43xlUoFMrsy//PFx5JZMbQVXI978OXgCfEZNOND032wgwjS+VNMfrKy4rfHyvDScck7aZSIwLusjeqUmSEcvEXzmbdbVdtr+sNaV7zg7kL8Z1I24PNbmqW6y6e3ODINajX9fRz3eS6HZuOEk072d9Ypib7dG8MoYWIJ4+T6wKajwf0ySFAYhcKCs2OprPjv+f3JvRh34c3wI6+nCDnx18Acog+uEdv9emDXTqYpMDrDAfO3j8lho69kxLdH8G/FJTBuDiaZ9Nv6R7ogQb4zUM034HHLb+oGMAY1d2hc/egmDY/KDrnzn30wScrq4ibgtaWP2I+eMbHrKYv78PefDTxqwP6yPzG1CuS2bSP2IrjNdSNICoIbHf+OJ39uUOf9t1a2B8h58/B7MUuEzc7+tp1ZDRgPH/0nR+A4/Guc3iKeGt3ZY/YFT0f/brQp4tX1ny21QvrTKKXpLUL4AFyjvvzh+MSHCFn+pw+uA8xZYLpg9v0+xPYG9PxjflozEx0nv01/35I77xIK5Mic8biq4gPEy0Vi/w3EdkGQAr4C92b4VyQiqlfa7B3ncIlAt2/joM0VdzWGCzFngd2n9ArIKvTainmViXdSBP0fv9raLZVSXTDBKnf7BRd1diLENDH+h+zOQe3n+yWryy7154ssx99ZBnzLsJ/Acr9C1HCJ7M=",
  "evidence_text_pipeline": "eNrtfV1zJMdx4Pv8ilaf5Z3mzjSA3SVFDTnLAxdYcUO7CxyAFeUDcB2NmR6ghZnuUXfPAiCEi/VJ65BkidJGULZ4Jh2SRcnShR585wuZjvO92P8E++bF/ofLzProqurqmQFI3kU4jhFcTHdXZVVlZmVlZmVlDbJ05ATBYFJMsigInHg0TrPCCZMkLcIiTpO80RDvsv1xmOWReO7lj8XPgzA/GMZ74jHNxa9Mls4PJkU8lE+TvXGW9qJclsxP5M8iGo0H8VBWLeKR/P2e0sx7sVbseDT0oyKLIn91GI2ipNiC306YO6tbjQEOsx8WYW8Y5nmUi3HKV6zEOCxwHOLrOjyyD8XJOE72xft7RZSFRZo1Go3NOxv31reClXsbTpfKNwGZ0Kkg8PwsytPh46jp+YA36E/j7eXNVV60rMc/OvFAfZmEo8jpdh33q4vBxZ9/8vIvfvDi6acvf/jpiz/8qetEwzxSCjdWVu8uP7q/Faw9Ej1xby4G//o/nrz8s19c/Je/u/jDRy9+/08Li7eCi998/OIfP3zx95+++P2nwcunP7744ScA8+IPH0ARV4JZ/ebW6sPNe2sPNwFS0/XH/YHbclz/4Ggs/h7Tj+NhLn+MxA/xd49+FMeF6zXuLG+tfm1t40+CO2srqwzqq+0lLPBa+wb+WVpkj0s35d9Xod69h99Yvn9vJbh77/7qw+UHq8Gdd5Y3sP61N2933IWdne+89cq1xsNHD95e3Qg2VuFDFvm9dDQGCjQzt/nWm1/a3jnyd732W823Ojv906XWzTP41YKfN8+869/Z6V/38IuPP9768lvQ5vo7aw9XLcCgyg2o3caqrVv099YZlP+T1eWNALr5iAZ2mhdZ8yQKM88ZpJmDv5w4cbIw2Y+aNxYXF1vOjcWbS95Z48Hyw3t3Vze3YHCr91dwdFh/u+HAf26eTrJeFCA/ui32Kosex9FRlInncD9KeifBIB32y5e9sIj20+xEPEeAfv4zh/k8ycXTKCoO0r54KqCc1lgvTR5HWRH1AyI+e5lMRntRFvTSSSKhgpiI4PeuIMIXORi9z4/D4SQSD8M4iYIkVbpfsKHvwhz99+Usp3+drTQdbhI6OlQeGPpVYNSOA8RzvuM8TJOI3ufpYBD3osr7AuQWSIBetUY6jHD2d5w9aILBDhPgoQCk3X7cr5TnX6MsSzP6KPrDhAh0Nu3FJIfLr0ARABfDCLO84xST8TDahm8tx/d963BXjwvsLADZiPLJsGCDZvxQgmXYLZ8RgeWTxhAdknR8HEBnORzkBjZGEEGurS8raW+Csnkj6qVZn/ek5A8Gmd4KFin7oDGJ0jXOJuUbrefzjFNpW6VO/aDZaJX50IFZXug4gOH3owECGcT7uLrCL1gQYD1w2rcJChs+CH5YP8MCBAcsgn5e9NNJ0cI5Iqu6HitK45FlfKVEE1CT9mGN6rqTYtB+HYQoMVXeBTjjYdgDGDWtQbmZrUGZS7fGht+Pc3hxQhhuamgGFBEm4G9HUgIYByUoFpT9TfNyOQTJU3YOvmEd6GCYFflRDC24O/DfWzs7jx7e2dlRB8JYCrScxKEyrnOdKm+/3tmdB2ANLAJxi4NQ3vHRD/JLDLyCqgoGvmRiQGuR5sgesNikwFkJlRCID68IIhaSEEW5WUPlDYjiM6tb6uokAbSLyts3dLzJ9pUyHI84gCA6jvMiV3BJWERJ21HBiHHz4ioJPMGVo8N+nJmQyikJIEbhYQRljPrA5wg1SA+7W9kkEuDyXphAYSpY6aDQFreZqMb3Leo1+3d3l7WJOMSGOSyj36jHgtjM4igvEYwaBr48QRUjT1FSNXmhlnMYnXSH4WivHzoxaNQd+pfYyO+FeYRStOkZTN07iId9YB1s01lgsKmKVuokjqAUlW3xMnEeUJ/T4TA9CvKTESzIh3n3bgiaqqcUwlWtphRH5VE4PKRieTNL08KGR3yzKxex3iFqGliWMdMR9CtiH8rB9SYZKdld9sEfp+OmJ79C5yx4gApQfgjkbur05cC8EkB03IvGhbO2uYpSkGh13NNB0ordHLjb7y5vPNyFRQstLGKcjnOqzXwB/sxpngKYM8/19M6BchMnk6ih8QEnByME/UUkku4ZwSKWA2uIURlEh+nMaumvJXr9cDyOEl5d70o0ZJVJ56nUVvhELAawhPSjYO8ENKgmqgagKOFvXRwytmYLDA6gyRaZdh7vo20gVhy3N/7qra/ij2jSax9m4tvSa6oQqtCWSwls3Wf9katZhaCPkhgLrFAxom3HTooawDNXx94wCpMAjQyQPcAfpNR2LCtEuDeM2NpIkgkUuiRvnvZAzriByxkAcWW3mM7YyAg6QKG/PsEYgvLUJOh6ETB9wEZHsye/joh14B/65oHIz2KYPX7Gfri+wxmUI4CBgB5Bz/gwQVkZT4oAJUkzi6DN+LGq77WcfAKa9rFl4DkILeiOiScNiI+FvO3O64tMAvTj/SjHuc7dEn5+EC41tTmm1fd8YgALueL9JEV9yPMPomMGtgkNLd3Q1q2Be4o9OAtOWZGzUzacMzH8/ABkc8CNkibaKjTSFgiXUcxUR+jtrcVFfexYsIYU+ElSQizKwygh4J7zZpeDNtdi/Nwwnrc7rLDTdm7uwtLrgh0heg7DH4XD+D1QgZm22yzSwyixEEpoIvjZ52zedFvYYcntsAJlHE7elPaFbZEk7ABaWlhArJDI5dzQYxhAho8AHlaNmkxzGw/jAr/lTQ8rg3bSXfI0lVEY+H4OxnnvgKFsyrwWTY/Cgs0x6W/wBzEsC9BzGwzCBJCPqvn7WToZNxd18SkmWw2WPVNMs3kr5t2XXQ97o/gfqiK4MpBSLhOwVonQKpMKsjH5H6Bh0yT1B0QlZ+AITUsLN8TIOoUjS5t8SPX0kgioWozVN2abeH32hnNKFeVMIyMzHpwEwii0iZvqajMGNmFLpSpW8K2ibZVUqpmUWF6flHJxBumC8HU/WGUZVlvoUqUqRTkW8GN9dVUtp7ZxVgO7QFemF3JMM+czNRnM02R7niaFhSAlKnMZCG5Vl5I9UG4VOqtKN7Gs+GdXyC1Gcq71+pIFirSJoBghiRWI7Owr4w3NT0EA4OX24q4QxvTsObedReayhe5X/Biy2lK12pJWTbA0LYd1bK6txOJlS3aypTfekkDLlWoy7Af5YTwOGI6n4VY1vebBIrPiWQlpVSAXoVGiWpL/+Y8sdiQaXOpSp9PCwymz5ICeXgUXTIPGn8kIEVY7iPUAfoSTYRFg95nLBsfd4aQAuQ+KXoQ8sT1mMgQnuPDw+7gqoEXkYW/Hwj7yqH9jZoWBTBX9K2eI6+02dGEToGkDzcjNgwXHvbEY/Ounf3fxyZOLv/7pi1/9yMV3CoxGZXYSENkLxS6SAwFzuUAtXxlMpfqVx+Spor5sUyFKGOOWxgnqUavHcdF0H6bOKwoMUOqyqFcg+w/SSdL3gRp57rTbSB5YJ2DZ78HCf+K7mjpUtsX4o8IGZYFtrj0qKEGTuqnY0eMOjBP9iU0U88EIN6Zawr7iDoFa2MKRAUUDwnWOu2vNLDxSvcJsbYqK7VJCcbRhQbP3ULBZ3bURnUBfL9oMUMiTSx16AdiOxBFTllBJUxUkpnCTs0Bo+mCtg07iNTTBXzC3wnS9SSmpzUjfFPq8WfgAawI+KEPFcfhhv9/E94aAw2/SuCyAQwLuUof/m8oCUGJYFf6accg398YnxUGKIMwPR3Fy8wa893vDOOLOXm4nrtIfoKcw/R3n34HFH+6PQDlKUsALsAjo1/0INancgYIHaV6Y5GTdG7iiJWeShI/DeIjmWcchZ4DLrH62nZDTTto7767fzXDyvXM0Xtv7FuAAFZLqW5921srH5TFOGtpc4JNGDt2/k95L4iKmuccJr+GKGUgo/HY1RYf1C/mL99DQiU1jnJbD8RgAGdj1V2L0WoJ6zgB5lWpWWBye/x8mcdGs1jHJZQcwxu2KGo2E9aYl1uQ6wFYXUIk44VIBXZYBPBPk9ew84b4BU+NbaZw0GQA+n+MkHA4VNKgEfJTEKgm1OVLAwp03ewdR7xAnCtuuAiKQCMNJU9khU3ex1B0fdQML5y8JeISrLENlO7KjGryWCcUyk2unK99xwwmXpN+G6Xb31uJSQ2koD0SRbrno1zOCXoFpBaraIBHTLKuw7cMuqQ3RcdSbkEOl6fIPrtcq91PYlmK1LP/ANObK12G8l0WiRAlNbkRW4clPank+sK4yyJadJl2dQmYholRXfWip6KjsXnY5Sc33TRZVoHRR3d0UtdR3TV7WU3U1ZeCohmgGKRf7wtgD3QHXRIpG8Y8OYpAwWEVqDVSistDSW6nDkuDDH7s1W1RMDYWpQEoRXz4V5QiULW2XjYAKoUCA0GCjQhZ9U1ajeBPcoSvHX406Qd1wswcLeSE2/7Wq4wzY4FipNqM86lvBJSr5B+mIF4AFZyUsQtJWN9JwBBY8/V4nqYU/B/z3KbaEWhWwSBAng9Qfhd9KszPL+xh0xDN3ZvvyndIP9d39tBcOtTfrYe8QjKVcf0nd20wHxREg9y5yBfGxzz74N/2lpeDbe++9mtw4HGTh6+PFait3QhCI2ushvm6PLQ1eGSEqeB0xu1IVLFXqOLFq5eSIl6xFJgEyqqHziYq0gykgLigVTaeBupkna9c7AfTNEVm+YSyS+gpXETRS5bZ7q2btsPL1XluB0B09hOZIoMiANj+bJE1tLNtub9RH3Wuhh/9Sr1zynbEe7bYa+qYLbu53FYjr99ZXK2VA6k4vg9YJAnq1ZextwWrM9tzKD96MVVHHAq7pYuw++0TeHEDeYl0dQUx1B6gEwsZcOs80WuriX9PpWdDL3FS06RFzU7HCn+4YBXt+EA2Hbqv6tf0wXc9SZETr1zvpaATMbPv2tahor7MBw+r17kGURW2mtDunzh8F/kMcY5v5o6+tr9y95rRhpsKHlSwGYVD9fAZgNqMhABBw2qvHY2gdOjiOsuLEwTpGV75gtlxa/Pz4stwDmZ8vm5oEIY4ir7fgQbndgHJvFtsquw7kelDgCGYGhgoo1qjHCN/kfzu0xUz2fksgh7aEFE5n0QKG5/JKXMwb/WJJy/9emb4V60n10aHtRWgAkd7zGo1yAGQ5TKNToxzI1LLw/QoMxXsn4rd41JTgJtTqXVj0CrYZcGqDe6aJSjRW6sBxnopYZF8A62PMuMsMPrHH/mmswwJPQPU0Q13cbM+lwBPQ8/vmBr+IWVKRyMrBgEIMLWnMERnBh2r2sukOQpCbtGiyQE4iOu5AdQXltelbBZAeuko4F6MB86ZHo3FxokBmcVE6QjHE+TPhk8em+/8xHt+lmJdKNA/uPYK01rEqrAv+kVYzikCp7AW668ynv7CePd7CQaKhWaOi1dFLtEIEswL0qj4UA6l2F8o0sgDmEb3tMWuvpIAK4Hg0DKSpVWmiEpIktFVa/QUSbF2vM8igPdez7h403Tu4y5QU+QJ2XWDJRM2uIewmyWGuiHbdVWZ2l8duyUFbsFrr7uLbA6tbPp5UQKok+zplyb6td4Zt+evolLaEt0zdRZYGAjtsQT5lsUtQxxZAAF7cJ35EfKsvpjNViVhhKtvqejNbBtGitwwvrtyyUvdyi9k0mSdOWFRlnhov6u4k3CnI+vbZJSLNS5wKdqmI/Pl/dZHBWCqK41HWlcZUxscanx8dGCrqyDBlks8zMa42GS4zAa7G9PMy+hfGhVUGzJN4MGA2/QEwQZRpu9J5/F4kgqiWbrzOtqZPpCfjc+BDPhqVD7FRm4pT4a89GbQgtTWwZ2ka8dACORJ6foX9SXsZc8uzR/LYdxT3sywVDMNkfxLu85gy+bo/jgkp5RvVuGBvR7GwSQ5CPExSfgiPATv7EXsnds4a88527psfxMV7n9tcBKy1TV0Q3WMPJmDi2nbKhLBUO4Y98rfW1u5v+qMJ0kE4l9i2SpOHIs8sfhRmCYgbrUK9lUobStXO9NMecCy1UeVIT6m9H7FDJhiODOWgXvk1J7Mej6gguTDeLMYIElGlVZKSLFP5xCZdWbChKoqBiHcxtvfipB8dl+fZ9La9qoHA4AhRAv3eJgi7/n5UsCnAlHBltBWpUsKpl+uDOMuLQOOoir50tZk/cxWyERLmYRSOuliv5aDUKk7GURc52Ajlnk7aL4S800h8FTJ/BlLPRe65d1cH4XC4x04DmII2QNoHbHM0GMZ77IiGgrtuiUXTzhJgfXaOjHZ4YA2rIkEWRMmE7CBlE5NhIJYkm5694TAeKXstC7Fd/hqDShRvzGNsVR2LQppaXJlSula/Wd5x6Tv/iN7ge9PyM6JDxdiZ0QqX3jLWSw+3FPqB57xpLmKNL4oh6rsiB8L7dHt6n626haTsfMO+3TXGzQItC9IbGsr5DeJZZ6baNXMlZqD0dVix23mgD6kpfnk+dvYCL7U+Ay5f4d21OxtQ+duTKEfn5t6kKDe9nSOQA9gq7c5a13suXlD7AfgollpmH1uaEtUSulNLVZlahhz2PletBhr6YhxcJXCbNTd9Mgjl+pWWXRmcogoyXcpqF1kDrvqD8ZDC3hsWJaTWh0LreVnXpkIhQfDk7uwVlcJu+wOfUXfGqqoXrqye+JoioESR7Y7e5u48KyiW9AWpaP0kH7LrXkJVmumum+GmK7ErWWguvjeDnEo4U3VzysBxAoVl/o3+YIPMvYYypUIWNC2/TVObWcg2lhKUkKTctRBWLVqP5VrK6NSvLB5XJ8MJy8DxWShwQifYNeTPJ5KZXGiL5UkVVJWgNCFhdHGrJUtoObq56kgzVdv7MgVuuSFWbtorNmZjDpkxCgH1x0JPf0BPTWjdWXC+coP6Qb+88jCuyEPjb0XYVpidrIj4hibJlmI0Vpaa0RjjoUUaGHj0rmw7jZN9gCMgYhgIafKnDMZ1Z+nMhyK6ljiOj0chRlIaujd732TD77I/LSccjg/CrmHulmD8PHysbFok+4banh6aO1M408wdzoqY21ZWXRV4C9OTIDRkrzZ6f8r12G23xyy7zGvublUTnbrn6FXPq4CGdGgJNaH48w0wlAAM+XFg0pSKBk1qBfulbssGf+Z6M2xfdQdSHne3yW19jcaJfhmfK5NV7FAknUHUYrenLMwxtYnseWrm9TmbYlSTzMaVd3xyPBRTcpiG/eAozQ730vRQ39YRb9HcVUuZNjlK4iBNhiddtguKpnT5rCOb9t1nbrXkB1FE4xNN+viD3lp2y7BGlh7R3gwWIedtAG/yui0CoNPQYk6XzuqjIEnVyFlbkwiE+YuP6jcicFMayvns9GCcK+GNtXsXdRs4le4RZHiqLSnOLKKWWnbDK4987mTsPJz6KhGvzMNxlrGxI9DTh4O4VmKoqR+9NM36cRKCKdk9JSDmrDQQmNc3QixVNrB9ypigiIthdLZLnHHKUHbWcVw6OPcdsRYSaENcSp7rDdM8as5wgVDr3twawiwrQExPTYOY261iahIC2jRNTij2sAbgRjcq4aUYYFMOpVSffGlMxpkCgErRZmiXRcAzvz9/6JNDDeP4GhrJqh7LEg6sOJnYbqUu4JweVeYzn/kBCfuWEAJUlYRAnQwgjqMO4MzQj8oTUD5daXL7Rcp29ZF2lvIUvxYm7m6jboLUMK+dcWn8Kt+WK1mFfRl076p6JpG8TSStPdIwzR6mmjOUzFmT5jOZzMoA6mNDgsqp0+r+TDWWy75k4yjtyxeLpu1U6lkSb+GoyK0hEpCVmwwERPGBYfehVDkEzUhhxQ1nJ20c8i/1VgxTr2ycyatyx2hJUSoO7NfcVkrsKt2gGkSh7V3PKhFZwekjYMQ1Am2w6ZJvASltQIpDRSkhi2z/zK1urGKX5oIn9UQNIJ9cGhANgCtOoNDhuQTGMsT5UfraGLFF3rrG5dTyba12S0tS5ZVBiEKl1vgDVDdkBp6A7CrWrDgUU/bUmOclV7Ah7DIhSaORvOA16jGP4Dnm0TAXNNVV9WnYFzAqvkXJXuqMmMJpUxm/Ihc5L0+XYUkU9fOAZabD4HGOUrvIZO3YBJgmAS4XyqEfvPqMnlA2HHYuEg+m8+Hg5NE3esWMmHOjN6XjW7ySvzaMMArv3lqds+hAuJWgBnkS2WZG08Vq79BHUmi1rb/BMNzHFR/kO4WgiChIKr5987XOrUWYTe4wLkBtdD3h1GffyYV/iycGWNTieDO0OzGQF7f/m6yVP3aWyoZnRJfVryiVpGE4XCyHB7nZkPOuiDtNM9oJYT6CapgiMvSCWJoRntXYrpzFfzvtn2B44cJm1GNHT+fMWsJ3YQ0CsZZtm7I5gx8wZLXkMxMv+nqol22yjduSFJ4tPoafl9erepZFgRcEO4XUMOBatgqwCEAUDaiSqr0robBklYhqMSnMo9VhcsICbzRlk97QsR7WLZJXiDp+7MblQZ4m9nlMpoEe/pYJDHOGSEha3K963IfVro+w0wOOePGq6VbBhYwg/Tzj7kzJo+/GVE7cVjwiiB5u5eGZVDRUQAOkPFJ0ZLANkhhU1f5uY75DyUrsiDXeykZuduhW4YnPrBcz3Os7c/oqs915dde60mjMU5vzTYQnm8nTll4jcpgJuVR3A/MuwL8VJ8SOONjOocvt1Deb2//pzdu7r3i3oVjm7iwRjNI6lsYkyzsmNX6WXUs9GFJJOrVrd/FRebUia4KfFbHhzSKTOOZU0cRixJSjIsphEvlz18xawhcH5RgVAd5Vlor6RYQfiuDtN8x45hYe5e+DkbkXg5mP6fuabhYeId9jsjnAc3vpVbDwmy7mcodHePLmSBOg5OXgkqLJmkMwxDOsT1x6K53wDJnEJxzVi2oTNlp0SybAnbItI2rDa0zprDtJSrSp+ykUsmpTSmcBKetSlb0oL4IZMakMY+WRzZqDn1yM2BdJmgnGOUx50p9ab9iC7stgezl2bybGJ6gIMbS7VvtPrO5UzwzVKDvJQzIUFBltK1/QC0o/1JmsfBeN1U9Zq6iTFOnMCh4ep9iHRSWbKb657tyivH4YJYZU71SVV0MRxWLbWLUjAKjqaLlRit9AFy3XtHA/oCwPHPAfO4vHN+/eLT148Xuo/XFF1rl927mx6FGpu0opNNqoYJd9qHCFGNRt25gkXUCp0KOdeOtXG6t1vLIvBLq2O3pXxuEJ7mDwNUtrGsHsmujFl9puMEPyl7rOa1+hdDHvvLu+tfy1YH15YznYWv3m1vTEOoXY6aFezLNkmlWNpU13DXNtYFpgeFlBm4paYlMNuuXgNY0acyKF+1k4PmBdyyg1fI6CCajcG04odx6tm4iGLB2KYee+8/UoGnNQLFmU83UYbZgs3A8BYQv9eD8uFsYgNYsJnQN3ZKh9P0vHdBwSXu6BWgcGUZLGeeSzHG2YopiMMEV6svSPOmLoNB+YJRkmyDWEIZYGFWCnMJxPHLj0Nziu6lqi9RWAvuncvHHJeu7OZPnO4qKLggKafxO1xcnKV5ZvuhShsDO5uXTT+Hpz6fW707unDov17sCP8zDvxbGpzc+uSSj55394/mc/Pn/yv86/993n7//y+fu/fv7+R8/f//j5+3/9/P2fXPzVT59//6PzZ784f/bL82d/c/7sV+fPPjl/9uvzZ785f/a3589+e/7sd+dPf3H+9JfnT//m/Omvzp9+cv701+dPf3P+9G/Pn/72/Onv3Ct16vzJR+dPPj5/8vPzJx8+f/Kj509+/PzJ+8+f/OT5k58+f/Ls+fc/eP79v3j+/b+8GvB//u//8pf/8g/nP/jk/Ae/vgwE081YxwE03YhdXa5w8pLmZ1veSv5V350Tk5p9K+8ywAsR2LxOKcbDeoaBTuwb72pOMFQPJfADEDQAdu5f5idSTg/TB+hEed0EP55QJny0JPyaqtVyt6h2YceUHOEKLiiCjlXkkaQCAy0bVG9aRgxRsz4hRgnbZWDb8L97uW0902sq4VhV2inOUq2u7pzTRk7L0VnFhS2uWJmJZ+LeCs559Rqk86/So13BewmbchiqeUnEl2nJSXkRUjTK/EyfiRAKoMtSQqurUELBuTqnGlpmepaIlM8piiXGZZuOrDFcqkk4vJYWh0g+HvoyhxnHStqJiZk0WCMVgvIGakip9aJKTP5++lllXgjz71FH2gxNp7zlsysmfDPoq4CuNR5n7C5pMLQZl6QOnjfgPZ6DY1RIfVjT95Rdppr8c9xBWV0PTBkodkaNtcBRbkIyL8KxJWCsSb7IQvrYLR68AZ7vao5UhizjYHmdk8qnl8xGWOFxzD64Ee0D30XZg7Q/wVxsuImAg7+DKTBW7t/HhVd7J4q63mW8glP78M13jsbvkg8k9+8V0ai56PnfiPN4z8hrd8m20PNLsxfbWLOdpsLBDdPeYWeAbbwBenMvwlqdIptEb/BdK35+jpfRZQlvIs5ZL43NxWqgnvsOMZ2DveFcC9WprgIYAxpFvzfh97Jys4tYcfF4/8pdvTes3qU7w5qg6WjtUsWRHFOedWwIvfrVkK661Jm2VJez01xqNJ2ZNXLqsls3z9XL0yzBEIqC1pk1mfci4CJk21OWZrhMUWwU9/eH6V7TfYWuLPTOGvNmxtEzTAi1oaEnbGqjv2MIdd3KF46gdpGa38rr85TigAgYq/He5EaBgZpSxK8NS66mWcl8ZiXyqQ2orSTwuUyOnDmz7lQCBOfM7FOpZ5mavAsYFMaAzpeShy+H0fGY4rUpcE2jkLPgNOU1JJS4sTzoaej2AoiRBUCA/lIJ2zizwtJVjlIlKFvCalVYx6uEFogv/JKwAdP31MzlU2cSS/hNU49FpPApKU0ngqgLM3yDufAvNShR63KDsmcCnKlf1msoTJmdV3JNTTxnWSKoH6wVlPcY1gySaTymJDCYG5qv22591luunyTW99HjSDnwq34Rd+5q4dtS12H9kvf1wkOL/RENzd5itYXRj0+oBQbKGZGik1vjOFh/4PdM9W62zQwUySKNtxRGEnc6hCP0tA+uuaWWf43BTwYppYnFyOJN/HeV0llGq8flqjF4EOaHXYEhf3N1NXiwvPn14OHanftrm6vrG2t3Vjc31cS4Sb+rZL0bjr8RZXtdl9rWlo4hJY3q1gj74Xgdux5Rilw2CvWjPKSi1a8uJ8nmQXrUFZzkb74bvHNvZVUV7ezEOznbBykeJmm6B+uMhdTbJPVT8yUL+u+GcXE3zTZB0xtGLPEgzxEmp5XzirO0uLjoTTu3XzKu/7WowOsB7oBk5h3hAL1Lb6f3cTVnNyxhX3z8p+nhrZCsZ8r2i/r9TVlRE3dWbmRRF/w6QkAf7gFY2BEvCLGG0GkCTqzNfj6MonGThwNZZlyjVD4YYykqEJ6flcsNKqfiCG0vi0J494bzIO5lKWhAhUMZKJ2C2ZQ4kuV+uhfRE3wmXfvbE+A1JyQlGQzIcJjyQ0nmhnafX/tqdxfargRjWUsm7JgqXk4325UYZvvwVlxTTlkywWrrTc9bMv3EjNxas2egkZNgz13ZeLD2cJUysPEAsM5rt3ZRx2CPahDUnrt5Z2Vz2XIrieUwXtLLTsYFi0/pZyMt0E9+XIAJUjCqYh/fICNqASjNRD8sWOEElPyMrtJgNm05gwkFXa632K9qqmStKeeTIIoUvL2si5Tw4YeRHhLJ1qV/K1XkAWxZtzySXSnbH8dlMTyrXSkhlFhZyno8TD9Bzwrr74zyMj0AKyoezXyWKk4plZeqB4OeOo76geQszh5DU4NF0WrhnC/D/GtXL6Pld31MJ5adYNOJNpVwlyXefAS8HBGvQsj5iFk9QcjvIpGRaAOWMW0U56j99duA9uunWiHDbajfW6K6GXS+0Gn+5lssPSGZLtPKzeQLLYPb5cYGVa84NltXd477izvHvcHO8dLSznEEv0P4uwf/L4XwvDRzJNopDO7wb9ES4AvF3+oqnjVMeLrKMA1hqecPtUqF/y8U/i0Lhc9DJtgovf71nePFm/D/rXlmyPGlJnvZ5/fi8b8tOfaZxZjc7MNZZB7jLU9jBOKohxanrUqlms1NDlZReJ2F+mtzW5qTyQqsZelV3fb9bFGgg7bLguprmzS2lCo3JdmsYy8Ub03LvosIY2AVxJN6U8CMRADagCz+cQB0Cfk5B4pmC9SZQvUqgnV+4Xp5AXtVITu/oK2STieOb84f7XlaRUM8YKaTU2PCnF0/rVSoz8xWFtUuQvpc9BbJr7JvOseWDZXJ506Nl3hXrjlAcbIRd3cr3tWy/qwEEfyH+LunZoww9CIjn4VXBc2OBrp+L388BY6R8H7G+YZJIt2r7N4VoVjhPYSaPS2uZCGfSFlJuDOOsriIAuia5j/GI+E8pqgf9woWUJ+Su2t3FxNARsM+y8peBh4ZW9/c1Wm6OetzVmLa9CQ6Qj9UF/svrrDvsjvM23m8b09qSUNAFRNG4a9Ad9+lF9IvV/a2W/70jOo+/eGOEPtHOieP/xi3SEOzLI/G54xBcUsmQjK4ZiqGiaJS70YQmiuP3Higr9R68ZgHryvjtWspFv4/olg80AZpOFevQE4pdVg+1MFwkh/IrWLcPQ7Y0bDKPLGSkTLMoZzBa9vkWRaNruxrNapHd/grtNanacvZ1nhIdJUS0HDHMiwY7LrjvDkKk3iA0f7mheG2S1nV/mgVPetdrZ64vLPHD+8IoNo1rTMSVuvtXJGpjCQ3grV4SjXuU6/qRQxJGPyPWSxoR4BfxY2dcUWKuto8L7G4urvm5k2BGbrulZWcK8l2ibzynk4GSZPcEnXAUKMwi4HcqHFL97IplVVMc+2caeZucPGbj1/844cvfv/pxfc+Cl4+/fHFDz95+cNPL/7wwcUfPnrx33734pcf0xJ2NWbht9pQJ09YAl+YMyQalTNb8j6gFu5NIg+dspAH6RYOpgpWJbidItHmLY1K4ZxFYZiTcDhP6cYXyOyXZPTD6ETncNpjIMe7R2sVe800wsprkaLcOHSmE3Mb2qBJr70lAPCl5SziRtSScQETz+Vq7ZcR/1Oo5WSPGtbjtNYBVcIP1YL4zIuZl0uNQ0o2clrNIsx72+EDseQZ5u13xGnTagkcSQeHZ/kmrriHArKj8h0jUrUSmBxJ7yQYpMO+XlP/UFe9FxbRfpqdqDXlu7pKhL0OIdXyVRWnClT1tQXwmWnfqkkp5A6S5TpiTVLIMwiMjt4UoPzMtC0NthQllwDHzhmzCyrsu1GqiXoJ4GRUyOw+5UlvWxoNWy6KEnlnHYu3UEq3Snc0CY4ltIuMTl2a8UBhnsbd3c/SyRiewXJjXT9rOafc6GxhgERx5p5pEk2mZBaHs/FAVosBVC4VMkQMT3rlKXeB8j4HpCJRPxXRUgoFLkbUOVWZJSrzCymhsu6uoZ8pK6qxhgYX3/3Tl9/96OK/fnDxs/9NS2lLQ2cLr9UkHErk7Xr10C9+9sHLn3/68sMPghd//z/x149+dPHxP3G4+ixoGQjxLtHnP//k5Y9+y6EqM+ESINfubLz8q+9BJ4P1lbsckuD5S4C5+P7PX/zkw5cf/uzi40+Dlz/7HuBRG7LCulWoTGXam8TDPksV1qzbWjcUJrwcWCRAJZcGvvCkC5Q+U2IBurUZGgnBPg/wNdfexAhUGNIVwtUoLO7HeRDu5elwUkTqITTegbeXN1eDlXsbgAl8o9bmLdQBKDugwOAv63Q5GqQZBLV5guGAGKzSHLhvY7fKy3FlXiFKDKXcbkuQyoQ26ENjnVEpS6wmOc+VB64qJVGAfq56bEI5nHMbGP6pLMvsbTEEzwjokq+kx4UyAiJzBeUrRv/y2SujP8pb4o277LuqK82wt4SNJ7KPzjD+SsblFiixbmlqmLauVrnlPFh+eO/u6uZWcPfe6v2Vh8sPVjeFpdtVoNphqahuOQ8fPXh7dWMmoPJgHOe5Gv4C2YRGu5JLVVRaox0Hsxon9NSaq5JKUPtay7nGjovwRUih4RQoZeTSFovwKb2tMrFX9/TaSZRfqyRBY8S5lqTXYLk06mG2q0o9kVCqth47t4YsVWlSPcxXC0A5jlWBwEPMa+vK1MOVmmVS4rJuWbOlJKcSGDaCtStMb5wEA1pucHdMfhiPx2BgOaeU8t6s6J054RCT/pyw5NZRX/gHELG5QWauarC7bpizjOBTSPUinwQMinhmkwBmeBEOy0qkmuhGMdmTug3M9JaZxWjfkZpUgvwa8sQ3Hw+mDQ6HhxT/VZH3qGwepBNYK3E8HDncedFyjMLWw/zMXmM1bFFoipOch1+X82kGaIbw613FmBRCbRiPYnYUnhW6rby2ZT5o6E4gcRcWXfuuCAvVa6O5fGSFOJnGhCpjaP3Wxqc4TNi+KEpEppi2HE0txXQnTCmVOEa9n/YONCLVh6BWNnFkOKMAUPaCr27azs4lU+7K5mYlSiUQakBgFo21GwqVXOS0Ymu71nyJqeYCYvn3MW+QkVeOTcc5XT+K83bI0oZa7wejfklNx77njftBdBvtjAu9JEixQ2J4iDQP/NQrv7hvm1TrpjIEwy/CEMImUhf3NJtyN1Mu4FCA7kFYNLI+Z5MEGbMvSlZO5alzNUl5MaaP284kq9t/fK8Fn5rKFzTKCAq/jEA58qk5BSoH3YQ4pOTALUpZBcApxKDgW3eUIJ0PRcNY7U2wGvZQDOEiwyFQdkS1QH2ubhsiccWprVBNKWPMCJltqWa41RR3lT5cmUADnUISsBMWzqmKkDPa6nKr189MoVFt55SZLQZvRdBpLVINv5Uu8Vv11VSvnZDf9aVNd50u6OvrKX468XNKaemH1Lb+p1QglEN5hvr6cpwmbkdSZ0qXGdnIQ0S/7GXPLPESDcU/xQwSdH6bTtlLUGw2lS5DmdnUqPPzlj5kThmbK3ku8rlyqXA7uvpSriEiCRJbpFg+RPMspxZvYoKyxah4ypqoB6/YG+AzU3gLVRFplmRuZVWaNKpcYu7CGzbrtsoyu1Yb1quFpdusikyxGLAKlwqtX1P2NNX/elcbuHG9ndhS0XiCNlfUz+RG19nG3GXRTIZtjXsQnPZZBSf8rxxcRdFGOxEz9gURkPtE0bi/bP3e7ZrHgUANGuemkeK0hfHSqETGVaX3wF0XKjfGAUHrZ0rmv5Ykwin/AfaoUMJP+Q/NRFXM3DAHK1XD7NmCEWHZEkPonvIfHX9pcJZbbpY07VZDrAkTdYVr4LnseXEQ55jrvFOOoTziwodQNXVLOGK4eycyHEGO3NV9HQ+5omFpWuVbvZrLjip1UF9viWwaA/f0MDopLyyhFZx2AOV1EcKNrzGz8OJ7WgsPCNtXbkJncKOJWRvoOoYe8HKmJ0n3r9Xg1aykypVKHYZVsTVQ47kCu+LarP2Fa8zxSq5vEVCKnkXDxT3D+1i6tCp7Pk0l9b3uusKgCSXXfEupgvnx1u6virTz4gLsMrlPjUOL4MlSb4hLBERGdKMRlrPiztoD15qvip2U1yA6TVt6J0+Hex89YGssK5KRk2namLeEi8utXNY5rRqiE2y6PO3FITeMJZrZyT3lowSU8VwtkQFNSaSTa/PJkg6ppIH6WiGE1lszG6t6MYzFGSezhss7jdTtGXKZZ4xJ5fbMcrZPQm2dPnbEaeecoshqSpVLRj/Ke1lMvoWuy10QTh6FWe+AKC8zKKqWSpSzw9ssCSMdMwRFEbdRuTfQdxU3JCafoFal1z/DiJ+gfN+EXhRdl19Z4cr9VPGdivPBu4R7F+8IGo67Lh5rVhgVZtACEHMBL5LFrLOYh9YXAAmJPDGIBTB9loDfpsI1w/e5XKIqBCTk+G267TY6mCSchxYcyc0h31lh22I5Hr0tDiJ0WQAunFcCBbdl8anNshv1+D5bd2X17vKj+1vB2qMt3Nji3RFOf0sXgOmMOmdTmyu9knKsd5CA7TzCY+poxpZFZCsiySdly1lAcsH/31xAH9lwgfJrT22UrGGc63glFLmuWMtc32HpFRCPdFe18xAvguwdoFeb8+U04EwTAOjMDdd18VqEKMBURnKIy6QGI7VoPxAh39n8Rs7uSQEFQjiRcUHKFY85dUosiNO7ITTENmmI2mAFcZcWF5WRU2mRYgKlEQgP33mEV01gR7njZ3qjo/C4zZfe9jjK2v20Z235xmLZ9IPwOB5NRmJu0BoI7ZHy4AAQR3hORWdQDE4SomHUn96fJOXdadOUm04UIbdothILUM7XlBzorDvqniX1dHrz3Fqb3uwW6O904J7KUnZbfoCe5zCi/kTSrztj+tLdzVOa25gkDso1WM+gjZy5RFnS44UkFe1IqT2ztbY4SKJIDfcwza5HyX45RrkYi9KYUgAZDgdUyg5Rb1ab/XFs56sbJUdTGq0EY7RX1u8R08CwZw+Hn6SwT5gS/BbPRYEMiugUd/jmERCyPx9fTG/rdWtbyB5lCNLcLdL5qLaSu282T/KMMo7ICIHX8jlouWEf2kXaRgzLa+ydB+EJedbRxF8Q+SFZfod8lqSCwgiQ7g7miQgVZnJVkgrIMqeXZWxgOvr7vnNNJp+4NkNqxQldI9+mw0dWarxaiqs4IXHFJwrPiUE1xYRFfoildThbYtIpJstydJ/c3OyyaaQ9NIRRODGKJhx6gWs82JASvyKQmVQRrvNhedMgAYYZxPuTjHa1ctDvRZS5UPh0RVFmyig1LxZ2ge+a0mDmURSkfVE4HtOxDPVUmEi2IHwTCEbi8WSSTK0661jOSPKytktNazu9zQHulv0wY5esPeSQIp6naJIcJulR4vBOgPKj9okp3tDRgPamgoCwEgRIkiBwRcA40qfxfwCkS1uo",
  "flatten_normalized_materials": "eNq1Wm1v3MYR/n6/YkMEMGmfaEmOjfTas6taMqrClg1ZSYpKKkGRe3eEeCSxS+qlkoo6cYGmbtF8aBu3eUGKpilcpKiLuKn7l3zn/9CZXS65JO9FLlJ90B13Z2dmn5mdl+X1WDwkjtPL0oxRxyHBMIlZStwoilM3DeKIt1pqjPUTl3Gqnj1+oL4OXD4Igz31GHP1jRXUfJClQVg8HfNWD0X7bup6ocs55Up2MSQpEjdF3mr2HjzKifQ4CaK+Gl9PKXPTmLVarfs3N9fvbTmr65ukK+hN2GAQwvYsm1EehwfUtGzYC43S1vdW7q/lpOW6fJIEPX0wcoeUdLvE+NaiM370+cvfvz/6+fOXv3w++vqBQWjIqUbcWl27tfLW7S1n4+7mnZXb6z9aW82lGMuLzovnT8ef/2z86QejP//q8uIV58XTj8efPB999bAyY7Q2195eX3tnbROX3oe1pvHi6e+MNjHGn4mPl394Ip4efW5Yrbc2bt5euX9//dZ6KcsZ/QPUezj6y2NgdvfulrP2wy1kdGLYRyE/wsX4RX0OxZfE7xlnrZt3V9eceytbW2ubG2JJi8CfcXVhyeiAXW0vHiYAqsmMH1/d4Re3Fxx7Fz6XzBudndXT1y3DassV1xaWGyuuaSuWGyuWFicIWVqcKWXpyqQ1V+atuTp7zVV9zVnrzsoP7m6eFxS11L5ojh999uLfHwO/8T8fjv/4BL68fPwQjHdam5gDms7xrw/GH32gcwSXGH317HT86cNpc3MB1vkDG8njw6ejZ++fjt97AEP4LCbm4G4uXSl5WTfM8eNfjN99CjyAweirf42++BIh+Ojh+L1PTmvPSPHus/GHf59jppqIJVQNjtCj/zRFSNutb7wN53DVubV+e21j5c6ac/P7K+JQXfjO9Y5xeWfn9MbFCxA9vlvGH/GfrHgYBztCFw5BMeMd+GTyOc6YR8tnn/I0iETgLAchlEoSPJEGiPBpj3hx1Av6GHThG8QkCElk4TrZiCMqRUHsgbDqpikzIVraPPXjLIXjyWix1LAkqdCkoLE1CpNGXuxDmOwaWdpbeBPON2UsZrwLfJLQ9YDHFGlAN1ca0LyyNLl9P+AwcOxgdDfxX0dEanKKOAkk4FPKS+lRCsjBsyAs9I15GZGj1CiVgzlcAwq6LOWHAUgwduDvxs4OxMidHX0j+McoJL+ICBqDXBKLt9/s7J6H4RRegsUbOQttLN99j7/CxhtQNRB4rY6ALhGf3T3wsCwFrHAN8rBhSDBEmoKhopu305y/Ip+7fMLaqkUAdbV4e7kKWyFfo8lhHO77AdNArJ0g2OrQ3adAw00dcQvc8ijgqRPvd7dYRpVPeiF1IwdjDLCIUvPADTN5cGt2cfdCKj1SCEiZG3HzxAMtDMcgvZgRb0CCiEwOOWcSbsEduIhPW/AI3ZSagnuVBCIfz/Yg6u3wS5ihCfwTcxYgzYIEqhkmvxg2yQ90jp5kARqBZvk2D91wX9RD3GRxnGrQqRpqG0d2i4jn7YMO20grDXM4gMVyojSrlzFRMnXlhJ3EoFYxm7Lj6jlB30ADcc+N0IjKPjkbyyIuJ/CFBZRXVwphoIEPdCgNij/qmzlpm+zT427oDvd8Vyw/7sgPcVBsz+W0F4e+aZWq0SOPJim5e38NQ1VVFkQ2COcZLQbRtoIfmpfRAwr1sG8qdWqxQAyDigqay5oqFUI4OnIm4I4AIw7D+NDhx8MwiPZ595YLxaXVhEEi7SYJjXIlrAoNDXXOaPJzsz4OKOgueOZu4/Yhuh874jTtAY5z/AaRAoACekgZglUpZfVoIEmQL0CFjAEnNapHYMiiFWoFljXHZNM8Tec1w90qFp/hbEFKhx3xv+FqTXT/V4OXlqngVnGsPJZBKOnH7NgB/bWQBn1NWolokHfKgJknnEnhBhcW0aYwMGR7inMpOECEAFWaBhvR4LqBYN85sc2py7w8/UxMoch7uqBqJf4NScqfEZBJMNZS9kQIOQ2pBz7S0WYAUsFSbQaxxD2IPIwPfLuzsLSrxVJQCOPGdBta+kaRvLoxpQUyqW1OTeUbZBSSTnBAHbcHYDlKpLbNtnby1YiQWLgRjnVyGSGIFPsqGKexWTKQeos9A2EISRjOYShBkHM+ixMnHbA46w+AZGGpgC2IfHrULtCjUTbEsEPNEsKqA8zCD6vGJm414UKiINjHFNGVim9XqC6RJdLZRXGV4etdsijvBMQa3QLiRuLidr3akGrp/oFCd1Vx4h64QYilgaP1GCZ+17yxtEQeMFW1J+qdsgrCZVazJsNhZSCZy3FE3ZTIjEOHahi/y8Gs1wuOimHxVLWayJZu1KfmcpssLcKfJlzEZWi6qARYZsqecYL8z4h5IjicWSeS8ZlRTwpT9liwtaacejUvbeMGYKvNDJLHkIpKwOwZW3FMhm50DJ4Sgq/ibZjY00mlJBdgnhWdjbhp8oM+jE4IFlIVOQ37zS/ObD5wl69eyyOrzFiQ1OtVq8H2DJGpBqB7qPmuLDmzaB+BhljITJmVOjklnEbXN5cWl98gFwl+ALM9o967SK3sLEFUTMGvUkvm8wN6lG9PbZlD3pE3bJh90ZlD2ktVsGBBf6A76V4ch3m+0atC2c8IS/ZpyoOf0GL7yA1S9GvdqRRCxhRDi2Rad3XdSoI9BgR9UHJszaoQK/zzjiQ+kEiYnHkKgPKQQixNkhBqUsQAHsTNAu+IQLgt7xl227X7gua1wMxDTcDgEwwC6khFNIz8LAkDDJIOSgRhcOjw29m3yynihug7x8RN9cuNyiGUe6pA36j3836MUUSoUBm0sqxJZW2aYX4wfAqKUN+oVrUVc4jC7chrSsvBVeXxxFpKQj55rtSka/Qg9oIa7emE4h6oW4kLuLnpKzQwu81wMn0dGqgLhqoaD0x2AiicTVGxiXFzRHpz2RaBU9eKitIsSnZpmBraObA5fvKjPR2leWjIXVe3rDlOrrrMOcAmFDkSNJ2eNMsrj6rzyiuFkkme/GZ0svLFht3w63Zxy1Nyqzl7iSiu1tz8HC4+z71nufa53PrVXHqqAbXNT16pHHqOG1eRs1pTXLfqtiXAoFAUKYj/X86q71XuK3dUVWuL9ECHSZr30bX7lzLKi4Qurw1AHpC2RXLimN8hjOIVTpn7gAnIS+PEjw+jRu8YyJKEz2mTG36N0XqIh0FdyZzrxiTBe/O8EAqxAYscWf2bzS5ifi6cgIrIarhvhEI0ENp9ltZoNBEQq2Rzrt9P/vT1eh30TRwtvh8kySudrULFc5www5hxlowY6mSPQjc/TEgYe/ti9694pAr/0IIs9nYlkLMaPL3JK/q7VrU5x1msVWRPtbiLBVjjFSYSlCJlB6BdrYiLJfVas2rEshIr1uudLLQY9Qas6iNW7p+FZ0L5nUXijVDQC6C1xmMnkZ2H3KR2Pr+CChuNPAm45vHTmU7FsNnVNhejLznSoNMuACaiJjv/UudzgIxbulwIbILaM/I4QU6QtuyjDhk0Mg60X0Gv2kqdL1aU7wYqKXx6b3VoQMymh2EQUTxfpPYua4EH/cnNl1AUa3WPH9irgZe+IwZMSYeBm4Y+uhTvmoaMD3irJmMAftPranjEQ2xolYLkb4uPAdTiwNqqdH4SDjwLCpjqVbu2nsWHpiSyHbCQlzqOwlv8tsNxWZ/LN5Hq5x72BqqeuF6+ZTGI2y0IVlg/GwLC98QMllceCxIZqG7lxhW9Cv5cI4rZ0A2hd/OhtQaFAjckeJgp45DCCNhvIaQH4JrKE9WsbViafNv1fVRWCDaNhQW8OTYwN/fcLEy7k39+MZOD8ExDeReE8DRmEN1YRmdLlml9QaT1ycvbZEDDBF+CIiURlAR2VWxcnD08TYiQSGWzRapToW0YPb/SredLdbOqJtUNovrb5gmvo2WpBAvlvUzJBcfFRX2XmMXPZy4LUhvHtV/ZyKOYa6t+jSMI1aBGLMpxfU5eoOXvBAzn5W+ejX79W/wtwxdfOqO/PRn96RMbzlylpqvGBHxvtVuUDlpgwrNSe5+hFQFTK5e21K8aw8pWopgTQVk86jWffnNQrwSFBq3iuJZxT32piUtYgCnLOLlwB1uHCzX5ArgL92TRe+GsQ05CCHmKw5lRZXInl9Gp3WwVNpJRGUQ4DgYyxxEv3R0HPclx8hfP0q1a/wWjcvV2",
  "generate_focus_review_report": "eNrFOmtvHMeR3/dXNAYItBstlw85RrwJdTEsKzDgWIJj4y6g9iaj3Vlq7H1hZtYSQTOQ5LXAk4gz6Yg2ZZM8CpGtxzHIiqIVCid/uZ+zM/sfUlXdPd09O7uknA8hJHKmu6u6qrrePXW/3WS2Xe+GXd+1beY1O20/ZE6r1Q6d0Gu3glxOjvmLHccPXPleDT6Rj76bqyOiarvRcKsEJjHV3LrTbYQ1rxryNTUndKoNJwhctUYOiRVu1Ws6DTl7jr8W2TutT5yGV7vQcX0ija/uOOGVhndZrr4Ir7lc7tzb59/88N0P7PMX3vrw9/a5d95n8zSVt87M2IOD68Nbe/HNfvx8O9p/aRXYNLNmztjx6v/Gu+vR2vX4fh9evoi+349vPxjePrISfG9d+N3FN99/+2QY5+w3333ngoFWYbrw4Qcnw/K6GAdqhl/sEzW53G+UzOg3O+fV667vtqpuOcfgp3rFrX5st9plFoQ+jTiLMLtkt5ymqwZ99xPPver6aqTqhO5i219SIyD1rgbSqdXtarvbCsvMa4U05F6ruo30IK5zrznNTsMNFDRfao7TxG86fhtONlyiN1AbFng1Nx+4jXqBTZ3FhZwz/PFgFiZKCS2gsTUGOsuHdXoSGM4u6HmLWRfPnbcvvPfuH6wRjBqoiVOxnYnx7f946+13Uzjl3FsXPnwPD/v8eWscr4TZXnQ6imFP30qgci4H+RTjUyN0F8Zt0uo2Xd+r2nSgaiNhYOxT9l675Y7sSUZvC6PkuxOCsdt4gQ2H1a0CtNNIb3a53W6oLWgSTICwGuTpB8NXeUGKPo3G804jcNOEC+Tz/KEUtm0QqbuoiCrQCaNMOUPs1/NsbmYCWw3wge6/kCON1LPzbHZmBojNIW3mGSmDHXu+oXstZFIyvttpOFU3bxWtIrOsghr4mRgAVF4nX8gJ8tEoEMOIruAGHD84kPSkoCOPkIXEGXTCEceejZZz6rtOzYbQk0e/XybfSUw2vCBcwCizALQWkfdKheO56oVXGJxli0CKrOVebXgtd94C3sAjtmtea3He6ob1qV9OBd4ieF0nYFdAMxqjtoC75GH30jnY6X0gxfXzfGmhIAi86nuhm6KwyPz2VXB2KSLblz+CaFmpFFndcxs19MxyEdJPfKkjQ3QlOGi3FZaaH9c8P89fgvkP/K4LvFwDQLv9Mb0WMlm3rlo/kX/iygeNkcz/Ow0I5nUGCimYEv25wmWVPYnCyeMvKcNqw3VapCekxUWQSdPjoQVIOPPajBkRhDL7binoXs771qXgNOotg1801fazlLjh8i3I7PkG6ePG6VzqfaHMF4PjPVNhp5lVKpUsQXfQaXhhEtwUA0pD8VzLx5rRQkXfdYGLA047LLK514H5OnCEryAQQlCijfMW+/RTBufnkUMIJcMVQR1ANZ3QriWZQh4fy1rqYIpVC+Do0EzeELSkh/jCaHgfA2WmABwOYz2418tuA4CWE3moQF2m5/jr9ejhGot3V+OdNauo1mnhF1bGX2/En18fu1iLx7j4q434yY8sevIo2lin1dsvo5tbLPrz03j3Ozb85n/iVXi78yD6viewrCwQI0h0RdimHwZA+UKySd2aYoP+l+yPy7SUHO3KH8tsWXG6wvLAEltOREnRe2X4zVqRcQ7EnBbbcbYgqKhIPdKOQakSkVRyOmD9tXzdYgz3irdW4zvbQIUGsjBTWbESqzAOZwI2QWCC0IDTUMoc6FLLKn3U9rgaJ4audNEOIHu3P3aXspUy7CJasP+iqgfgTdiSDLokLSPoou3LuDOTMEnrzHiuWA09cnQzXJ8bwk4lSDqzGYGbFXDB6NScLpE8DhbZlIrp8EK7JLmglFKjDTFPiSrIV9tN9P0w5mshkAILF1QSA4vc7ShpyqiI7pZ8pginGkqsOvTKJe4/incO7XivFx8clmCxONpFv93tuLXySTfGA1LVYB7nOSJ0Z0APejOKk4nckGeAUkjyRqoka5x5gFqw5JtVKRqrtLqHL9QG0mtlOcQXyrf0KlkiiX3FW3oVHSpfQo/p+cTi50GR87QuGbIqqLgzBRNC8wMKRhvMhtKdNJJTWnQhVOijPMnL2msUzhxPQ6r4LpRjgbt8eTZFbkdSZIWKdCk4rJQB3AAqg1SwEaQwXymhw8jD03yGEzF8j4AyUuWg22w6/pItKPGyo7Wp09zjCPtRgOUJazEkVBK2skL2r6yC6WUBRA/eyRyE7uoVnuXQI+Q5+X+7+GspyrOXaqen4H/hUh6HO74HxIVLZxf+c7pyujBNS1FHztIaCCG0S0EvSghtOUvTgUt5UMY0/uQJrERCzitjAG9mTEiCcAJ114Qi7S0UCgZ24xAVJUmO6ASBV1+yIYFskv7Ik9B8ThFV6Wrbr9lXvDCgBNLMcSgC4CmRq8FDogc4JMI4JljwI221W1o4mIxkUghJ4rhB60gXYXDUHxxCxN3diHtPp3lPiA1vPoi/XYv+vD3oX2fDzUeD/9uCFfeHu5D3fNOPe7Di3ma8c8Sw/fTZDStJfpGZrD1W4837gJ8Nb/QZd/+M+3/cAJOjrw5ZvLcZ4XYc83CzF39z11JJrS6V0S2iF73BD/vx/fHoh9/2omewy/Ze/PC6ZWQRwEX0HP712Mz07PQc5miMY/v/v0c7L6ODR0i7gTi6v80AHT497hPNe5sM3kAaLN7sRXe2ZPZ+ues1ajbUvuhY6u1qN1ABtshGYi7Ub91Qf/05qvw1m8IL+QWpbaqGE6EnHXyTzTD0Gl1IfujRMxRxvPmjFn2lspwcV6Ir8cNNkHEGLrLDYFxAR4+WiuHIYTqE64Sp4ze3WMinQzZLxdIC1FaYTOVkHiByHyBgUjpUEO2a0PEa9sSiW7lmrIb1utvI46344Ub8YEOam1DNz27EPbCppCOrFRfzFvs5m5vRRvTZ4do66R+dpj4B1cJRP75/yI47/uj2XanF0XOobF7GL6A6efI4ur8DWs1VehVsQVoBkE22TMaAm0T9LcmHANv9HEHuPCiZBCH63tP4zoPBsx8gyZ+OVnvRzo/TyPzO4TQQAP+m41tr0ZPPuSHqDsr0TIAk3ttGUxTOafD0EKoq5GUGH8dRcLcPnMS9bWJrF+je/CvQHO9cZ6fAU0R37woucQ+s0L47HByA53t+Fzvo6+uAH1zIqei7l+QCvt6P//IS9+QOTJ7pjXXuHKDq+nY92gePtrEd74BQv7gH4mXDr/4LnIt4pv0ZPx1i7Pld+IOMcTc43Hycycnw9ovh7ipVhDNFNgt1fJGdoRITeIH9GIrnyeMiCL0HcgRS/gJ87xRxdxYdrA63jmDuxWr8grvyA3CYX0KBi7KBA42e/RCtb+nOc28d5QuMD2/so0zAbQ6/vB59jwq2RZ729g8ZlGrPdRlxJMrVLajzsGuiebHCigGRHDoDbsEwBIDuEgwIKyllEy8ikHutmnutKN7QrbhU14F/0LdfKOset4KeCrKa+VktpyLrhhI6HMleFkZymbq1rG+/UmLifeGUdFWnKitqUKskcDyfTMjKAUYLVjFjH2XuFKXj1e14u1dWmMN2x5YpEyApigPVViDjQbXtuzQtzkjop7my3nAWF92azHWXuA8GsGMou7kPIVjDAyEmaLeCUcCKlv8rZYCzlDtCsil44QWAuLrwWuOzcbGppYYgJmQmp0s2z7Lm9TBBxYpCosWZpPYogv83E04jRgE+M2adCOWMiZHnjBIfsJmfNfLDFAvZ2aaJUssefwredPJZGKkV263G0qsgxYYW3rKoXl0ho5L8yVi1zp6Jl7dJRIvg1fFqTUAT70fd2mLTJcGaNYaJsWiqh6b5qQxkbOW0PDJCLjHRrDLLULZsGL2locAmNDoSyKTHocDGtT0UhTJLKysLz15JdxJeazHxZQCSuIJjQJSjsMqa1xgDptdOsN44mzEgmomZm2lWOwZUs6Kg6TQaWVjShjoGlTQ5gUJCy+GxUpJGlYJTE+MOj8ym6QVUgaeglU2NgZbGAWvl4+jKlVQZb7rEsfE4OybzqAQzy1LXVioyRJGtLyuprwz62xALKceGwCUpzAp0Eu0U1Y+9p2Xsj+N1wbKUPEfG+9w0oUTLp/gFgbgfEPn1spIgX5RZUS9rGiaWjRaxt9aGW5sJRFqbECyDrUpGB0U5Yt60Q8dmuLKF8i9MOLoHEF9ojGzBD1A4NZIgJKSUwnIRiCuBlFeVKYF0z+PxZ+0BmsdOj7nA0q6rL7WwIXmpRQCpbpLZkX9FZkANbuElUiGXDWgJJRcdgZPez+bF+pFSnReYvEAphddCq8BvSm1sHebVZQpRURhzj5tL7lmpMZCQfqJNsTmg1EsLZ2rQtFUVpEytNKJQaiqJNKnxJMCkxkfDybgFmidOLTECRWpuTERIrTre+acAMl38COWZDj0tmPHOO7UycdRqXIRzoapOt+aFZquDt474oTCoaeOHN7DuAI0YaXDM/nJMg2O2JJ2zSOqxkgZ/Njh4bFbEJ+lzjP0gjxZAcSsq1OjOIyifqNO3/5wq6rU17IyMaSsEV9p+iO0e3AMvuhjvcDB+w0z9BmohDDe3zTbK4NlLrNRhiiV3zBOWE++i8dLfxoIcOyA3+9h/WM9uFBiNhqLWg0ie6dp1Zyv629HgxVr8ELh/+CB+8VfcfHCAFFJfAttA9yYX+NZcSTRmZD90+Nk6Ioy/3o+eXzcK+6lx3Z1Jhf7g6WGqC5KCQkqp/7NzGP2Nd1G2evHuvmAEL/DhhPjO5I0f3oj3Nl6lMYV3+Yh2YxsWoohk6ww0prdNOgaePbMVMpXqI+NaErtQN9V+SwROfTfZLrmzN9w6ylZB/flMidXxWy/WaQde6H3iErbPdrBTR1seQhVu0oWZCnYBtbaRbBfx9pHgF1Kh/95CBKgkNw9Fo4iJHAN1k9vXFz1Qz3skYNE5y9ZNofJgX3Mzc7+IPu+x2T+9JhtWMPQ6DSUdrFtrg2e9afwf7/VwCltWZAr3EAceyO0jOGN1b0Htf80W7wHdB3dB1kl/7+Xk9tWU3mFLNlItQtkpix4fJt2CUHejdI3CO2bx1j5IatDfOOYAXyvhIUUHq9hjHPQ3hfKmRJfqGJNe978Ej4fqaXzZwpuWRZb+iCUZF9+r6J+n4BKSzxGd9fMebjSmFZmVkaKkM1JQsPVEftREFYgPzQ6q3sOiluPaOpjIcHNPHhaKc/jV8S1HC/SeCCP3nXJAyypvOTUxWTq18pMgIaQkkJXJ6RmPjTw0TkrPKMaOTc/0+2hIPII8XRbJz+xL7+HXdB2nmnz+B4P4FUmy4E1/sYsh/iLN5GtuUPW9Dn5AOW/91m1Rs5TuhZzLDZdRcJ8Swb3utZCegNEn9BQC+SWKF7RbmCB2umFQEikk37nk1GpIJm2Zt6amOEIQD35mt9Rx5/ktmLgemh/5An8iMnGFczw67QP8iQiBh+ORiW/wJyJqOtemZINZYqPvkCSyN954w7ipFnj0cxVH3XRAJ9I3grCAX/mr1ThuXEbieCm52CsSUEm79RIjQl3Na8h5mtJHBLs+XsPXrat+G9RkWUdwEgv7J5FwY3t1JGnLIyS5HBSsNlUYtk3NPdtGWdu2xYXMBZ/7B9JpGho=",
  "make_result_files": "eNqtGl1v48bxnb9iyxQwmci0bN/lYiFK4dhyzojPNmxfk8CnEpS0kllTpEpSZzuKigS9AGmvQJuHC9LiAuQhDy3QByNJgzz0F519/6Ezsx/8kj/SVLizxN2dz52ZnZllP46GzHX743Qcc9dl/nAUxSnzwjBKvdSPwsQw1Fg8GHlxwtVzrH8lZ4nRR0TdKAh4l8AUpp24x2PeW/e7aY31eN8bB2kPHgTAyEuPAr+jFu/CoyFmohEPR2engZoKIq/nnkTxcSeKjotLnHHqB5pg7IUD7naicdjzYp8D/8b+2t7m7oG7vrnHmkTDAon9AOS1nZgnUfCYW7YDwvEwNd5e3W/JpRmcnGR+Pz8YekPOmk1mrtTdy6ffvPzijxef/vjyTz9e/PCJyXiQ8NxiY721sfpw68Dda+3T187OAZDQ1BaYuVh3X5w/v/zqx4vvnrgvvj1/8d1/Xn7+3DSMV9haFEQxG3qjkR8OGEmfHnGWROO4y5kXBFGXdoslR5ynDkB8wGHwhMXRSQI7x5l5+ffzy8++fPHdvy+fPn/x4/mL7z82YZt7NMlPu8G4x3sZ5hfnzxYuv3628PJv/1wA0XIkHGNtZ2tnzz3YcTd2ttZbew2G+3mYpHGNwZ82SDUxGHzMg/utB61GvTFfd5burKzcu7eIn3tvrCybDWYCCbOWX7jSqDvLK7Du7t037i6uLN2B/7gQ+FALN+CzVq/XcRhYK8IvA/w9oLOytHyvvnxv6fXlRQJ/+g2smxqt99e2Hq631l3ifx/ZJHwbG4Bvaqw+1BK5O3vwFxYcCiYlC5KkwNg2jK3N/QN3Y3Or5b7b+uA9gAEIWPnXF99+fPHnjy//8Il7cX5+8cOTy2efmcZB68Hu1upBqwKgN9oC47n41/c2rF3de6d14O7fb8Hf7dUHLVi3HYWcsVfE95B74GEeeNpjrrbcuN9aRd73W6t7a/fBvt5DEZfqxtr91tq77vYOyo0jiOVtY2fvHUItR5dxdI1GDz7YVaN3cHTdAJa33P1VEIAmEO/hco0t1didGlts46KD+Iytsb4fJ+DmYEAhe3thfWEV2BJEWoDhQQvEQpE37zS27oCYRKg4c7exdddEgi2C23+4sbH5Ps64S/Wl1y8+fXLxlyfs8uvni5fn/2BC16yiQec0SE5N47291V0hzea2q1FubrjbrRbYAWA9iMfcMDa3f726tbmeLVm7v0r2Ec+9+VbDXHj00a9enYMwAtELIlzY9wcYK+EXRA4IHGz+LdqVBtkiRIgjL/HSNLYgLDpJ2ovGoBIz5hrUtMVS/GRrnNwKi4fdqAeu3jTHaX/+DTA6HsdRnDQBzyjwuoDjCmqw7kZqsOYnUxPi9/2w54KsLmrYOuZnEJF7DUa+L4OImx9lHwmDFfZLqsL4Kzgaemn3iCdoTW0a6EOM60cB8Mf8kFkqNtYKUbJez7lVaWoxC5vu5RefXz79Ki89KAtONUnB8RO358dWbh4/oJTUD8Eq1ADyhKcUciQhB0HUscxXhZWV4MWRQBB0PBQmkQEYgy3w4jQ58eEgMn//yzKKmWxIcKlbZIYoYQC3SopnfiKUDpyXp1B+CTuDqtwQB44ZHvYsFAI2XpIOeGjJBTb7RZMtNnI6gsMWnfRRaDq/jfzQ6puMzbPJiHQwNYUWkbLGAAOwxgrRLEyNKfZ8PDXPkpQPW6d+CohapyNIKeBs4qcQ8IIzhqKh6klJnh/iiTg3kRJO5xy2gew0HoUT4msqXSXmkOJoBg7rbeXRAQRTN+WnqfXYC8bCSMF0tTvTqFJqJrTEZ5p57AAnscAmx/7IUo6TeKGf+h9yyjtQKS5kFJIieUqRbOp1ArQjGHCG3jFPIatJrEn3CM4zV6izSyY5O3ZNhciC86b4dghH4KXcIuzFJTFY5bhjxeaj5DU84hj8KQlSxRiLKdNhRR3TrIoYUQwqd0Hn3WM3jH6OjmGdn/gheE/Y5QJRDXSQ2hWAvjkRmq0vw/5fDdyHrDK1yYuERBAUACEf8EJgyLDCpJRAoha7BcYDSqlYkiKMQ4C65w/Aoq/Ei6vsHMdyFod19JW2k56NuCAUxQN6mmFDMtzpFVdrFZfdcFhWoBXaggbUYMFmlALyUc8yhdZpAmKNHLbzsVDixK/Dxcb8YruAVc5mdmtZh7+x7Par9iMbbDc22aNF+CalzgbIDJ0WFbDrnbEmODmFEKXOf0jEMZxaXUiwlcLlMSd4h10K0Bpg3sHfSgk0novN+Ax1RZryODwApV1p/jgo1NwfAGKC6w+oJNC4Bw7qnYqReNARmT0Mwu8MHXCN4DTqjCHGx1JaHpRQwCnPT3lPo5HPyCHaSpFLidbc3F5vvQ+WMskApCVX8EOCOOQaOz1dj1sk+BO1eCp+gtdoCsl1ehMnJ8bLCeQP4oObvyE/+Ltel7+nsxHlngGXNAcor9zMJE6SGo40KCjNNA2M21GAjJRz6rLMeawO2pIFmJvwv4ZT42HYhC/bNorJQfFMz7FbUQuxP+CpS+WDddKxddSolB8VhZx0DiuL2kZhgSOqk3ze2PNSz6Uo4IIcIBfpCHSVKQfGUTlUw1uLNTaEXALkH3qnLolerXFs9hpbzEWNTjEQX6W8fFFkO7lwTdnX7XDkS6gKjt7tcaiCq4ID9kLWkyZqpUMOY6riHUa6YuTy+dcvv3xGI72iBaBOSe0FxeIDaK3GMtWWtKg1IY/tq4VR2G+nWfXBg0KmyrfFe622cxrTLKNqFJ2qZDljVaRm+Y5UTd62l5Tvc6/nYhkV9xIr8JOUcrsGFThSlWmc88gTNM1CGyuDqgkmojA4a2JRKuPyaZePUrbL46GfJH4UtrAsY16CM43rkuY1auBRh4whEUodHLYWRLDSpxqgBcgDUbE32ERzIjN2W7SCgI6IoCdYphXCBQ1rb6ZzaaaPS9WRmkSpN9vbNdA1Zpkzx3Je+T85+s1meGsLVKnPz3D7cqHqhWfWoRKwppmtaVrtm4pXiUirDXSuUsKqV4xizEL75v6xL9qMqP8J/Jk2GNkfDBEmFo6HHajRI/w3gMLmQ9F7RJSmPZuj3AlClqBqzMLqScVLTSBvNpCTWnVOiQULtJaqq5SwsEprcPYqVKlchT9nUcRDWfBTPfbtIsBUP8mdVRFFKEDVhV0wFUKVWHKm1FBKvOEooC5JrpUuE7cuFLgpTuUa6xYmHzkfI6TkZgJ9OdEQw4dStjYV5lCWU5qZq8wFqUOAabPXmjIi5vhzEp5KLrAvVANHLxxk2D6QSwmJzd5kd4sGmJ9W9tGfm2gG1Xa3pw7To3p729PiIO1mezondSGs21znqegmoHFTRi6U35CGiyoj9vmp1DtoTonoQ4BN8gWccpl5hv0H8JNJTk1Tar7nHKJPDRnChVgVhdleSP4gl0x1/60z9oOeND15G6DMpkY9+hPXG6dRg8HxIqqTYm9e53jlHn45w5scqwqhwR4LrdTgB7BdglRKmQo1R8JGpUkXW3s/1RRLuS0d6Pgtm1hFUkUtFueUMQFw1s/KtEWI0T6LUDaWKjhcuRnIWYC6kmmSoq0PoYIsYqmxKrhdMiBzFbnoFu54OmfibGbIuxdj16JRsiUCqOU6phK2aqdFw5oQ4BStYyKApznMuk9GuIS6ZkUpY2bqodGYG+hcJWNhXkq3S2k0YlGfYWcLMqgEYgblHokzAs/LYo5JRhOPoZ7w0yPdAOxHY8hqlSNLS0OF0GYqJQg0ynF4mGDTfsjjAXdhv90EsxjYHTFCKYjsY5d6clkkhmIElUBViUhUMEsRIzJdaVauH60cAaGzEy9Evqm5V5yVSR+kYsKk0G+tOBzYwn1gDDYZkyMEAgODzCJxCDZpK7OW2KWPKGS5LFSCC+gs62pqocSQTFS0zOA/YpkqxXBALRJqsFWqmOU8GmkRnUx5VCNR7hGZQzROR+NUt0mluRU7W7k08Iq+avXMsI1SwncDZHauZJAyv7sCckZ3rnoe2aX+0kRxiGeaojmdKIDppHQVNlWNqJMYvDyrJMD/RtjbzdUgeN2Om0/JmRrKqTMz7JvKlALy/2dpohDfWJ4UOPhpJcqVnl+6mKyxq7b+SgylC8wiBrnfgrWOk3iPuZXbEH0xEIGzYRYpd6Zi4VVTNkqV7jWtZatOHXJLLS2aH3ZY9Ix2Qj8s32nOuPSUSRW+DxIDk+rdEGc1HoyHPEx3aQYF7sb+CAVumu/wkMe4236YjMRrIkxEf6ZMLcFzL19UOPJkEpQcr9dzPUnCMufnsWydR5vALi4ovKkMnxLRJopQY0c8GDXN3EsSiprD1sXCBE4kOpiwAK9cLs2+0r+BM2Wxt+SupAftGbdksXoFfj13IsrOx1GUzuZtxtsqmlUAgmwH7DKN4jOZ2T32+QlXt7fJDdQxnZgX5zZQ97rCPBJAB3EzHqO6BKkdlJUSl+y8zyXtlLiJ5Jxy7RvI9uKzeUgmrie5S9QoGwF1e7gl46DHOpx1Yw4b0qtRMgIKpBBMmcb1qn7MYwrWN4iqlmUJgEiJrsceRvOY9QhtzkPucz2VPf67sU8v/IwCvwuRtpyg6exJUAVSCV1nE3H6QvKJCgC6bSRiQOJkfST9YhUl24U58W5U8V2Cyss06tYmF/gVkeJpUCVUBJpBbObLOJU3GJpX8CRTFPIDyVFuKOPHKLW+ZjcM9SUZIcpl2rnXNq5Kv7NYbshAjYVhGrmyJmjetmBsUr8JGQgjGshAKg08WcbNrOhQjnJDQaaipdevxCkmfQrQ1kWhgb0n+TyjbsTDkvcyAWCjmoE37PQ8FjeYVRLfgWTAijUjUFmYdi134Nq2nX+HQWpsJpKiSPbsV0pKly+V1GduO1Jk9Jt8sguqOxfY72pPa7IYnJQI626GNEN8gwWvXXMGuSBJFJZJ51EQC9em2ka5vTFX5K7x1h3sdMjKcWGiCIi0bK6gHDIpiLsuxN3GDZ1ByZ0zPMb3cigKutFxrh0ucWp6tARCER0D2oB1vJ1V/priQUDWlLk57CFsVS5U41krX31ySh1NZaGFHthsga5L0WtaDJ2TZ6026RREwrhWk6qoX4eDGOvkboSHIcjOJti5UMFiKg80oB+GIK9pz7g11bgw79NMTOSPqdYWm8gfU4f6UsCcS/my69LlrutiDum6pnrLCxNK479jxDNb",
  "normalize_inspection_materials": "eNrtPWtzHMdx3+9XbDaucJe6WwKgnhefZIQEbVQoEIWHJBuAthZ3e8Cad7uX3T0CIHgpWaaq7EhVcarCWJYpllJ2nNglVxhRkukqfcpfyTcC/A/p7nnv43Cg5VQlDsvG7c729PT09PR09/SM+mkytHy/P87Haej7VjQcJWluBXGc5EEeJXHWaIiydG8UpFko3rvZLfG4H2T7g2hXvCaZeEoldLY/zqOBfBvvjtKkG2YSMjuSj3k0lNVuR6N+NAgbfaSzF+RBdxBkWZgJQmURgxgFORIivq7Ca9Naha6tJll0iK8MLj8aRfGeAFvOwzTYHYRN9pQnaaPRWL+ytry64V9dXrM6hMgBNgEpvu96aZglg1uh43rAkTDOG3+1uL7EQVU9/tGK+nphHAxDq9Ox7Ffm/NP3f/n0n3588t7jp3/3+OTLH9hWOMhCDbhxdena4ub1DX/9xubaFdGCvTDnP3n88PSX75w++MnJLz64NDfvn/z745OHD08/uQc4P3nyu/vsiy0RrNxYe33x+vL3lq7WIrnsP3l4//TjxyeP7hpfFJIbmxurm0DM5rVry28hDj9O0qH6vra0Tj83bmzg1/k5HeNnD588+urpP9y3G2tLbywvvbm0hpSsA6BjP3l4z25aNpCPP09/9mt6e/+XtttYXLvyneU3lvyltzYQ9tj2QCTws/fSbfpJg5R+w709+g0Gt+1JY3NlfXN19cbaBnRYoOAdR149vff56Sf3/dN7906/vPf0gw9OP/7Kbvz1EnC9CMxATn/+k5NHj21Ae+X64vr68rVlxUji/Zd3T/7lQ7ux9NaV65tX4RsfsNXFjY2ltRXqZcOCf2nodZPhCOTISaG/P3ny2Tvb2UWg5fRnj223WQHz0cPTH30IME8efXH6/n18ePzwyefvILDbaPy5tdjrWUkctpJ+38qScdoNWyRiwSAKcKLsh2loHeyHsRXgtBtGeR72rH4y6IWpRZC9qN8P08zCmQEI832sPEi6NP8vgayPB3krxwlC8J61dBgMR4OwDaOBlH95n5F08pv37DYM+8LLMACvL65sLl73F7+9tHLluz6IHsyQ9Ta01c23sjxtWvBnBwd00mgsr7wBAFf9a8vXl1YWX1/yr3xnkSTjwjdfbduXtrfvvHbxQuPN5ZWrN95c96+C+ABrEZAkgnhmX7mxYjP22atr8nFx8y3xuLJ5nT9edPoA/vpxNLGBD6kVWVFspUG8FzrzTWt+znUV3PXVjalwQH3jW1IJOcDC22Hc2UjHodugImtxL4y7R23C2N0Puzdh0rSx91SCDFVvaXgrCg/CVJVkeRD3grTn96LUV8BnNXolyMO9JBXNJj2tkWHw/SRFfFpRFBeLQJOCLoyztpWPYay3QChXWREO307T8jxvp0Gg3wJVPgrT/IjeemEf+jEA4bkVIkonCwd912q9SkqUEcT6CktOzDQrgnhIpWv0THCwi4LIagbsWedPPs7Ue0DcNlgsis7iNM0d9d4LszyKA7M5WBUZCE58G4i9srix9O0ba8tLaooL3juyq9izjv1Ca54LoDEK8MGzuNL+j7unH/3aevrhXVCBDmhDmFcW/P/pD99x9apitAhnZWUNWgxkR9FTUjLO23e23p5rvbLjvgAKZqvlezuvwcO8w0rvfAOaxxrLbrMeCSMDtRlRAg+clrMrTanBH/lPHW9fbC1U8/ZFYM+Du6f/+gPQ4BZrBvh68uhzB7Qq6NMavgI+qFiu9cx8fVHn68L5+Epk6HzltEyt9OAuAj5j1VnqzTYw83N1Uj8/x4YGOfzThyef/1gI/Okvvjr96B9rBgbxler98AdYQGXuM4/Q/NwfIPrUNrKJKJrOXqIWYanSM7D0ci1LL3vWyaMvTn716dOf3z394ccnv787G1MvE1M//NHpuw+BPBPHu5+f/vS3z87Vy38AVwVBUJORhBqCqLLPV69caWZWv/C1sxrU/Twaxe//vojha+LyC+fi8jyKIpFzXjZ//NWMVRWzcX1HA6GbxP1oD11OeAJfClwptBBWwJBl6zz4TOBUAhPAfjjKvCzvJeMczPs0lFVtVxkTCsbTIBxY+JMeOHode5z3Wy8DN8I0TdKsA3hGg6ALOGpaA7gzWwOYc7fGut+LMig48tFddfBPm+wg6w6aFsQJ+GXt5eFhDqYFvBOgpDfJlCcZ57YiDr5hHSAwSPPsIIIW7G3499r2Nngv29t6RzQ7jGBs6zmqvPVye2cWhDW4CMXzHIVWxnvfz87R8RKrShz4syIHjBbJAtwFERvnwCyshEg8KCKMCCQxCrizusobEOBnVq+oaw4JsF1U3low+Sbb12A4H7EDfngYZXmm8ZK4uJskg7aORvSbg+tD4Lo6vigjo31WfFGG0JXohjcrEKkZDhiGwc0QYArkwLRBIv3kJndpGLqsC54QOEEIU+qviNtsMWeFxX2QaPZ3Z4e1iUOCDXNcBbqtILPCOE+jMFPjhY4fFh6h85clKTjPDgdqWjfDo84gGO72AivKw2Gb/pJUel1wvtHJdtzCHOnuR4MeSCK2aV1iuKmKAXUUhQBFsE0Ow0cGkA6SAz87Gg6i+GbWuRYMstDVgDBIVQPFWXkQDG4SWOakSZJX8RFLONNAmrs3geAthGWyeQB0heyD6lx3nFK4q8M+eKNk5Ljyay6cUYMPUAHgBzDcjjm+HJmrEISH3XCUWzfWl1Cp0lgddk2UozSKc3Dat95cXFvZsboUxSTBaVvHhiIR6CeWcwxoJq7tmsQlMTh/47BhyAEfDjYQ9ItMpKhAeCtMMxAN0avCoIN2YLXMYsleLxiNwphXN0kJB6wyNlWurcmJWFoHYRD7uEzDVAN23AoGY+azFvQrBXRoZaGJmKdBnDnHXZhWtm/z/mLfqqMzE0YlYQcs9OsRDnD9Q4ewmyBgPGTjXTActrPnMFJnwR/65oLCTCMhLCbGlH2yPUst1TiorFeSHaISUC6gWP0x8DUF4wL6URU/KmPo2/4xPU9sXd9RkdBEQT/0ZYiDKxBQ+G0ZSKYQSSHmwcaFAHEysYmEXMYipI/h0Jdd8eXYpsgm/fHsSbtaVGWBakiKVUEm8COfWnoY5qJW1UUCdJopNE1gwGS3khVM3/8JcUKwAcPgwSC6HfqwItTOOFOy1QJRJfbMYHTs//rRfeysXTeVHOe1Njgcd04ffHrn9N1P3W2XwMXMmoYa/ZT3P3760QdQr6KNIvSDT0/e/7eTz+6BwT8D9LuV0JzFknzyU4LW7ScP32k9ffCLHZN2xltQfaChB+Ne2PNZjM7BVVbF86oNlCA+crgT5WVhkHb3VTWXSxt9RbGqC9tLd4X7hz6ai89isPNCsqMlk8CoIzXoqqJLpRKfl9Sj+c/fMQ6rkhYWtMw6VdpXGb9lI51Cr1H/yBd9dwyNp9kMwnkGZijrjrdqMs7A4MppL4BwJFQ81Zj/FaNJxBcHUqDyhP9c7aAIML3vSLs0hNPcHwZ5dz/MVP+xlASuKeu3ZecLQljZfdI1Z8jojL0S6hcXR0mhT7Z0xTBVEcxHq6Cao742Gpk2nsbs1Rpo8M0CpsvJkDO+e0yF6kt3QbvrapetLFh6E209Y3GI4l542JQrQRiPh2irhmzF0cYZFAbILtamKughD8KYg1kta14XLCRIVAD7c8rQKya6Zyw7SLtYb0pjzpYXBKF1hfpZubTKUd2FhYKWVtyRoLc/xqhK5I2qQnBUpBDqy73w6HH9y32xvVWxAhrKgfjLdFJ/PBjQK2qm3vF88/JELADCKjSEh2BLxNPc1d779nEkTF+3PXe5N7E5pWk4TID0JN3z86NR6Gfjfj86dEZpCD8Va3ZxxaIF98mjL3CD/Kfv3cFN8k/unXz5DtsBvcM2a/A7gUCRu/W24+5c3Ha/wRc31pSyeaXKgVHmW1W4EeuzfVfmzelT2di5ImLZDqPBYRYaIh9T+ZbIQnRN9fDE336jIjjBVCHzqJBYvnyMBhHMbttfmFt48Y7/2sKLJ+/dPfn7u/g4J16gg9hGEwbqkCp05t2tuZ3isIshf9t5bfWbcfIqH3p329vOLmLZbtI7etV7jrjGOTajGBgbrWTQkyxQDW8vTcYj50KcXHCFXCA4tgagOoyNZbbpmqDUcM7WyBFWcqs3bhkpgrCJZx0jsOllsJHUotAcuiMeVEwVUXYEQU2NE0w6OuJBfSpR1CkawyUIEbjlQjpIgh6T0SjMHKGXioJJy1CVB4CrwxbrIg8tCFxt45up9wVa7mZrTWi+OSUzdCxBEqgrAVpU9nqQi9U7S51rnr9f4fqbcYtKlGqdoUoCn8eERvMEKJbpHQ6yQ7vs6pcIU5vbNNWnKRBOv2CKWyQv0JIUSuhhgMRyxuD4Gs27JQdRCWEQYRrTUQYG99IhKA17JRGUMqosij4Ba8ew5moDy4cy80yHgcfdREt1gTcxS4S87o6h1z5ZASCv1cJGksmihio3hX9raukqeh0RTtw9YirmmPXNU+kdvLMoOvwROikImIjK0It2XRM8NUYIYBmLYrfI9BG5MKZseKaCKHxQToMD/iBzHt0ZoMvO40xNPPj0HE2UPM6Zmnj3PE28W9/ExFAAxGNiP2O2OVVgIIH7ZhSAAEsTDYe8NMuYKIALkIPUBjA9HHhtghJ0K+cdnxQkfU1eWRqNoIt6YKmR+s6qPGWSMtTJIrobDkWCo3KQPSxmK6nASLEaLG5aEm5H6AGMGKIuxLmo+AZKmrnCMDeZ9UDuLFkExsJqNuRFcRamuTPXZChc1UpVM1pFzi1qUHnQ5EKDYW2olC1Zjfug4o38LdnnSOPpTpnJ4+EuaKsZ2Mwhi/Eu6eKpNT+1wcJ8+45/ZztzHWEXQREUeHdaWKptE5NBKo0n8cU1ty+Y3UVa1uuDNgRqU+F3aoNZWLQYxWi1FIx73Uqad0sSzuuhM8Ufae2LBQvK4s8/iNFjr8ZgcQgREeqBsYKhCTYrCtxvsomhK1apzs/QuXXWNDVQNccV7xqGomgqAxQ6XpkK6WHdzHHbpoFS1iCIQRDQrl77qcPeXoh7H2TsyoWQ2biIGuNRspi27fZgPrrM9zQ+zmwciCgGfW5o4ZyqqVRUSFrHxYrGeoKaELti8kLicUkv6kxD516gcNHbn6+M+QgQ9EaY5hvvog6K94S/j5PTJOrIXAMEnVEsCCUPM9O0WPVaPWWM1bLksjlT/1kTBELIFMk4jv5mHGp9mNEaKXV/0tAYamItslVMS4xqkTYxwSVblO9ehfPVCpTgNtYhM6w+LpOEt55lrqvEkisjUyaL+rsolebs0jXTlLlhzovqEKMMouQJOJHUtunlm/uyKpZj7KNyvOTmaxhpV5eRyXdN30Ae0sZpdeBNBgrUxs4wZMcd0O1A53AQ9nNJXbS3n1emB+heFs81wIouOj3FD4SlHH2grWotyYNoA/ZnMMJyx56Qkr9UB8Gw16GXqcqgB8MsL+YTyPgP/sOYBT/h4mX7wcILL2oTnqUUwMJVTGKw012bUgr2QdyKW7fMpxzHN/mkTh0m120ODaMZ9Jz5uYXnrYsW/gDCXbuYbkPEeeMRyrJD6Nxib/e9/fCQ99E0H3khG51OR7wztnEpCG4F0QCdeF/LhjbkVGRN890QlHDgloxB6ukYyhiqEBJiWqVgo1kuarHWSE+WBZR9bFpTUWE1cj5hoUYeH7Fkk8wWa0KfxbtIpNibbiazYmEb8/M9vJgfBTLi1erEwAKeGIB/boXhytx3RHUJY0SIHxMTCMPEPWZk8EhRDf/U+liz4cE/g0ltQ0vAhZ7VG4dWnljdZABWEKW5H0s9IOJS5MmvjWM8k0X6A0yMjSQBPR0f6SnyCktGvT8u521NVGg7GR2xuESWdoUkITLxDJbgAJQ7S+BhKf/Sbac3sOXECsDstWYhNb+QbARmbzAgIW4qSn2sAtDVUs6AgUBXO3ggqth/adne95OIbSsYm9cOghRbccUmtqudWkA8wKM4Dnu2jKiwrutDXaS3M0WKzcFXzWQ3IzCue0qE0OwzgVm6lmIUl2fTGmSn+DwcwAWp8JBHTZnVpzAULElFDVSPZJ/58ArrX1ZhA22mvzLYDiI4spsV2Dvsp8r1xzBqwTKqBsPIaG3oRMZXOUQ5zKrOknSMOUBcMqA0aTNBNRaaNVAAOpokaim2RqD2djTyhyEaK8yyiOI+mIH8LKX3vWi0DAXmQsdD1AjpIZS+a8AKB8GevxvlmfUX1tzhy3NzZXNQVKmyUSjwQhmy4JJ3R89ffsl2vV4o3l95/hXbsFg24wi/Vdss1JKWgYL9Jc6ZPri2KBmdNINAbBv+Em9+NE4RxDgz6igXb1rWClTxvq7UlahvbJNsBa3bi63v7bRpPwT1yBn1jaQXpXimbsEWd9Hq8pyElIH1n4IwIveLilzfFzi/Mi/nglOc2dSMdXqjTnfo+oNTXlAhmhqRerlZHRI/W5vMrFFm1yqza5ap2kWMT0Ud0i82HbA1vrmN8hMTEqbD2dohEfN5TDwOMdd1rqwWyG7WNNI1ylrVVhSynTEnAoSvbDyjSiIPmwF4+E7ZBxUmMteEIMkYjDT0RZWiLCxbXAI1JKUt9OmZowUykEuY186ZBSafQl3VMClfvmHlVrfJ2K/hd89J2jOYR1Nbm8mWkFIgxpC8KOwuOk/Md0LjBqaB6WDpy6NlH3A36yBFP6qaQZrZgvKW7H7fEZhZtTJlSnyf62gpI8WBoTACGsfVDZeyno2PUA30zXBEW8+HGO/GIm94E38c56LZQtOao/+15l3XrUUK7vCYqldyy5FtNlXzNdj4QrxEP/IAbtW/EZ7VreCQKUrV9aep8VlU+hT13mIyWqHDqzS+HPAz4M+h/59pLXi2daFijQBPsrRMTNrt40qdNzmj1+e3VSvXFlMi6itUS6RhIEq5LB0F+ONaBn3Qhn+ahgHuEgCj3fPZBs/i31WPwlkz9X+dn1fDcGI2zF7ZzwlLl7Dr3D3c1PNfuu1UJttRqDEZDgOWaeHY7N6Ul24H7CfVo4p0KKkj1suD/QjcD17ZNZ0a/SqHQphNxFZxJ9vnGMmNgrUpjG9FaRJTON1eTZO9NBheY52z7Cvt7W1eZrFCsAIvAZEtMBBterrthYehjAfqETC9wXL4D2XXgKiJywufBg0TZOnX79eoWHkW3gpjtEVxk1UMoe7xSIBpQez/947+R5UgxmfTrkjdGiQHeLxoYt2KAgsn1oyq0cKTjfW+E0+S6mj3ZHnpWBuyLSkaMG0OKd3/yMbYbis5ru4OptWW+LKj6MXEczpvqScM4oHqjkbD6vLqkvE9TFP9+/rG1RubGwqidCJafSmcjBaJhn/sFYMFVpC7HhsJDDJhTHWO7YRXL+//x1aWl25jvDjvHJc4MalYY4w06CrOic0i5sb5IDOjcV6dsc66xc4ymkVqw3P2vHYRm6J83PJ+qtaY0YzLE5t4fj0W6Wc1yvn2Wuo/D9+nFE735QbQtIMY3PktIlF1S2foCvj1MxR8H6qUuau6SttTbinvv/5wQ7Fu5UGHwsEZxrRLlrbnNRUCHmjXjCe7M3XBtoBYt+Qyq93FpMuEvJJJFyemLbRlmB8dCUc+F8ZM/1C9Rmt3RMnNpEYhAnmWoKXdKQJGw1V3mOrrk0WNxmnSWKqvS19JGpneKTahCyQ/7aaWxMK5xv4AE81iZBdXj4U+inAaSShio1g5S4Jusmxo8Tukh1Gvb0+k7cNr6NcEthuFII6IuU3TULTVZ4xikVNag7gHR/cQmjZwISDeNNrnBqM0FYVxqJ1wH3BbrsIInRFXIVAaDAa7ASxUvP/FuVl3SeIlUwrKtxMYG7hNs5062po8uhwnFq0/1JskZTng4tiXzJLVp7C5xvIxrOlS6Q7HQl+mdUNHfVYv6NxWkkZ7GP4QNe1qDxg6VD0LtFQAkxLj3FVR1cNUY2czpgqUucFbwF/kmsGierym9s6TkT8AM3TgY/qUIzMldeXMLz3QlXBB02rqXvlW7LSNnKVfq7ovqHbF6o7WIvCkYJ6pktIpmUbNDQ/oeVOGCvnAoKS0ezeQUW7x9gptVZT1WIKuJhJyeIyOzyQFRgsMrWL9szTBrxPBEDrYFXHUxwyiGldYfPdrb4Th2xg6nLFxUE61MmBpMwDPn4UHgyiGOYoXMJkOSCuL9qqTstg2ACqU7JZ3Nermb1KB6WywSqZJ3ccLOCi3tePYrOu4SjHPA5+k76C/ALxN90sx4aIaNBL4pFn1+IoKx7hNq0C0Rz/7tJ9RzEQlemifjA+KQbteP00O+Mh5vo/50b4vcwHHw2GQYm5dzdCOY2ZU9vgHujzGCJAURroLOh9viaA8bJYjJu+EBYGPc378ZYZ+sLRsQTr7EZLqsVHQzhhQu1tQBxtgbxSKovMWc671nDXPvXJybix7cXX1+ndtmZjDncSra99trW2u2A391pnteOsYq012OMeObJUW7gjKOElN1rp2qRCnhqeF6zE5jt+yjhmSSfuYYZkgl6jeRGQvSFg5Jhbo6RbpadIklCHMsnE5gDtRV6uokSw0D73blBhDdv9v1rZNcRN6TmLZal+e26m+o8eyWtOTxPiBtnQvY3FNcf24t4KTbRR0uSxRYUrWHQdYTPfGQ9Aaq/RFTWGYV12wZFnAYEUkDaO4hXt4kryHAZLuIMkwBSSKs1HIxA46E6ZRMMisXfCjwJDfiwFWLgxot/KlwdZcZkaXF/R62AkiyLFbLTnNeTSyU77iu2nth4NRh2sEICoN0VI6wmQ9mets17ciOwzNsSVN8+pFszgZVSlrkAGrBj3rKoPOsOkL32QEvepLInoXPCNyU9Nnmjp2UwZrMkAODlQ6Rj6wpuHTGKzII7JULnHrkIW9JRXoKfTSo1Y6jj17aoO4brXEunVGw4MsoUZL1hw/owjikdCd2Iw73oyMxxu0D1pyJmgjUEVMYSD6URxl+2zBw0gNXUeJIapbYczyYIdhaV7z6652Q4u3OdPYiCVUk0gSDSESNF2AKVfW35BLOE1084Qmx65PWnENXBDFTkn/l65/ZCYbVFRnWTkW00JEZS9v279ENTz2VbuWv2A/iv0HAmbFGjApd/WFqXi9jeo79IvNGbaI0aL4UtWmZCm1ahigts9u9D/97MOTX33qn/zm1yf//LEH9ondaNQcaFZscqecxu3b60XNgojIEStelKZhnNgygZwOFnM/voJR2n8HoFE44GMeIdcxReYFhNonfkhJNNS0jP+AgKuO4ooDXnhoo/Lor9uo9wxUUl+lOaM+o98jVtKKm+s0jrkNY1G+Dn0Pe9qh92Pj6FJxDV/nqfVTB0SvcIOkp1hByVSpwut0L/zxBdLPF6RMapbOBa5vL8jBp+Qrdgq0fBZesqZdzFV3GDQeA+HwZ2UwyvM3Vcf9WFzdHPWKcznVqVpR7aVSXsU5yFmTVM5KUJG5yzwWOCXhQm1bMBKngKqdCXsGKNyYmAYn9yamAVVtTZA7W19F36eYhprFVjTT6wwOlNNE3MZsWW9yjosxjWTUfNZBnzbgMw62GOgqM+F8ozzLCJ85uucc2VlGVcb82KE+MS+puxVV3Eb9W3kgp0Wi5GSHkg4pLLOY1FGH67CKXTy+PVe1K0c1tcCJAaLUbUc9FhpA/dpRqrZZuEtAi7kwKDMMUyWkHRGa0aIE8kSJ0unoSutWRyH+UYrhFKI2bsknfV0gKiw6ZjWx7hSjUeRYChS0TCAZeUievrD0wZDKMksZqtgJ7lKgV0KEwyh7YoEqRSu0+ESzxHvFJuUws2O4OWcc3aNb4ReXzKoFtHgBkU+z0Pdpl8D30f71fVvcfIXGcOO/AWTZcpQ=",
  "run_review_distribution": "eNrtPf1zHEeVv+uvaAZc3iG7a9nGHGyx5IwtJyr8oZKd5EDSTVa7I2nRamdvZjeyELpyEoUKOFySIiYJsSnncJHkLleI2AmmCv4h7+p/uPf6u3t6PtaRuYOCKiLvzOv3ul+/fl/9umctjrZIEKyNhqM4DALS3RpE8ZC0+v1o2Bp2o34yMyOexeuDVpyE4nc7eUn8M0rEv2L5OtkYDbs9+Wu0OoijdphIyGQnmVlD6p3WsNXutZIkTAR5+UhChMPuVqi9pr/Z20FruNHrroqXC/CTvRjuDLr9dfF8fhjGrdVeWGX/GkbxzMzM1XOL8wvXgvPzi6RJW1aAF90ecMKvx2ES9V4KK34dhh32hzPfO3t1joOqdvwl6a7pD/st6G2zSbxvzwaTm/cOf/Wz8WsPD3/+cPzFyx4Je0moAc/MnJ+7cPa5i9eCq1eeWzw3p/fGOzUbPHp4MLl3Y/Kbt8a/fePE7Mlg/PuH44ODyd1bgPnuoz/eZm88X6K5fGXx0tmL8z+cO5+P6nTw6OD25M7D8f19440b1dXnLlyY/xfA5gX9KN7yJMzi3FX658qVa5LWyVkd92cHj+7/+fDt2xric1cuLZxdNMd6Ghp9duPwp3cnrxxMvrg9/vTPJ2ZPBUD+SjB5/b+xb2/cmHx4oGGh7xbnFq4sXrtaiOokR/XBW+NPHy6cv6DhuXDl3HPFCE4b3YAfb45/9+nk5/dgXjVcc8/Pn5+7fK54aN8IJr+7M/7T++P7D6FDweFrv2C4Jl+8AyAGry5fm7uMjH5+fu6FQrxnAoaRvcjqZlmufZM/AhyHb35q4lg4e+77Z58xR/oNEPi7++Obnx6+8fHhK/cmH91AzithmHz08uSzj8df3NDw/HB+oRDHKQ0HwGutL155pmgU3/62WIUf3n70RxzCDGPm3OJVaFfxHh3c8qrEg0WFfw7f/4T+unkPIM+dvTb3zJXFH8A0nJ9j0GdqJ/H9N2un8M/JWfbz5Gn594ze7uLZ781dxIa7MwT+R1s36B/CF/Af9ie//oQcvrdPCTMoRN6gfwiy7IO3CAODLo7vPxBQlHaD/SWT3+xTuHcPxg9+JiFOc4jTCPHe68iUV18m4/uf44x+sD959c74lQeTd/9Ha3CGNzhDTiLHb/7JBP/TPsDuzcxcnL8KS2f+4lzw/bkfvHBl8Twqh8ndt4D7uERefTkAPTX+Yn9y63Vv5trcpYWLwJJUA6kdKqAix58+8D2cHKpSYFaDy2cvzSEYCpGADCa/enty8443cwkQLs6fvWhCngqYfjR0GuC8MLcoFqYCPi2ERV8mgPjy/IW5q2YXQIrG//XJ+MM7wfj1/fGdvxy+fwvY483MX34eNMt5OjKEDc49e5bK1fHvfLfhnVhe/snTXz8OxuaflVmj/yUXwNIshu0o7jQo7+PwpW64HcYNkgxj+mSz2++oX+2NsL0J2lc9aa2H/fZOgOZGAwPzuB7FO+pJEo3iNkBQ24gPOmEy7PapgWdPyU/I5agfMmiw/KNEtQZXgGFHJngMpPvjMFjdGYYA1gXr1ySzrgEu8gFdBYxJ1hjBzo56Q2p2NWz4ZgsGEndbPde7djTohp0gDyTZALvfHrlRd5MEnAPVnnKya4OtRe1R4mrPXsTRtotmr5sMHe/SJGZmOuEaDKW/1l1H/wv+BT4HuByk9l06H4xn4FtstJLWcBhXwGuqJ8NONBqCpolD2dTzGSjtg4SpaxAVIB51YMxNbzRcq30LVFUYx1GcNAHPoNdqA44MagBXSA1gpqbGht/pJvBgJ0A/roL/kRIJEkI5AX8ZvWF4HRkHvymg7G+UKI+rP/RU5+AdtoEOtuJhst1F+7AM/3t6efm5y+eWl/WBMFkEP7hPKIxHnqKNl77VWCmDMAMXRfENjkJ7xke/lkwx8BSrUhz4is0BgyKVwlUQsdEQmIWNEEkdHlGMCCQxCriioXICArywuaOtOSXAdtF46ZTJN0lfg+F83NrsdGONi9YSgqFutTZDgEkqOst9kMvruFyjzea1eBQKoeTOv3tuuLKkJDTlKRcrwpNuoj3UxiDV7Eut3igUXosxmfRNvZsEYoyVNM8YjIxRdC5VZKBygoFpsQwfHjMIbHQQ14FKEtFd/TKIEYhDWw2voSM3GINN6wyXT6LYoJwOatLdwECi1QNb0pmqKyheSFk1T7En3U0FTLtKuS45vpAaDaiS9AhSZFwDNsMv35ojhh+VmDUFxhSyl3VcNdS4V/gDusafIpnhmWAsMG0TzM3RTLBABosnc5a1UCA9zT/uDo6mJ4gorxc8kEj3YA38qAAWXnC9l1yvbIY72+h04Vqukq+jCmj3Rp0w0F/wRQ0zZaz1hvBMwBlL4OXSCvcHYvh/D8wgGHfVtyrReunNzmoesfXK4d76hh0DL4xTQM2Ays6yN2B7wacbhfIh9ompor5oud6LVive1+vIBtteccVFZczQ2v/+NRvUSY7j4DxEohIdafU7pGJxWWhInEz7FQ5WR+Cgz6eg3hoMwn6HK1Cxnnthv8IBfLSJJxsaV0bQGTCTy32v/qOo26+seYTUyG53GG5RYnseZR3+xk5IPPAMICt9FAdPaYJWF9M5OwmAz13vDgHd3PVB2B6GHYKjQ1ZTZrW6fUxHHd/lg9w7XicXsDON5f4u7dWeZygBTnhpdiUlz61eL2pT9/1oFpbCR4dpLpdUnCd6ASMGd2R4RGpGYHP0wBk8ppZtM7Oj263eJvPiK3EUDTUPQeQCl/DJijQvKH/CO+omuNiE04Dt/ZQ5FnFTexNVAsIwtbC9AUTZC9WkPYppyrDJXtQH0YAbB+rqQeBmCDsuQuxM0oZVpPWEo/F90koI/CPG0CK9TKEHHYCjFicGoaxw0Cqu1GavtbXaadHmEC/SP0wBtFtJiEqj4quuAb/DwZBcuTqHrjwle71tkhzEENvAElh64ezi5RUIRTGRTLDrZNfwXkXv9xpkF7AI0c9VZrR7uCQhiAxBxjoVMTpbFeJjGLHg9AltZLa+Ym+4SoUR96LtINnZ6nX7m0nzQquXuJQPmziueig1s/thT8eMklca9U43hL5TnCI+xH8H1HV9gtKbJWasiVPGjDnJkS7UpA0i9asuW40jmQ7GM+ypOdXSl5cJBo2DVTMXQRmKkbvOTG7SmwK3aFHMadYyzWvhMEhHj3JNAi1Rg11ouQXtbIO9UpU4S80EA/dNlomMRQHXOt32cIn6UsPRoBeyf1IucjaqzEcOKGZI92ZsxyU1eaoDGm9FZqyqZ8QAITVFAX/Wa62GvYryKXTvSqbWDNkS/V4Sr7GTFY1ElajATQTCvI30xFM9kNk6yjyND/CfFc2/RLEL6/Sfldj718rTC9/pR99d7uyerJ7e85fry8nX8dlq1Nn5bv2pp/3K043g1Oypb/7kaz54l2qIXFYoonTsjSlrTx/AmreLOpyC19fjaDSoHO9Hx0GUZ0+Di1Il+hsPqXs+iF/cHUj3xMyq7TDdpQV78Lv8GhQQ2ApYYqKx1yQfA+Ko6HpTw+G7e4liXraLT1LgS/Dub2wJyBGJxDRMBzh6Xe79afZM5ru6qII0GNxlHSY0jHA8Bw8ZOWfu1DhSAq6GRnrp0cODRw9ujH/7Md0EEKqw1QnayUs0oWsnmNicUonFxVtlwqvmHB8KmWAGFjwGOwHVD7fBvIVNXIpW2rSWdNc9anw3YOw9I5vUYpYJulY/DwQX6YMKg/PtwdMFwdrU19Ba4tQl6GkvrfhV/bX0mWMwE66BV4lC0JDb6nz8LOstHypORKs/gphoZYWyDfQL13Mya8f30bkcgdHHLsHwWM8Aq5/PRG/be0xO0oHqnHyBPuCc5LxRQ9b8YdayTv9sMPa7X1IOikEZSwUDVfVC7AeM+sNMkZO8s9wPmsa0kptp74PtRARsoqiBsaXb1b1E+QV8DQ2jIIpZakqXDMs/VVGfEdVw5LTbGkbmas5okcbzmL2kwUaqscyackUmWDfYYe52ErdFp3Czy5kUZtKHrw3pY5UrdcR1SvIT8IGUiV/YxlfJpV4Y4I5Y5lS5Ri/mbT0c0rbmxM2k4y3HXPJB98JWP2hHWwMYGVhumvNtODYRcEmy7ROaDB/GrX5S2QWngHgBy3m0qQlyb2ju+UbamiWgKQ6M2isUuwkC3ksyWgXfZTl5Ct0MAv/hCWnuL9Rj9g+vTszcB0OBGZfA09k8inuVLKOhS0Z6p0Qv6GklgKhbMVUdoA7ElqEh1OBZwyTlSFBKf2H0yiJ8lWVSLr5hX72leYCN++HwKqe94lUNiDXvucWLzV05fNYdf88C8+aBKu4oN881lpdf6PY7sG6Xl1li6vSp5eVkI+z1Tp+qd3o9V9P5fie83jxtv9J+r/AQgcb5OGTGN9w1qvARp9SupzQH7nOj3gB9UmG/GtoOuOVXcXNhTOyu7Ikn/B+vwRHXxRPVXQ83zxUA/tJeCndGAUgHSgFpbo6C030fDR/3bTR8/IkGxLL4XsPcyOPwfOdBA9d26TPaaBA+c5rsx6zaTJ9Gj+3vq46y3xoAbvir1/hLby23/jUM8hkD3DMWlnS+0diAwUh59XyhyeID8CCMyoEqL0NwOOdmncBW1GGKrzqT3kI3oOhWMWp4batUAgD72qCVKPuasP5q5MXdVKnJ3osNwvzFE4dvvIG1fff3iV5yQiY3703u3ibwdnLnz16aDPYjQH+FE3E2fvPTw1sfcxwNskvZU3cWP+w9OrjNqNDsU3q4Qrk9wSEfvvlg/It3yPjgHay0OrgBL6cduN5WDtgs5dBHmoRfcjRa5ymTyeTdn07uPCBYJ3j/IXn02V+gO2T80b3xzdcP/+M9ViE0/ghm6O7tR/c/LxheuWmdpQOimLAd3VdSWTbvq4INYtHsEbaPRGSBnL6+tX/j0Cev3pns/2FyE3j6MjBUFM/Cut6uUEO8hj8rx4/9oHZsq3asQ4492zh2qXHs6nHDxlBUn9+e/OdrhBXWiRHRzaqG6qLK19B5MjE4GC6bOsQ5pzGrweLyqHBk1fVY6NKTlUNq8tl74999Oj74hA3UaJti0UdvT+69zWrKyOTg4aP7d8m5q8/LDmoVRY4Rmq3v7k8+e0AOf/UzqzV65K7GWgmb0cosSrJaGqbhq1/lSzhHnl7ctcrzlFpAfSBkhNYBErmpyeXENQXaak1Tsov2kJg+0CphRbnA649x2R5+sD++/wCevvryZB/Y8MZb45sf4wykRpGq8aOoEYVYVJN33+IrdfzWe4/+cDB+5T2c0smHB+O3hF4nrCQwh6GmKFGtksndF81NklSuy1QD/t6LvDVzzYw0F99wntx6ffzKg/pWx/N1f035pVTh+NmOG4gqzFPAJA8iRXdk5cqSyfBHBc8ZcScGV8JMK99O7AJr6QZ3OJuEPbbfSvfetmkkg38xfYzk0DGKtjHSqijstMRD/DLSQFbmAwdZNcYg6PluN4exKhltbbVormsthMigHeanFFkmqYsuNPyHO750zdLEIHZVw4OTaxajU13BhL8OXfesasRySLR1pZAonYM5EjM3IXH7jurHNLjZH75dP4i7ETBwJ6DARsoUOaGypKzMzZXsUCi1jEdWksPqs8i/agKT2tKk/aO7QFyKxDOaPGeVAZXVXqu/qRUGOMa2JH7jqKx3FLF4ViWzPnmKnMz1CZQ34DA6Qt/8+p3Jrb/keQfcyFDJ4VZDszIW9LPzzzx74tLc+fnnLjnNjWloDOVEc9yClVgj4yEyZCDDh/+6eOUFq+pFb2ELisFpyiOxI4td3RXguMOcOQ9i35k1B82IzbNCdTs21jkw/vk7BHn+5vuEHeA4vHXbNE2T928QtCsfvDH+5W3O8MlvXgO3cnzzXp1QG3nnvfHvHz760xtgfAiWl9+9hRYM/iA2Zmmob3prH5v9/ga6puxAC2FrGPxmcnhrH6b98NYnHHVGCF8pUAZUch7LcOibSVKUqpZ2sCrVhDBnlpFkK00zVmQpeJVYEOl3y5Y2ncWHrO/RaDiAWIOBpUrpfNeulNYktSNFHWIGZhloy5Eyg1lni1QUw2noM+kgY3tRnFK/uwbGLYuS5R5pxkA0sEt45NvMCj150squi+QJJC4HmRTSYJmk0sem/FS9LJ4MzB6O9j6TinaOyig7dBQFKkmoGrNcNeevakyMpg5ZspEXFsyobAn03VgC0g1sSkeRDxfXAywue4WoMkbeRVrt0CTO4crDfnoDnhPrYg6RsdIsH9DQdtd0KrKWkeWpVN2k6rpkJXiJ6YqAL4H5sfZOoRPW1in6h2rM3E/nUIqCvjHBHGeFIR05k6eaYPyVs0snTlg3wwipaaykymJSclBNgWBWFI9kUNKiAs+rOsrIGIea6SypsVPMWNd05kglLp4YbXoOQiwL2kQupd5pac0m9cpddVnDUdL0WILMhV5mKptq24bu7Rigam64/Gmb8TDX0+3Rm2rdWipFRQg2HiYrAhg3SMN+ReuGRU2qErYxa21Pi01ZpQBE/oR7zvBULj8x81qxVxIOKw4NALaZ4CvnmHUX3Sgj0OTfUiG8pSpBUCpR665zeeymJMDcO8haF/r+QLbIWxsEuVLvaeNLZ/PlK9/RMuT1xIEmGqjibDSZpTEW0r0ZU84LUmdcyHReq0praihL5ZpRhpSgohRpYmvXb5bWxwqHpZaNE4C8/Zq3K1Dv1cmuvXGqEfP36qO45+l1McyOIAp3M9kF1nTGoRtFf/R1zw2G3lOz/De9OWlhq2qMtMbvSJ6bpqXIvBSZmJJmRpkaOW4pIW7oEgZnKqNjGh6WrmSxb0YHuCFSrHWD6TbJnhc3Xm6isKLVaaLEkdumx33HAk45TZrVFYdW8S2DZ+45o77n3BKrVSun0+QtXYyLS0Zryg6A8FReVolW4cESffowr2QTyATW15xbfPWV6AQ4MZXe8HJwWO2sKwJoBkjuGssR+C6Fkp7BDF2R4kPVYJ7vdKGKdUYZvVFGd0yhP6bXIVPokal1iaFP0lv9GdpEZ302sK5T0vOXTaOMZlHaRS6nElx06phU13w3At8huTNTeARH4Q1o8aN2GChTj4EOEyHc4x6Fi8NegNcQ8Q5SlzZdPEejwSzzLVGIhZGHJNv5m8lQos66XL3PZkse51qOS7qP2U4hrbdgbpQ8BW0KL+6Yy2RIx1R5KLX4XoQ0tajf2/HsaaNbUo4Zygm/013g4eOMa+GoSzDSWtNZl5FWnq7t7iy3DMgdUYgvUkyC7l8rys9WjkcS69vVS4aSM2uXHjsJ4LB27KcJ4o66zRZmBZzPNaG15ajtkWsp2hO0QF3uJfM9P7YrQLfrVPdNQVEhr7WBQSvkrGcy8LWe66Gu3UTUvlnPebmb9VSvarMbsMo06yktR7MhVRWaeuO76jv0qahq2x+PxXJWdMKih4Dd/8OuOrL5r7FcZ6nFRjMpUM2J9PWhaXG4PqBBGCdRH4fMAcqcl+EFrI6m+VxyEgNuBX8T7NK2MpiMaif5tV1m+0Sf2Nzw9aSGhkTsH/LsmGwgD+/hqP2yRyN9flPTsAWjpfvX1PyrLYvUjh0teWG7wGr7PnvXXEedPSauvQzgfEY59mhKcizd8nFZZ28aGFQb6dy9uROaSt/jEQ26dYxJJEdBDHMpqg6fK4/9ydA+AJuqEfs/ztbw7VvBnqJcjecVZ2myYHJ3BYq8hXIegx4ksVnMT8BAyC4nfo9WhkyTgqHzWzLvYlf3VYmrcg+Er3ypkS2M7lpsfdOXE7VDiarDnZ/SbbFuc/yH55LjuSw5/URWosQiAroHSpXAim7/xYlyNoccwjgipF3kk31TFt/D7tAAmx7Q0rfM8aKQbW5lzZIkfXPccYOnp+2OV92N0hd3phut6EffWn2Ued7dr/BD+vSMaMTSjeIlP5yT8EIv9tIYin5SLn2hy2K4NkInggwjxkoy6gvfgzBO7tobOpSwv+f5BVZYgup36rETaPHWMA5DB6R2r5JcwHxyK9oki2pDfm1SdvGhdrER9QfFPUtsyuP2BiYcVsG+wozzdyyjKUvQgsOffn5465NH9z+f3LzN517cvKkj4XWI9IwWHyUeRQv4+wpe7acTxJOdQNFjRwuRcFMdlFPXUlUJAtPXUuuZR5I1+lblESvdSfLvr9FukiuqDBL5D2PNadms1FLkGH1tAuQjseiCEsUbdOWX3ehl+kSkbPpE3ofbMMsSpCbRBmzUarmcHq3DwsPjP62iByNAZiYvCLDbQaAg2Z0y1pk4JXoNvlfcdJ1AqMrD6OK1PBAAkmXhFCZOwlonEqoyVSohrIMpDpxZu6rNwhMLaWTKQWimDwRoBXZNd+G/Z1c1CFFlF7TRydWcY3iqxUX8CZd940o3w9vVZJ8iQH2cofqEmrFv70mtEEMflZReMQIhXU5lydeZVJGaMNvy5/1wfqHh+c7b1ZCO+x4kYhkGWneMRiHXf4pkKaHHfSXbnAb/cJ4e23nStNNKmYyQPhvaReb0UBC7tPvxpkFXVPY7U/OkOO/KM6d4ZaimFIEstWMDasGo+5UjLvJMxeOUqW7WHCrDYGS3+IJa4PbUWlfCYO55unNE9Q2OUii3skZeUz7Zhj5XG+ZeDqW7LvmeJx8uAUgw4Lhvw+6BxGsDU6pFIdXczi+njwt0sWkiSmhlq2xa57OVmSy+y8y408e6e9nlw/OOcZrZjDQQ6xfTFZoTbTjZJiVtTrJNSbEZYYfrR/0AhjpQV0tVSTvagjC8w11BfjtNtK4CAedVBhzCXHPL/aVdejHmCtk9To6z8n+OXzIIyxp6ITuFpb72UoeuVbQqWNpGrfX2dkf68yIUNFQB3mLe1NAtzC/MGe/DONbfX712/spz1xQEnlmgFzurR6mbydUb64ZyXf8gY3jwwnnEIqCc47THto51gmPPHrtEj9IGjIV1aO0ZGM1rFDgT5SXv7tMVmIs1bpewG9ZBWLpDekSj4i/VTp5qrEiNgI1tEcNn5qRfjNbthSF6rKsXRZhFWm3c7IVYfDZXrTFekDWaUmaX+oAuGhLaeteFUwk7voVwPthoJQF+cKAiHpiSvRpFvRI3ImqNHdfl0AsHjTiyv6M3EenpYPLu25PXbvB0vHLNfHpuIKsR3u/SC/qjrdUwTkrAM0h4tMbzfKKRpgYG3UGI017W0hXcR61SgbknZrTeZhzbUFzOPq8hPwnkp7JOXXDo2NaHE78OkEvE/mKQTemJnW8BG9AxTujYuHWATPT6x4X+zk/QCBWZhVpYqiy0/LNAqUM5IndIL3vfpFZc+/6IbaqMFKf43EV4PQS/mn5MzNq9iSvaF8VOoBu/GRoE6oMdz9oU8Go1NNE1tqmZQui66tlPoxBnN7LQpO9qdiBhqbAa3UJNo8g/DuTbpwx1/zN6KYzFBgQyw673ZswWvhX2RDTwfNsbZAt+MOjtFCHpxDs10IkaCukoGReusxsyPOkzSUcpS2SkIlQ9kI9kxUGq4FTTpRhKso8UsTBSD+ftiiY1XuuKYTCqyhXxTHS/f4g3Vtx/8OjhgRO3f8SCrobf7SeYGEd5lfk0t9Q78wSIOnUTkiGejgbatx1SjcR05DWTc5YrxKVkjgJ56YzYZhgORJa7UPoRuCaAM1YAXpzWHrpEMaNjuN1RG/XZrfId96I4ZdRvMJGaYl2sgYYB46Jt9rMHpdcEu3wEv3F2JGvCRPfXXhNi8JrbVLAicBPj6MTbZv6TE+443AKFHYCJGRbiYLA1CusWwtOGGEwhfsKFajg6uDrC0wsCIi8LYjhihalpA/qIBUjixthQevYu0aGj845CniKngjVGmWoDwfc62PWkhtfy77h8GMPBFeABBXeJ9PUaD4pqgzCudaJ2IU5oIwKpYIAxUtTOFXfZEP2RrbBYaBHKIfkSD7jVMJphESIOlocpaseF3lE7zsPQ6251M3rCd+OWqNMJUF7VwUz6xl9xLk7zq6X8W6VTWAiMoNLrsxNt93tRqxNgIGel8gTt9LnI2TM0nhvfvDf59YPxh7fHv7ztKNBZcpfs5KzE7BVJ48xON2n3ooR9sI/GtY4lqfY9+NAySoecS870/1Vo62fi2I7iTZD9PDwyTuaw2cg6Ya+1UwoVhcxGlKsYJNyZWceblfQjLlrVrINrTmGi4aE79WNw1qcmAYU1L7+UyjPnCOc3qXAaXy7+fyecHCRbNjl7vrR8KpOTgyrTLpVeLPpUHZVM5cpDeXH4J/71aONDs09WIJjPI2o4aboHMw15AnGEM5UvW8WzVWrCVUYuEwF+0KaWjFbBpg3Djiu5MZ1slJ3wb4nPhcsvZz/Z2V4P+/g1oNCccLYy86ecNqjBUL8co/l0FiAqO+kFaPTM4lHMpnmTrL4RskSHvkLQfxmEnQZXCN0k6hPUB2ojEffvrK9caYOVVR8pp8hMyTaytj9z4hRWKcIcM/DK9m/bn5M39iQyxsm7UWPd0AYsoyZ2aUnOeI04Qd86TbM3K0QqswCyVJ2V285RdiBjoq8ZglYQ9uRqJybnqTy7EwNGOxvdIQt1HMlbw+kSSDHcwUY01nFcG7LiTJj0I8kh/sm8AJWj67ikHXH0oxpvQvWpl6EGvdlvBzwyMMxcmfgAepcqlnTVaMpv2NKqobL7XCU3sv6u7jcTK/uy+k4uybpgVq5W0egCjdQseMUAG/wc/+QB12EpPZgalvrY5ZRXjpWqMqE4xYkd+hWTo7o1zLdvLypzk5OrSJZTfSJ3plld1Blh3MVkn5JTX4nUrnYy7nQyr3JKn7cUpJLRVuWkUdminbs3sr6y1/kftQs8jZysLwNKs+mnolJDreZyV93mV+pbt9daZjTruln7zltDr2vDUAyb7upiPenIFpmbOam3gkkOzYTbSVSLs/MYj8UtN72yXMtonfpGqD3eL8/FktXmTJKbu7qq2TOOTqlCb30V5pSdC5TmWtqzL05tqmt2HcjSjGNtUs+nq1XXLgDz91xbJvjFCfbenODweguLeei1CkBRfHFZ3adDdnU1o+54W5pd2fNSF89xGkuNMyt+RsG1UGWCMv3ALPunKiJqddBb3Nriu+r46SvwHmLNkTgbr4+2gGML9EWqPhQf1hFLi8NVtB1NMtwZhE1xR/paC08n0CMv6U/K+1WyEfYGTU9GqETtoEq1yq8RFTbXTV2P3F09wO4LapoGnoqGFiq6SKTcIEGPtWOriF+WW4qeGZMUE047RaIHJqZp+sBDzWLiWiWLoLoxAre3hrqNfnyLoSpFlbu+xVQXzp77/tln5nSqsV3rao9WLQLhYaP3eQSLYMaIbJhg1fDEqBbVtDeibhuUSoVdCFPVbgus2te7aHGNGDdvJZ+zIXf79Fvg5tcWWglhpfL042FJlVg3qsETPCJOV22NmkNBPSGUvFGBniGhuCUoZgsG0KIrF9bzMILAfxiP8KGYFtz6JMON0D0xZDUEfSdegg7Ll5AfdweCbpJPmFbS0K/RQxvKC7zAgdhiUkyuUBh/OL+gCyKSm2a1U15iCXQpRoryfpIiIzgJLwaMj0ziWZqAUa9Qcc4QdV3CwXRlQFX0iwTEN0ya3uKoT3cTxCTjhOOHeUmvtRphzDIM1zFT16mpbD2fDLzQcxh3V0dU/cstX/0aftTcQ7yBoN1rJUkz1bXzbEqSZ4FVFwSwLshYLk27n4hLtNhMqOf07DouNBq0i7wVLS8VHFHAdaZGKD88LLcSsyQSktxjEmtQmh5peeAhnRh1DwM80RjIhSZls1WHFIChz2wA9TsleqrMrth4KliCe2mrUbTpFRJQlysXoefsEqiJbFlIw8jA5ptJWWsrqOJmlZ7adC3bXP51o/KW0i7DNTqxcP5CabJmEi+PpF46K8jJxCYWWNRYJD7luHvRejFtXosqyIplLXOggKQ8o2kNTq56hFcjkNEd0gbHYxieQINJHB5fUicvQGwNIwYV1YU1N4AJbMVMsjGfl2BHSCfeqRf3S1Zs1niJZ34fJTgRaqJDLMFPiomahXK5BBFUVLvzFkwjkc4oRiMiGMSUQCFpVn+nnO586njSgd4qKL38YTSo9cKXwh69wgYclVYfkx+rITEr9HK6YJRSlTKYCElA0Li0JaS1ht8+5pViyuPIISp21+mKz6cqQIlY1omwy0rPlNQssr6BLTP6nSSxyr5haA9BFIbFGrGPEJWkw0sfGJU1wKPRma3PnskiFYf/NsL6Rta+vNrihUUFk4cwxFBWbCsGxRYvX1SekCwHLCE81ObKrhRMZYgHcvgH2JmxRq/G0Sc1xXpCeAqW2EUjjhk/OTs7m9LhkiMCAcsJUHcrfqnVm6ILzgo0Rz9Oaf2ANoS3oc617Bc0phSmoC9Ks9IUJTl+4I2GKaDHYpC9y4omO65bnqAoS8uVgWG8Q559YYEw4IS6PxsYS1B5oCkXYy6moI8FbflrAOzQlXOLMFyCZcxU3KhC+ZI9EFkBxy6cc7712eYbYgTb4fmwFpj0jjn3FFNhJ/qR7IexyZbLEbYIzVXGpYFphGHMWtNTdhBhJCdAOYBfgJam08VziyUkBPeAuUGv8TPgeZ1CcBh0H6jYXiw38+VsK1ItaVkpSVXeIbNZ05pzJMntYAmCypuq8YhzGhtKiZVTvJSaoWZPMC3Dte3UbKWZuDKzqGfsMGF8gievpqZoZt5KkLZSdSq/Pw1pWFSlkjKUYiphZlAyrgQoioAVUa6oOUZpovleNlndIWovwh3gmjQzglwbSN5NUNRTdq+SStTIHqa4IbzFbR4tcBeA3oi1EW55vkXzySYzjUgph+hRJqxyyDzJpJW6sbBwNsUlKtI/QN+nPYrx4jMzq8S+gZglcwY5kflhh5/ZM55K2wJzV7EywrBw17rrrPa0n0Q9WYgg+m7m3/jNYOtaHorf6Y5ybd1gwUPlbqLRy22/RJNRK9ZVR7LsCG8ah/cNo8jdOD3tq4+pp5qKVV5UqpKNAVefcddSxm0leTj4nJtXdo3MdolRTkOZRO85qKx5o/5mH0IYdVXErk6BbZYB2YDe/hMElGQQ4MQHASfKpGDmfwEd9qpy"
}

MODULE_SOURCES = {
    name: zlib.decompress(base64.b64decode(payload)).decode('utf-8')
    for name, payload in MODULE_ARCHIVE.items()
}
print('embedded modules:', ', '.join(MODULE_SOURCES))


In [ ]:
MODULES: dict[str, types.ModuleType] = {}


def load_embedded_modules() -> dict[str, types.ModuleType]:
    modules: dict[str, types.ModuleType] = {}
    for name, source in MODULE_SOURCES.items():
        module = types.ModuleType(name)
        module.__file__ = str(TOOLS_DIR / f'{name}.py')
        module.__package__ = ''
        module.__loader__ = None
        module.__spec__ = None
        module.__dict__['__builtins__'] = __builtins__
        sys.modules[name] = module
        exec(compile(source, module.__file__, 'exec'), module.__dict__)
        modules[name] = module
    return modules


MODULES = load_embedded_modules()


def run_embedded_cli(module_name: str, args: list[str]) -> None:
    module = MODULES[module_name]
    old_argv = sys.argv[:]
    old_cwd = Path.cwd()
    try:
        sys.argv = [f'{module_name}.py', *map(str, args)]
        os.chdir(ROOT)
        try:
            module.main()
        except SystemExit as exc:
            code = exc.code if isinstance(exc.code, int) else (0 if exc.code is None else 1)
            if code != 0:
                raise RuntimeError(f'{module_name} exited with {code}') from exc
    finally:
        sys.argv = old_argv
        os.chdir(old_cwd)


def notebook_run_step(name: str, command: list[str], log_dir: Path) -> None:
    rrd = MODULES['run_review_distribution']
    rrd.mkdir(log_dir)
    print(f"\n[{name}] {' '.join(map(str, command))}", flush=True)
    script = Path(command[1]).name if len(command) > 1 else ''
    module_name = script[:-3] if script.endswith('.py') else ''
    if module_name not in MODULES:
        raise RuntimeError(f'No embedded module for command: {command}')

    buffer = io.StringIO()
    started = datetime.now()
    ok = True
    error: Exception | None = None
    with contextlib.redirect_stdout(buffer), contextlib.redirect_stderr(buffer):
        try:
            run_embedded_cli(module_name, list(map(str, command[2:])))
        except Exception as exc:
            ok = False
            error = exc
            print(f'ERROR: {exc!r}')
    output = buffer.getvalue()
    log_path = log_dir / f"{started.strftime('%Y%m%d_%H%M%S')}_{name}.log"
    log_path.write_text(output, encoding='utf-8')
    tail = '\n'.join(output.splitlines()[-12:])
    if tail:
        print(tail, flush=True)
    print(f'Log: {rrd.display_path(log_path)}', flush=True)
    if not ok:
        raise RuntimeError(f'{name} failed') from error


MODULES['run_review_distribution'].run_step = notebook_run_step


def run_distribution(args: list[str]) -> None:
    run_embedded_cli('run_review_distribution', args)


def zip_integrity_summary() -> None:
    zip_dir = ROOT / '40_전달패키지' / '02_개인별_ZIP'
    zips = sorted(zip_dir.glob('*.zip'))
    print('zip_count', len(zips))
    for path in zips:
        with zipfile.ZipFile(path) as archive_file:
            names = archive_file.namelist()
            bad = archive_file.testzip()
            url_count = sum(1 for name in names if name.lower().endswith('.url'))
            content_count = sum(1 for name in names if '본문검토_' in name or 'content_review_' in name)
            print(path.name, 'entries', len(names), 'content_review_files', content_count, 'url_entries', url_count, 'bad', bad)

print('embedded pipeline loaded', flush=True)


## 1. 현재 상태 확인


In [ ]:
run_distribution(['status'])


## 2. 전체 파이프라인 실행

`RUN_FULL_PIPELINE=True`이면 전체 재실행, `False`이면 기존 산출물 기준 패키지만 재생성합니다.


In [ ]:
if RUN_FULL_PIPELINE:
    cmd = [
        'run',
        '--source', str(SOURCE_DIR),
        '--normalized', str(NORM_DIR),
        '--material-mode', MATERIAL_MODE,
    ]
    if APPLY:
        cmd.append('--apply')
    if EVIDENCE_RESUME:
        cmd.append('--evidence-resume')
    if REBUILD_EVIDENCE:
        cmd.append('--rebuild-evidence')
    if DOWNLOAD_ALIO:
        cmd.append('--download-alio')
    if CLEAN_PACKAGE:
        cmd.append('--clean-package')
    if ZIP_PACKAGES:
        cmd.append('--zip-packages')
    if CLEAN_ZIPS:
        cmd.append('--clean-zips')
    if APPLY and CLEAN_PERSONAL_DIR:
        reset_personal_dir(PERSONAL_DIR)
    run_distribution(cmd)
    if APPLY:
        move_assigned_sources_to_done(UNASSIGNED_DIR, ASSIGNED_DONE_DIR, PERSONAL_DIR / '_배정처리_목록.csv')
else:
    run_distribution([
        'package',
        '--source', str(SOURCE_DIR),
        '--normalized', str(NORM_DIR),
        '--material-mode', MATERIAL_MODE,
        '--clean-package',
        '--zip-packages',
        '--clean-zips',
    ])


## 3. 최종 검증


In [ ]:
run_distribution(['status'])
zip_integrity_summary()
readme_path = ROOT / '40_전달패키지' / 'README_전달패키지.txt'
if readme_path.exists():
    print(readme_path.read_text(encoding='utf-8'))
else:
    print(f'전달 패키지 안내 파일 없음: {readme_path}')
